<a href="https://colab.research.google.com/github/amzad-786githumb/AIR_LLM_Research/blob/main/07_Adaptive_Utility_Based_Selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ==================================================================================================
# AIR-LLM — NOTEBOOK 07.0
# ENVIRONMENT, CONFIGURATION AND NOTEBOOK 06 RECOVERY
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib
import warnings
import gc
import pickle

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


print("=" * 100)
print("AIR-LLM — NOTEBOOK 07")
print("CONFIDENCE ESTIMATION AND ADAPTIVE SELECTION ANALYSIS")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. GOOGLE DRIVE
# --------------------------------------------------------------------------------------------------

from google.colab import drive


DRIVE_ROOT = Path(
    "/content/drive"
)


if not (
    DRIVE_ROOT / "MyDrive"
).exists():

    drive.mount(
        str(DRIVE_ROOT),
        force_remount=False
    )


# --------------------------------------------------------------------------------------------------
# 2. CANONICAL AIR-LLM PROJECT ROOT
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = (
    DRIVE_ROOT
    / "MyDrive"
    / "AIR_LLM_Research"
)


if not PROJECT_ROOT.is_dir():

    raise FileNotFoundError(
        "AIR-LLM project root not found:\n"
        f"{PROJECT_ROOT}"
    )


# --------------------------------------------------------------------------------------------------
# 3. MASTER CONFIGURATION
# --------------------------------------------------------------------------------------------------

MASTER_SEED = 42


DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us"
]


TARGET_REGISTRY = {

    "adult_income":
        "income",

    "bank_marketing":
        "y",

    "diabetes_130us":
        "readmitted"
}


# --------------------------------------------------------------------------------------------------
# 4. NOTEBOOK DIRECTORIES
# --------------------------------------------------------------------------------------------------

NOTEBOOK_04_DIR = (
    PROJECT_ROOT
    / "data"
    / "notebook_04"
)


NOTEBOOK_05_DIR = (
    PROJECT_ROOT
    / "data"
    / "notebook_05"
)


NOTEBOOK_06_DIR = (
    PROJECT_ROOT
    / "data"
    / "notebook_06_candidate_evaluation"
)


NOTEBOOK_07_DIR = (
    PROJECT_ROOT
    / "data"
    / "notebook_07"
)


# --------------------------------------------------------------------------------------------------
# 5. AUTHORITATIVE NOTEBOOK 06 FINAL ARCHIVE
# --------------------------------------------------------------------------------------------------
#
# Notebook 06.27 created this archive and it is now the authoritative persisted
# source for Notebook 07.
#
# --------------------------------------------------------------------------------------------------

NOTEBOOK_06_FINAL_DIR = (
    PROJECT_ROOT
    / "results"
    / "notebook_06_final"
)


if not NOTEBOOK_06_FINAL_DIR.is_dir():

    raise FileNotFoundError(
        "Notebook 06 final archive not found:\n"
        f"{NOTEBOOK_06_FINAL_DIR}"
    )


NOTEBOOK_06_MANIFEST = (
    NOTEBOOK_06_FINAL_DIR
    / "AIR_LLM_NOTEBOOK_06_MANIFEST.json"
)


NOTEBOOK_06_BUNDLE = (
    NOTEBOOK_06_FINAL_DIR
    / "air_llm_notebook_06_dataframe_bundle.pkl"
)


NOTEBOOK_06_CONFIG = (
    NOTEBOOK_06_FINAL_DIR
    / "air_llm_06_utility_configuration.json"
)


if not NOTEBOOK_06_MANIFEST.exists():

    raise FileNotFoundError(
        "Notebook 06 manifest not found:\n"
        f"{NOTEBOOK_06_MANIFEST}"
    )


if not NOTEBOOK_06_BUNDLE.exists():

    raise FileNotFoundError(
        "Notebook 06 DataFrame bundle not found:\n"
        f"{NOTEBOOK_06_BUNDLE}"
    )


# --------------------------------------------------------------------------------------------------
# 6. NOTEBOOK 07 OUTPUT DIRECTORIES
# --------------------------------------------------------------------------------------------------

OUTPUT_DIR = (
    NOTEBOOK_07_DIR
)


SELECTION_DIR = (
    OUTPUT_DIR
    / "selection"
)


AUDIT_DIR = (
    OUTPUT_DIR
    / "audit"
)


METADATA_DIR = (
    OUTPUT_DIR
    / "metadata"
)


DIAGNOSTIC_DIR = (
    OUTPUT_DIR
    / "diagnostics"
)


for directory in [

    OUTPUT_DIR,
    SELECTION_DIR,
    AUDIT_DIR,
    METADATA_DIR,
    DIAGNOSTIC_DIR

]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# --------------------------------------------------------------------------------------------------
# 7. RECOVER NOTEBOOK 06 DATAFRAME BUNDLE
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("RECOVERING NOTEBOOK 06 FINAL ARTIFACTS")
print("-" * 100)


with open(
    NOTEBOOK_06_BUNDLE,
    "rb"
) as file:

    AIR_LLM_06_BUNDLE = pickle.load(
        file
    )


if not isinstance(
    AIR_LLM_06_BUNDLE,
    dict
):

    raise RuntimeError(
        "Notebook 06 DataFrame bundle has invalid structure."
    )


for name, dataframe in AIR_LLM_06_BUNDLE.items():

    if isinstance(
        dataframe,
        pd.DataFrame
    ):

        globals()[name] = dataframe.copy()

        print(
            f"RECOVERED {name:<50} "
            f"rows={len(dataframe):,}"
        )


# --------------------------------------------------------------------------------------------------
# 8. VERIFY REQUIRED NOTEBOOK 06 ARTIFACTS
# --------------------------------------------------------------------------------------------------

REQUIRED_06_ARTIFACTS = [

    "AIR_LLM_0621_RESULTS_DF",

    "AIR_LLM_0626_ADAPTIVE_SELECTION_DF",

    "AIR_LLM_0626_STRATEGY_SELECTION_SUMMARY_DF",

    "AIR_LLM_0626_CANDIDATE_UTILITY_DF"

]


missing_06_artifacts = [

    name
    for name in REQUIRED_06_ARTIFACTS

    if (
        name not in globals()
        or
        not isinstance(
            globals()[name],
            pd.DataFrame
        )
    )

]


if missing_06_artifacts:

    raise RuntimeError(
        "Required Notebook 06 artifacts are missing:\n"
        +
        "\n".join(
            f" - {name}"
            for name in missing_06_artifacts
        )
    )


print(
    "\nNotebook 06 recovery : PASS"
)


# --------------------------------------------------------------------------------------------------
# 9. LOAD NOTEBOOK 06 UTILITY CONFIGURATION
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("LOADING NOTEBOOK 06 UTILITY CONFIGURATION")
print("-" * 100)


with open(
    NOTEBOOK_06_CONFIG,
    "r",
    encoding="utf-8"
) as file:

    NOTEBOOK_06_UTILITY_CONFIG = json.load(
        file
    )


# --------------------------------------------------------------------------------------------------
# 10. INHERIT THE SAME UTILITY WEIGHTS USED BY NOTEBOOK 06.26
# --------------------------------------------------------------------------------------------------

UTILITY_WEIGHTS = {
    key: float(value)
    for key, value
    in NOTEBOOK_06_UTILITY_CONFIG[
        "utility_weights"
    ].items()
}


EXPECTED_UTILITY_WEIGHTS = {

    "reconstruction":
        0.30,

    "distribution":
        0.20,

    "dependency":
        0.15,

    "downstream":
        0.20,

    "efficiency":
        0.15

}


if UTILITY_WEIGHTS != EXPECTED_UTILITY_WEIGHTS:

    raise RuntimeError(
        "Notebook 06 utility weights do not match "
        "the expected AIR-LLM configuration."
    )


print(
    "\nUtility configuration inherited from Notebook 06.26:"
)


for key, value in UTILITY_WEIGHTS.items():

    print(
        f"{key:20s}: {value:.3f}"
    )


print(
    f"\nWeight sum          : "
    f"{sum(UTILITY_WEIGHTS.values()):.3f}"
)


# --------------------------------------------------------------------------------------------------
# 11. EXPERIMENTAL CONTEXT DEFINITION
# --------------------------------------------------------------------------------------------------

SELECTION_KEYS = [

    "dataset_id",
    "feature",
    "missingness_mechanism",
    "missingness_rate",
    "seed"

]


# --------------------------------------------------------------------------------------------------
# 12. CONFIDENCE CONFIGURATION
# --------------------------------------------------------------------------------------------------
#
# Confidence in Notebook 07 will be evidence-aware.
#
# It must consider:
#   - utility quality
#   - number of candidate strategies available
#   - utility coverage
#   - selection margin
#   - repeated-run stability
#
# A confidence value must NOT be interpreted as empirical validation when
# only one candidate strategy exists in a context.
#
# --------------------------------------------------------------------------------------------------

CONFIDENCE_THRESHOLD = 0.55

MARGIN_THRESHOLD = 0.05

STABILITY_THRESHOLD = 0.60

MIN_EVIDENCE_ROWS = 2

MIN_CANDIDATES_FOR_CONFIDENT_SELECTION = 2


# --------------------------------------------------------------------------------------------------
# 13. RESEARCH STATUS FLAGS
# --------------------------------------------------------------------------------------------------

NOTEBOOK_06_EMPIRICAL_ROWS = int(
    len(
        AIR_LLM_0621_RESULTS_DF
    )
)


NOTEBOOK_06_CONTEXTS = int(
    AIR_LLM_0621_RESULTS_DF[
        SELECTION_KEYS
    ]
    .drop_duplicates()
    .shape[0]
)


if NOTEBOOK_06_EMPIRICAL_ROWS == NOTEBOOK_06_CONTEXTS:

    CANDIDATE_COVERAGE_WARNING = True

else:

    CANDIDATE_COVERAGE_WARNING = False


# --------------------------------------------------------------------------------------------------
# 14. DISPLAY PROJECT STATE
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("NOTEBOOK 07 PROJECT STATE")
print("=" * 100)


print(
    f"Project root                         : "
    f"{PROJECT_ROOT}"
)


print(
    f"Notebook 06 final archive            : "
    f"{NOTEBOOK_06_FINAL_DIR}"
)


print(
    f"Empirical evaluation rows            : "
    f"{NOTEBOOK_06_EMPIRICAL_ROWS:,}"
)


print(
    f"Experimental contexts                : "
    f"{NOTEBOOK_06_CONTEXTS:,}"
)


if CANDIDATE_COVERAGE_WARNING:

    print(
        "\nWARNING"
    )

    print(
        "Current Notebook 06 empirical results contain "
        "one evaluated row per experimental context."
    )

    print(
        "Therefore Notebook 07 will use conservative "
        "evidence-aware confidence handling."
    )

    print(
        "This condition must not be interpreted as "
        "multi-candidate empirical validation."
    )

else:

    print(
        "\nCandidate coverage check : PASS"
    )


# --------------------------------------------------------------------------------------------------
# 15. DATASET SUMMARY
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("DATASETS")
print("-" * 100)


for dataset_id in DATASETS:

    print(
        f"{dataset_id:20s} | "
        f"Target: {TARGET_REGISTRY[dataset_id]}"
    )


# --------------------------------------------------------------------------------------------------
# 16. NOTEBOOK 07 CONFIGURATION SUMMARY
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("CONFIDENCE CONFIGURATION")
print("-" * 100)


print(
    f"Confidence threshold                 : "
    f"{CONFIDENCE_THRESHOLD:.2f}"
)


print(
    f"Margin threshold                     : "
    f"{MARGIN_THRESHOLD:.2f}"
)


print(
    f"Stability threshold                  : "
    f"{STABILITY_THRESHOLD:.2f}"
)


print(
    f"Minimum evidence rows                : "
    f"{MIN_EVIDENCE_ROWS}"
)


print(
    f"Minimum candidates for confidence    : "
    f"{MIN_CANDIDATES_FOR_CONFIDENT_SELECTION}"
)


# --------------------------------------------------------------------------------------------------
# 17. SAVE NOTEBOOK 07 CONFIGURATION
# --------------------------------------------------------------------------------------------------

NOTEBOOK_07_CONFIGURATION = {

    "project":
        "AIR-LLM",

    "notebook":
        "07",

    "master_seed":
        MASTER_SEED,

    "datasets":
        DATASETS,

    "target_registry":
        TARGET_REGISTRY,

    "selection_keys":
        SELECTION_KEYS,

    "utility_weights":
        UTILITY_WEIGHTS,

    "confidence_threshold":
        CONFIDENCE_THRESHOLD,

    "margin_threshold":
        MARGIN_THRESHOLD,

    "stability_threshold":
        STABILITY_THRESHOLD,

    "minimum_evidence_rows":
        MIN_EVIDENCE_ROWS,

    "minimum_candidates_for_confident_selection":
        MIN_CANDIDATES_FOR_CONFIDENT_SELECTION,

    "notebook_06_final_archive":
        str(NOTEBOOK_06_FINAL_DIR),

    "candidate_coverage_warning":
        CANDIDATE_COVERAGE_WARNING,

    "initialized_at":
        datetime.now(
            timezone.utc
        ).isoformat()

}


CONFIG_PATH = (
    METADATA_DIR
    / "air_llm_07_configuration.json"
)


with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        NOTEBOOK_07_CONFIGURATION,
        file,
        indent=2
    )


# --------------------------------------------------------------------------------------------------
# 18. FINAL INITIALIZATION MESSAGE
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("NOTEBOOK 07.0 INITIALIZATION : COMPLETE")
print("=" * 100)

print(
    f"Notebook 06 artifacts recovered : "
    f"{len(AIR_LLM_06_BUNDLE):,}"
)

print(
    f"Configuration saved              : "
    f"{CONFIG_PATH}"
)

print(
    f"Notebook 07 output directory     : "
    f"{OUTPUT_DIR}"
)

print(
    "\nReady for Notebook 07 confidence-analysis cells."
)

print("=" * 100)

AIR-LLM — NOTEBOOK 07
CONFIDENCE ESTIMATION AND ADAPTIVE SELECTION ANALYSIS

----------------------------------------------------------------------------------------------------
RECOVERING NOTEBOOK 06 FINAL ARTIFACTS
----------------------------------------------------------------------------------------------------
RECOVERED AIR_LLM_0621_RESULTS_DF                            rows=5,775
RECOVERED AIR_LLM_0626_ADAPTIVE_SELECTION_DF                 rows=5,775
RECOVERED AIR_LLM_0626_STRATEGY_SELECTION_SUMMARY_DF         rows=2
RECOVERED AIR_LLM_0626_CANDIDATE_UTILITY_DF                  rows=5,775
RECOVERED STATISTICAL_SIGNIFICANCE_DF                        rows=0
RECOVERED LLM_EVALUATION_DF                                  rows=5,775
RECOVERED LLM_PERFORMANCE_DF                                 rows=1
RECOVERED AIR_LLM_0621_BOUNDED_PLAN_DF                       rows=5,775
RECOVERED EVALUATION_PLAN_DF                                 rows=31,725

Notebook 06 recovery : PASS

---------------

In [4]:
# ==================================================================================================
# AIR-LLM — NOTEBOOK 07.1
# ARTIFACT DISCOVERY AND NOTEBOOK 06 RECOVERY VALIDATION
# ==================================================================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.1")
print("ARTIFACT DISCOVERY AND RECOVERY VALIDATION")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. GENERIC FILE DISCOVERY
# --------------------------------------------------------------------------------------------------

def find_first_existing(candidates):

    for path in candidates:

        if (
            path.exists()
            and path.is_file()
        ):

            return path

    return None


# --------------------------------------------------------------------------------------------------
# 2. NOTEBOOK 04 ARTIFACTS
# --------------------------------------------------------------------------------------------------

FEATURE_PROFILE_PATH = find_first_existing([

    PROFILE_DIR
    / "feature"
    / "feature_profile.csv",

    PROFILE_DIR
    / "feature_profile.csv"

])


MISSINGNESS_PROFILE_PATH = find_first_existing([

    PROFILE_DIR
    / "feature"
    / "missingness_profile.csv",

    PROFILE_DIR
    / "missingness_profile.csv"

])


DATASET_PROFILE_PATH = find_first_existing([

    PROFILE_DIR
    / "dataset"
    / "dataset_profile.csv",

    PROFILE_DIR
    / "dataset_profile.csv"

])


# --------------------------------------------------------------------------------------------------
# 3. NOTEBOOK 05 LLM RECOMMENDATION ARTIFACT
# --------------------------------------------------------------------------------------------------

LLM_RECOMMENDATION_PATH = find_first_existing([

    RECOMMENDATION_DIR
    / "llm_recommendations.csv",

    RECOMMENDATION_DIR
    / "recommendations.csv",

    RECOMMENDATION_DIR
    / "recommendation_summary.csv"

])


# --------------------------------------------------------------------------------------------------
# 4. NOTEBOOK 06 AUTHORITATIVE ARCHIVE
# --------------------------------------------------------------------------------------------------
#
# Notebook 06.27 explicitly created this final persistence directory.
# It is the authoritative source for Notebook 07.
#
# --------------------------------------------------------------------------------------------------

NOTEBOOK_06_FINAL_DIR = (
    PROJECT_ROOT
    / "results"
    / "notebook_06_final"
)


if not NOTEBOOK_06_FINAL_DIR.is_dir():

    raise FileNotFoundError(

        "Notebook 06 final archive was not found:\n"
        f"{NOTEBOOK_06_FINAL_DIR}"

    )


# --------------------------------------------------------------------------------------------------
# 5. NOTEBOOK 06 PERSISTED FILES
# --------------------------------------------------------------------------------------------------

NOTEBOOK_06_RESULTS_PATH = (
    NOTEBOOK_06_FINAL_DIR
    / "air_llm_06_21_candidate_results.csv"
)


NOTEBOOK_06_BOUNDED_PLAN_PATH = (
    NOTEBOOK_06_FINAL_DIR
    / "air_llm_06_21_bounded_evaluation_plan.csv"
)


NOTEBOOK_06_FULL_PLAN_PATH = (
    NOTEBOOK_06_FINAL_DIR
    / "air_llm_06_full_evaluation_plan.csv"
)


NOTEBOOK_06_LLM_EVALUATION_PATH = (
    NOTEBOOK_06_FINAL_DIR
    / "air_llm_06_24_llm_recommendation_evaluation.csv"
)


NOTEBOOK_06_LLM_PERFORMANCE_PATH = (
    NOTEBOOK_06_FINAL_DIR
    / "air_llm_06_24_llm_performance.csv"
)


NOTEBOOK_06_SIGNIFICANCE_PATH = (
    NOTEBOOK_06_FINAL_DIR
    / "air_llm_06_25_statistical_significance.csv"
)


NOTEBOOK_06_UTILITY_PATH = (
    NOTEBOOK_06_FINAL_DIR
    / "air_llm_06_26_candidate_utility_scores.csv"
)


NOTEBOOK_06_SELECTION_PATH = (
    NOTEBOOK_06_FINAL_DIR
    / "air_llm_06_26_adaptive_utility_selection.csv"
)


NOTEBOOK_06_STRATEGY_SUMMARY_PATH = (
    NOTEBOOK_06_FINAL_DIR
    / "air_llm_06_26_strategy_selection_summary.csv"
)


NOTEBOOK_06_BUNDLE_PATH = (
    NOTEBOOK_06_FINAL_DIR
    / "air_llm_notebook_06_dataframe_bundle.pkl"
)


NOTEBOOK_06_MANIFEST_PATH = (
    NOTEBOOK_06_FINAL_DIR
    / "AIR_LLM_NOTEBOOK_06_MANIFEST.json"
)


# --------------------------------------------------------------------------------------------------
# 6. DISCOVER NOTEBOOK 06 EMPIRICAL RESULTS
# --------------------------------------------------------------------------------------------------

if NOTEBOOK_06_RESULTS_PATH.is_file():

    EVALUATION_PATH = (
        NOTEBOOK_06_RESULTS_PATH
    )

else:

    EVALUATION_PATH = None


# --------------------------------------------------------------------------------------------------
# 7. VERIFY PERSISTED NOTEBOOK 06 DATAFRAME
# --------------------------------------------------------------------------------------------------

RESULTS_DF_AVAILABLE = (
    "AIR_LLM_0621_RESULTS_DF" in globals()
    and
    isinstance(
        AIR_LLM_0621_RESULTS_DF,
        pd.DataFrame
    )
)


UTILITY_DF_AVAILABLE = (
    "AIR_LLM_0626_CANDIDATE_UTILITY_DF" in globals()
    and
    isinstance(
        AIR_LLM_0626_CANDIDATE_UTILITY_DF,
        pd.DataFrame
    )
)


SELECTION_DF_AVAILABLE = (
    "AIR_LLM_0626_ADAPTIVE_SELECTION_DF" in globals()
    and
    isinstance(
        AIR_LLM_0626_ADAPTIVE_SELECTION_DF,
        pd.DataFrame
    )
)


# --------------------------------------------------------------------------------------------------
# 8. ARTIFACT STATUS
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("ARTIFACT STATUS")
print("-" * 100)


artifacts = {

    "Feature profile":
        FEATURE_PROFILE_PATH,

    "Missingness profile":
        MISSINGNESS_PROFILE_PATH,

    "Dataset profile":
        DATASET_PROFILE_PATH,

    "LLM recommendations":
        LLM_RECOMMENDATION_PATH,

    "06.21 empirical results":
        NOTEBOOK_06_RESULTS_PATH,

    "06.21 bounded plan":
        NOTEBOOK_06_BOUNDED_PLAN_PATH,

    "06 full evaluation plan":
        NOTEBOOK_06_FULL_PLAN_PATH,

    "06.24 LLM evaluation":
        NOTEBOOK_06_LLM_EVALUATION_PATH,

    "06.24 LLM performance":
        NOTEBOOK_06_LLM_PERFORMANCE_PATH,

    "06.25 significance":
        NOTEBOOK_06_SIGNIFICANCE_PATH,

    "06.26 utility scores":
        NOTEBOOK_06_UTILITY_PATH,

    "06.26 adaptive selection":
        NOTEBOOK_06_SELECTION_PATH,

    "06.26 strategy summary":
        NOTEBOOK_06_STRATEGY_SUMMARY_PATH,

    "06 DataFrame bundle":
        NOTEBOOK_06_BUNDLE_PATH,

    "06 master manifest":
        NOTEBOOK_06_MANIFEST_PATH

}


for name, path in artifacts.items():

    status = (
        "FOUND"
        if path is not None
        and path.exists()
        else "MISSING"
    )

    print(
        f"{name:30s} : {status}"
    )

    if (
        path is not None
        and path.exists()
    ):

        print(
            f"{'':30s}   {path}"
        )


# --------------------------------------------------------------------------------------------------
# 9. REQUIRED ARTIFACT VALIDATION
# --------------------------------------------------------------------------------------------------

required_paths = {

    "Feature profile":
        FEATURE_PROFILE_PATH,

    "Missingness profile":
        MISSINGNESS_PROFILE_PATH,

    "LLM recommendations":
        LLM_RECOMMENDATION_PATH,

    "06.21 empirical results":
        NOTEBOOK_06_RESULTS_PATH,

    "06.26 utility scores":
        NOTEBOOK_06_UTILITY_PATH,

    "06.26 adaptive selection":
        NOTEBOOK_06_SELECTION_PATH,

    "06 DataFrame bundle":
        NOTEBOOK_06_BUNDLE_PATH,

    "06 master manifest":
        NOTEBOOK_06_MANIFEST_PATH

}


missing_required = [

    name

    for name, path
    in required_paths.items()

    if (
        path is None
        or
        not path.exists()
    )

]


if missing_required:

    raise FileNotFoundError(

        "Required AIR-LLM artifacts are missing:\n"
        +
        "\n".join(
            f" - {name}"
            for name in missing_required
        )

    )


# --------------------------------------------------------------------------------------------------
# 10. VALIDATE EMPIRICAL RESULTS
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("VALIDATING NOTEBOOK 06 EMPIRICAL RESULTS")
print("-" * 100)


if not RESULTS_DF_AVAILABLE:

    raise RuntimeError(

        "AIR_LLM_0621_RESULTS_DF is not available "
        "after Notebook 06 recovery."

    )


empirical_rows = int(
    len(
        AIR_LLM_0621_RESULTS_DF
    )
)


successful_rows = int(

    (
        AIR_LLM_0621_RESULTS_DF[
            "status"
        ]
        == "SUCCESS"
    )
    .sum()

    if
    "status"
    in
    AIR_LLM_0621_RESULTS_DF.columns

    else empirical_rows

)


print(
    f"Empirical result rows       : "
    f"{empirical_rows:,}"
)


print(
    f"Successful empirical rows   : "
    f"{successful_rows:,}"
)


# --------------------------------------------------------------------------------------------------
# 11. VALIDATE ADAPTIVE SELECTION ARTIFACT
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("VALIDATING NOTEBOOK 06.26 ADAPTIVE SELECTION")
print("-" * 100)


if not SELECTION_DF_AVAILABLE:

    raise RuntimeError(

        "AIR_LLM_0626_ADAPTIVE_SELECTION_DF "
        "is not available."

    )


selection_rows = int(
    len(
        AIR_LLM_0626_ADAPTIVE_SELECTION_DF
    )
)


print(
    f"Adaptive selection rows    : "
    f"{selection_rows:,}"
)


# --------------------------------------------------------------------------------------------------
# 12. VALIDATE UTILITY ARTIFACT
# --------------------------------------------------------------------------------------------------

if not UTILITY_DF_AVAILABLE:

    raise RuntimeError(

        "AIR_LLM_0626_CANDIDATE_UTILITY_DF "
        "is not available."

    )


utility_rows = int(
    len(
        AIR_LLM_0626_CANDIDATE_UTILITY_DF
    )
)


print(
    f"Candidate utility rows     : "
    f"{utility_rows:,}"
)


# --------------------------------------------------------------------------------------------------
# 13. CANDIDATE COVERAGE DIAGNOSTIC
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("CANDIDATE COVERAGE DIAGNOSTIC")
print("-" * 100)


SELECTION_KEYS = [

    "dataset_id",
    "feature",
    "missingness_mechanism",
    "missingness_rate",
    "seed"

]


required_result_columns = [

    *SELECTION_KEYS,
    "strategy_id"

]


missing_result_columns = [

    column

    for column
    in required_result_columns

    if column
    not in
    AIR_LLM_0621_RESULTS_DF.columns

]


if missing_result_columns:

    raise RuntimeError(

        "Notebook 06 result schema is missing:\n"
        +
        "\n".join(
            f" - {column}"
            for column
            in missing_result_columns
        )

    )


candidate_counts = (

    AIR_LLM_0621_RESULTS_DF

    .groupby(
        SELECTION_KEYS,
        dropna=False
    )

    ["strategy_id"]

    .nunique()

)


unique_contexts = int(
    candidate_counts.shape[0]
)


contexts_with_multiple_candidates = int(

    (
        candidate_counts
        >= 2
    )
    .sum()

)


maximum_candidates = int(
    candidate_counts.max()
)


print(
    f"Experimental contexts              : "
    f"{unique_contexts:,}"
)


print(
    f"Contexts with >=2 candidates       : "
    f"{contexts_with_multiple_candidates:,}"
)


print(
    f"Maximum candidates/context         : "
    f"{maximum_candidates:,}"
)


if contexts_with_multiple_candidates == 0:

    print(
        "\nWARNING: "
        "No experimental context currently has "
        "more than one evaluated candidate strategy."
    )

    print(
        "Notebook 07 will therefore apply "
        "conservative confidence interpretation."
    )

else:

    print(
        "\nCandidate diversity : PRESENT"
    )


# --------------------------------------------------------------------------------------------------
# 14. MANIFEST VALIDATION
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("NOTEBOOK 06 MANIFEST VALIDATION")
print("-" * 100)


with open(
    NOTEBOOK_06_MANIFEST_PATH,
    "r",
    encoding="utf-8"
) as file:

    NOTEBOOK_06_MANIFEST = json.load(
        file
    )


print(
    "Notebook 06 master manifest : PASS"
)


# --------------------------------------------------------------------------------------------------
# 15. FINAL DISCOVERY SUMMARY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("NOTEBOOK 07.1 ARTIFACT DISCOVERY : COMPLETE")
print("=" * 100)


print(
    f"Notebook 04 feature profile       : "
    f"{'FOUND' if FEATURE_PROFILE_PATH else 'MISSING'}"
)


print(
    f"Notebook 05 LLM recommendations   : "
    f"{'FOUND' if LLM_RECOMMENDATION_PATH else 'MISSING'}"
)


print(
    f"Notebook 06 empirical results     : "
    f"{empirical_rows:,} rows"
)


print(
    f"Notebook 06 utility results       : "
    f"{utility_rows:,} rows"
)


print(
    f"Notebook 06 adaptive selections   : "
    f"{selection_rows:,} rows"
)


print(
    f"Notebook 06 final archive         : "
    f"{NOTEBOOK_06_FINAL_DIR}"
)


print(
    "\nAll required upstream artifacts "
    "are available for Notebook 07."
)


print("=" * 100)

AIR-LLM — NOTEBOOK 07.1
ARTIFACT DISCOVERY AND RECOVERY VALIDATION

----------------------------------------------------------------------------------------------------
ARTIFACT STATUS
----------------------------------------------------------------------------------------------------
Feature profile                : FOUND
                                 /content/drive/MyDrive/AIR_LLM_Research/data/notebook_04/profiles/feature/feature_profile.csv
Missingness profile            : FOUND
                                 /content/drive/MyDrive/AIR_LLM_Research/data/notebook_04/profiles/feature/missingness_profile.csv
Dataset profile                : FOUND
                                 /content/drive/MyDrive/AIR_LLM_Research/data/notebook_04/profiles/dataset/dataset_profile.csv
LLM recommendations            : FOUND
                                 /content/drive/MyDrive/AIR_LLM_Research/data/notebook_05/recommendations/llm_recommendations.csv
06.21 empirical results        : FOUND
    

In [6]:
# ==================================================================================================
# AIR-LLM — NOTEBOOK 07.2
# LOAD INPUT ARTIFACTS AND VALIDATE RESEARCH EVIDENCE
# ==================================================================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.2")
print("LOADING PROFILES AND EMPIRICAL EVIDENCE")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. LOAD NOTEBOOK 04 PROFILES
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("LOADING NOTEBOOK 04 PROFILES")
print("-" * 100)


FEATURE_PROFILE_DF = pd.read_csv(
    FEATURE_PROFILE_PATH
)


MISSINGNESS_PROFILE_DF = pd.read_csv(
    MISSINGNESS_PROFILE_PATH
)


if DATASET_PROFILE_PATH:

    DATASET_PROFILE_DF = pd.read_csv(
        DATASET_PROFILE_PATH
    )

else:

    DATASET_PROFILE_DF = pd.DataFrame()


print(
    f"Feature profile rows       : "
    f"{len(FEATURE_PROFILE_DF):,}"
)


print(
    f"Missingness profile rows   : "
    f"{len(MISSINGNESS_PROFILE_DF):,}"
)


print(
    f"Dataset profile rows       : "
    f"{len(DATASET_PROFILE_DF):,}"
)


# --------------------------------------------------------------------------------------------------
# 2. USE PERSISTED NOTEBOOK 06 DATAFRAME AS PRIMARY EVIDENCE
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("LOADING NOTEBOOK 06 EMPIRICAL EVIDENCE")
print("-" * 100)


if (
    "AIR_LLM_0621_RESULTS_DF"
    not in globals()
):

    raise RuntimeError(

        "AIR_LLM_0621_RESULTS_DF is not available. "
        "Notebook 06 recovery from 07.0/07.1 must "
        "complete before running 07.2."

    )


EVALUATION_DF = (
    AIR_LLM_0621_RESULTS_DF
    .copy()
)


print(
    f"Recovered empirical rows  : "
    f"{len(EVALUATION_DF):,}"
)


# --------------------------------------------------------------------------------------------------
# 3. VALIDATE EMPIRICAL RESULT SCHEMA
# --------------------------------------------------------------------------------------------------

REQUIRED_EVALUATION_COLUMNS = [

    "dataset_id",
    "feature",
    "missingness_mechanism",
    "missingness_rate",
    "seed",
    "strategy_id",
    "status"

]


missing_evaluation_columns = [

    column

    for column
    in REQUIRED_EVALUATION_COLUMNS

    if column
    not in EVALUATION_DF.columns

]


if missing_evaluation_columns:

    raise RuntimeError(

        "Notebook 06 empirical evidence is missing "
        "required columns:\n"
        +
        "\n".join(
            f" - {column}"
            for column
            in missing_evaluation_columns
        )

    )


# --------------------------------------------------------------------------------------------------
# 4. KEEP ONLY SUCCESSFUL EMPIRICAL EVIDENCE
# --------------------------------------------------------------------------------------------------

SUCCESS_EVALUATION_DF = (

    EVALUATION_DF[

        EVALUATION_DF[
            "status"
        ]
        == "SUCCESS"

    ]

    .copy()

)


if SUCCESS_EVALUATION_DF.empty:

    raise RuntimeError(
        "No successful empirical evaluations were found."
    )


print(
    f"Successful empirical rows : "
    f"{len(SUCCESS_EVALUATION_DF):,}"
)


# --------------------------------------------------------------------------------------------------
# 5. LOAD LLM RECOMMENDATIONS
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("LOADING LLM RECOMMENDATIONS")
print("-" * 100)


if LLM_RECOMMENDATION_PATH:

    LLM_RECOMMENDATION_DF = pd.read_csv(
        LLM_RECOMMENDATION_PATH
    )

else:

    LLM_RECOMMENDATION_DF = (
        pd.DataFrame()
    )


print(
    f"LLM recommendation rows    : "
    f"{len(LLM_RECOMMENDATION_DF):,}"
)


# --------------------------------------------------------------------------------------------------
# 6. VALIDATE LLM RECOMMENDATION AVAILABILITY
# --------------------------------------------------------------------------------------------------

LLM_RECOMMENDATION_AVAILABLE = (
    not LLM_RECOMMENDATION_DF.empty
)


if LLM_RECOMMENDATION_AVAILABLE:

    print(
        "LLM recommendation evidence : AVAILABLE"
    )

else:

    print(
        "LLM recommendation evidence : NOT AVAILABLE"
    )


# --------------------------------------------------------------------------------------------------
# 7. INSPECT LLM RECOMMENDATION SCHEMA
# --------------------------------------------------------------------------------------------------

if LLM_RECOMMENDATION_AVAILABLE:

    print("\nLLM recommendation columns:")
    print(
        list(
            LLM_RECOMMENDATION_DF.columns
        )
    )


# --------------------------------------------------------------------------------------------------
# 8. NORMALIZE COMMON COLUMN NAMES
# --------------------------------------------------------------------------------------------------

if LLM_RECOMMENDATION_AVAILABLE:

    rename_map = {}

    if (
        "recommended_strategy"
        in
        LLM_RECOMMENDATION_DF.columns
    ):

        rename_map[
            "recommended_strategy"
        ] = "llm_recommended"


    elif (
        "recommendation"
        in
        LLM_RECOMMENDATION_DF.columns
    ):

        rename_map[
            "recommendation"
        ] = "llm_recommended"


    elif (
        "strategy_id"
        in
        LLM_RECOMMENDATION_DF.columns
        and
        "llm_recommended"
        not in
        LLM_RECOMMENDATION_DF.columns
    ):

        rename_map[
            "strategy_id"
        ] = "llm_recommended"


    if rename_map:

        LLM_RECOMMENDATION_DF = (
            LLM_RECOMMENDATION_DF
            .rename(
                columns=rename_map
            )
        )


# --------------------------------------------------------------------------------------------------
# 9. IDENTIFY LLM RECOMMENDATION COLUMN
# --------------------------------------------------------------------------------------------------

LLM_RECOMMENDATION_COLUMN = None


if LLM_RECOMMENDATION_AVAILABLE:

    if (
        "llm_recommended"
        in
        LLM_RECOMMENDATION_DF.columns
    ):

        LLM_RECOMMENDATION_COLUMN = (
            "llm_recommended"
        )


if LLM_RECOMMENDATION_COLUMN:

    print(
        f"\nLLM recommendation column : "
        f"{LLM_RECOMMENDATION_COLUMN}"
    )

else:

    print(
        "\nWARNING: No normalized LLM recommendation "
        "column was identified."
    )


# --------------------------------------------------------------------------------------------------
# 10. EXPERIMENTAL CONTEXT KEYS
# --------------------------------------------------------------------------------------------------

SELECTION_KEYS = [

    "dataset_id",
    "feature",
    "missingness_mechanism",
    "missingness_rate",
    "seed"

]


# --------------------------------------------------------------------------------------------------
# 11. EMPIRICAL CONTEXT SUMMARY
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("EMPIRICAL CONTEXT SUMMARY")
print("-" * 100)


EMPIRICAL_CONTEXT_DF = (

    SUCCESS_EVALUATION_DF[
        SELECTION_KEYS
    ]

    .drop_duplicates()

    .reset_index(
        drop=True
    )

)


print(
    f"Successful empirical rows : "
    f"{len(SUCCESS_EVALUATION_DF):,}"
)


print(
    f"Unique experimental contexts : "
    f"{len(EMPIRICAL_CONTEXT_DF):,}"
)


# --------------------------------------------------------------------------------------------------
# 12. CANDIDATE COVERAGE
# --------------------------------------------------------------------------------------------------

CANDIDATE_COUNT_DF = (

    SUCCESS_EVALUATION_DF

    .groupby(
        SELECTION_KEYS,
        dropna=False
    )

    ["strategy_id"]

    .nunique()

    .reset_index(
        name="candidate_count"
    )

)


print(
    f"Maximum candidates/context : "
    f"{CANDIDATE_COUNT_DF['candidate_count'].max():,}"
)


print(
    f"Contexts with >=2 candidates : "
    f"{int((CANDIDATE_COUNT_DF['candidate_count'] >= 2).sum()):,}"
)


# --------------------------------------------------------------------------------------------------
# 13. ARCHIVAL CSV CROSS-CHECK
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("ARCHIVAL CSV CROSS-CHECK")
print("-" * 100)


if EVALUATION_PATH.exists():

    archived_evaluation_df = pd.read_csv(
        EVALUATION_PATH,
        usecols=[
            column
            for column
            in REQUIRED_EVALUATION_COLUMNS
            if column
            in pd.read_csv(
                EVALUATION_PATH,
                nrows=0
            ).columns
        ]
    )


    archived_rows = len(
        archived_evaluation_df
    )


    print(
        f"Persisted CSV rows        : "
        f"{archived_rows:,}"
    )


    print(
        f"Recovered DataFrame rows  : "
        f"{len(EVALUATION_DF):,}"
    )


    if archived_rows != len(
        EVALUATION_DF
    ):

        raise RuntimeError(

            "Notebook 06 DataFrame and persisted CSV "
            "row counts do not match."

        )


    print(
        "DataFrame / CSV row count : PASS"
    )


# --------------------------------------------------------------------------------------------------
# 14. FINAL LOADING SUMMARY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("NOTEBOOK 07.2 INPUT LOADING : COMPLETE")
print("=" * 100)


print(
    f"Feature profiles             : "
    f"{len(FEATURE_PROFILE_DF):,}"
)


print(
    f"Missingness profiles         : "
    f"{len(MISSINGNESS_PROFILE_DF):,}"
)


print(
    f"Dataset profiles             : "
    f"{len(DATASET_PROFILE_DF):,}"
)


print(
    f"Empirical evidence           : "
    f"{len(EVALUATION_DF):,}"
)


print(
    f"Successful empirical rows    : "
    f"{len(SUCCESS_EVALUATION_DF):,}"
)


print(
    f"Experimental contexts        : "
    f"{len(EMPIRICAL_CONTEXT_DF):,}"
)


print(
    f"LLM recommendations          : "
    f"{len(LLM_RECOMMENDATION_DF):,}"
)


print(
    f"LLM recommendation column    : "
    f"{LLM_RECOMMENDATION_COLUMN}"
)


if (
    len(CANDIDATE_COUNT_DF)
    ==
    len(EMPIRICAL_CONTEXT_DF)
):

    print(
        "Context/candidate diagnostic : PASS"
    )


print("=" * 100)

AIR-LLM — NOTEBOOK 07.2
LOADING PROFILES AND EMPIRICAL EVIDENCE

----------------------------------------------------------------------------------------------------
LOADING NOTEBOOK 04 PROFILES
----------------------------------------------------------------------------------------------------
Feature profile rows       : 80
Missingness profile rows   : 80
Dataset profile rows       : 3

----------------------------------------------------------------------------------------------------
LOADING NOTEBOOK 06 EMPIRICAL EVIDENCE
----------------------------------------------------------------------------------------------------
Recovered empirical rows  : 5,775
Successful empirical rows : 5,775

----------------------------------------------------------------------------------------------------
LOADING LLM RECOMMENDATIONS
----------------------------------------------------------------------------------------------------
LLM recommendation rows    : 55
LLM recommendation evidence : AVAILA

In [8]:
# ==================================================================================================
# AIR-LLM — NOTEBOOK 07.3
# INPUT SCHEMA NORMALIZATION AND VALIDATION
# ==================================================================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.3")
print("INPUT SCHEMA NORMALIZATION AND VALIDATION")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. GENERIC COLUMN NORMALIZATION
# --------------------------------------------------------------------------------------------------

def normalize_columns(df):

    df = df.copy()

    df.columns = [

        str(column)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")

        for column
        in df.columns

    ]

    return df


# --------------------------------------------------------------------------------------------------
# 2. NORMALIZE ALL INPUT DATAFRAMES
# --------------------------------------------------------------------------------------------------

FEATURE_PROFILE_DF = normalize_columns(
    FEATURE_PROFILE_DF
)


MISSINGNESS_PROFILE_DF = normalize_columns(
    MISSINGNESS_PROFILE_DF
)


DATASET_PROFILE_DF = normalize_columns(
    DATASET_PROFILE_DF
)


EVALUATION_DF = normalize_columns(
    EVALUATION_DF
)


LLM_RECOMMENDATION_DF = normalize_columns(
    LLM_RECOMMENDATION_DF
)


# --------------------------------------------------------------------------------------------------
# 3. REQUIRED EMPIRICAL RESULT SCHEMA
# --------------------------------------------------------------------------------------------------

REQUIRED_EVALUATION_COLUMNS = [

    "evaluation_key",

    "dataset_id",

    "feature",

    "target",

    "feature_type",

    "strategy_id",

    "missingness_mechanism",

    "missingness_rate",

    "repetition",

    "seed",

    "llm_recommended",

    "status",

    "mae",

    "rmse",

    "r2",

    "accuracy",

    "macro_f1",

    "weighted_f1",

    "n",

    "runtime_seconds"

]


missing_evaluation_columns = [

    column

    for column
    in REQUIRED_EVALUATION_COLUMNS

    if column
    not in EVALUATION_DF.columns

]


if missing_evaluation_columns:

    raise RuntimeError(

        "Required empirical-result columns are missing:\n"

        +
        "\n".join(

            f" - {column}"

            for column
            in missing_evaluation_columns

        )

    )


print("\nEMPIRICAL SCHEMA")
print("-" * 100)

print(
    "Required columns : PASS"
)


print(
    f"Total columns    : "
    f"{len(EVALUATION_DF.columns):,}"
)


print(
    f"Total rows       : "
    f"{len(EVALUATION_DF):,}"
)


# --------------------------------------------------------------------------------------------------
# 4. NORMALIZE STRING FIELDS
# --------------------------------------------------------------------------------------------------

STRING_COLUMNS = [

    "evaluation_key",

    "dataset_id",

    "feature",

    "target",

    "feature_type",

    "strategy_id",

    "missingness_mechanism",

    "status",

    "llm_recommended"

]


for column in STRING_COLUMNS:

    if column in EVALUATION_DF.columns:

        EVALUATION_DF[column] = (

            EVALUATION_DF[column]

            .astype("string")

            .str.strip()

        )


# --------------------------------------------------------------------------------------------------
# 5. NORMALIZE LLM RECOMMENDATION VALUES
# --------------------------------------------------------------------------------------------------
#
# The empirical results already contain llm_recommended.
# This is the authoritative recommendation attached to the
# evaluated experimental configuration.
#
# Do NOT merge the 55-row Notebook 05 recommendation file
# into the empirical results here.
#
# --------------------------------------------------------------------------------------------------

EVALUATION_DF[
    "llm_recommended"
] = (

    EVALUATION_DF[
        "llm_recommended"
    ]

    .astype("string")

    .str.strip()

)


EVALUATION_DF[
    "strategy_id"
] = (

    EVALUATION_DF[
        "strategy_id"
    ]

    .astype("string")

    .str.strip()

)


# --------------------------------------------------------------------------------------------------
# 6. NORMALIZE MISSINGNESS MECHANISM
# --------------------------------------------------------------------------------------------------

EVALUATION_DF[
    "missingness_mechanism"
] = (

    EVALUATION_DF[
        "missingness_mechanism"
    ]

    .astype("string")

    .str.strip()
    .str.upper()

)


# --------------------------------------------------------------------------------------------------
# 7. NORMALIZE NUMERIC FIELDS
# --------------------------------------------------------------------------------------------------

NUMERIC_COLUMNS = [

    "missingness_rate",

    "repetition",

    "seed",

    "mae",

    "rmse",

    "r2",

    "accuracy",

    "macro_f1",

    "weighted_f1",

    "n",

    "runtime_seconds"

]


numeric_conversion_report = []


for column in NUMERIC_COLUMNS:

    if column not in EVALUATION_DF.columns:

        continue


    original_non_null = (
        EVALUATION_DF[column]
        .notna()
        .sum()
    )


    EVALUATION_DF[column] = pd.to_numeric(

        EVALUATION_DF[column],

        errors="coerce"

    )


    converted_non_null = (
        EVALUATION_DF[column]
        .notna()
        .sum()
    )


    numeric_conversion_report.append({

        "column":
            column,

        "original_non_null":
            int(original_non_null),

        "converted_non_null":
            int(converted_non_null),

        "conversion_loss":
            int(
                original_non_null
                -
                converted_non_null
            )

    })


NUMERIC_CONVERSION_DF = pd.DataFrame(
    numeric_conversion_report
)


# --------------------------------------------------------------------------------------------------
# 8. NORMALIZE DATASET IDENTIFIERS
# --------------------------------------------------------------------------------------------------

EVALUATION_DF[
    "dataset_id"
] = (

    EVALUATION_DF[
        "dataset_id"
    ]

    .astype("string")

    .str.strip()
    .str.lower()

)


# --------------------------------------------------------------------------------------------------
# 9. NORMALIZE FEATURE IDENTIFIERS
# --------------------------------------------------------------------------------------------------

EVALUATION_DF[
    "feature"
] = (

    EVALUATION_DF[
        "feature"
    ]

    .astype("string")

    .str.strip()

)


# --------------------------------------------------------------------------------------------------
# 10. VALIDATE DATASET REGISTRY
# --------------------------------------------------------------------------------------------------

unexpected_datasets = sorted(

    set(
        EVALUATION_DF[
            "dataset_id"
        ]
        .dropna()
        .unique()
    )

    -
    set(
        DATASETS
    )

)


if unexpected_datasets:

    raise RuntimeError(

        "Unexpected dataset identifiers found:\n"

        +
        "\n".join(

            f" - {dataset}"

            for dataset
            in unexpected_datasets

        )

    )


print("\nDATASET VALIDATION")
print("-" * 100)

print(
    "Dataset identifiers : PASS"
)


# --------------------------------------------------------------------------------------------------
# 11. VALIDATE MISSINGNESS MECHANISMS
# --------------------------------------------------------------------------------------------------

VALID_MECHANISMS = {

    "MCAR",
    "MAR",
    "MNAR"

}


observed_mechanisms = set(

    EVALUATION_DF[
        "missingness_mechanism"
    ]
    .dropna()
    .unique()

)


unexpected_mechanisms = (

    observed_mechanisms
    -
    VALID_MECHANISMS

)


if unexpected_mechanisms:

    raise RuntimeError(

        "Unexpected missingness mechanisms found:\n"

        +
        "\n".join(

            f" - {mechanism}"

            for mechanism
            in sorted(
                unexpected_mechanisms
            )

        )

    )


print(
    "Missingness mechanisms : PASS"
)


# --------------------------------------------------------------------------------------------------
# 12. VALIDATE MISSINGNESS RATES
# --------------------------------------------------------------------------------------------------

observed_rates = (

    EVALUATION_DF[
        "missingness_rate"
    ]

    .dropna()

    .unique()

)


invalid_rates = [

    float(rate)

    for rate
    in observed_rates

    if (
        rate <= 0
        or
        rate >= 1
    )

]


if invalid_rates:

    raise RuntimeError(

        "Invalid missingness rates found:\n"

        +
        "\n".join(

            f" - {rate}"

            for rate
            in invalid_rates

        )

    )


print(
    "Missingness rates : PASS"
)


# --------------------------------------------------------------------------------------------------
# 13. VALIDATE STATUS VALUES
# --------------------------------------------------------------------------------------------------

VALID_STATUSES = {

    "SUCCESS",
    "FAILED"

}


observed_statuses = set(

    EVALUATION_DF[
        "status"
    ]
    .dropna()
    .astype("string")
    .str.upper()
    .unique()

)


unexpected_statuses = (

    observed_statuses
    -
    VALID_STATUSES

)


if unexpected_statuses:

    raise RuntimeError(

        "Unexpected evaluation status values found:\n"

        +
        "\n".join(

            f" - {status}"

            for status
            in sorted(
                unexpected_statuses
            )

        )

    )


EVALUATION_DF[
    "status"
] = (

    EVALUATION_DF[
        "status"
    ]

    .astype("string")

    .str.upper()

)


print(
    "Evaluation status : PASS"
)


# --------------------------------------------------------------------------------------------------
# 14. VALIDATE LLM RECOMMENDATION COVERAGE
# --------------------------------------------------------------------------------------------------

llm_missing_mask = (

    EVALUATION_DF[
        "llm_recommended"
    ]
    .isna()

    |
    (
        EVALUATION_DF[
            "llm_recommended"
        ]
        .astype("string")
        .str.strip()
        .eq("")
    )

)


LLM_MISSING_ROWS = int(
    llm_missing_mask.sum()
)


LLM_AVAILABLE_ROWS = int(
    len(EVALUATION_DF)
    -
    LLM_MISSING_ROWS
)


print("\nLLM RECOMMENDATION COVERAGE")
print("-" * 100)

print(
    f"Rows with recommendation : "
    f"{LLM_AVAILABLE_ROWS:,}"
)


print(
    f"Rows without recommendation : "
    f"{LLM_MISSING_ROWS:,}"
)


if LLM_AVAILABLE_ROWS == 0:

    raise RuntimeError(

        "No LLM recommendations are attached "
        "to the empirical results."

    )


# --------------------------------------------------------------------------------------------------
# 15. LLM RECOMMENDATION VALUE DISTRIBUTION
# --------------------------------------------------------------------------------------------------

LLM_RECOMMENDATION_COUNTS_DF = (

    EVALUATION_DF[
        "llm_recommended"
    ]

    .value_counts(
        dropna=False
    )

    .rename_axis(
        "llm_recommended"
    )

    .reset_index(
        name="count"
    )

)


print(
    "\nLLM recommendation distribution:"
)


display(
    LLM_RECOMMENDATION_COUNTS_DF
)


# --------------------------------------------------------------------------------------------------
# 16. EMPIRICAL STRATEGY DISTRIBUTION
# --------------------------------------------------------------------------------------------------

EMPIRICAL_STRATEGY_COUNTS_DF = (

    EVALUATION_DF[
        "strategy_id"
    ]

    .value_counts(
        dropna=False
    )

    .rename_axis(
        "strategy_id"
    )

    .reset_index(
        name="count"
    )

)


print(
    "\nEmpirical strategy distribution:"
)


display(
    EMPIRICAL_STRATEGY_COUNTS_DF
)


# --------------------------------------------------------------------------------------------------
# 17. CREATE SUCCESSFUL NORMALIZED DATASET
# --------------------------------------------------------------------------------------------------

SUCCESS_EVALUATION_DF = (

    EVALUATION_DF[

        EVALUATION_DF[
            "status"
        ]
        == "SUCCESS"

    ]

    .copy()

)


if SUCCESS_EVALUATION_DF.empty:

    raise RuntimeError(

        "No successful empirical evaluations remain "
        "after schema normalization."

    )


# --------------------------------------------------------------------------------------------------
# 18. EXPERIMENTAL CONTEXT KEYS
# --------------------------------------------------------------------------------------------------

SELECTION_KEYS = [

    "dataset_id",

    "feature",

    "missingness_mechanism",

    "missingness_rate",

    "seed"

]


# --------------------------------------------------------------------------------------------------
# 19. CONTEXT AND CANDIDATE COVERAGE
# --------------------------------------------------------------------------------------------------

CONTEXT_CANDIDATE_COUNTS_DF = (

    SUCCESS_EVALUATION_DF

    .groupby(
        SELECTION_KEYS,
        dropna=False
    )

    ["strategy_id"]

    .nunique()

    .reset_index(
        name="candidate_count"
    )

)


UNIQUE_CONTEXTS = int(

    len(
        CONTEXT_CANDIDATE_COUNTS_DF
    )

)


MULTI_CANDIDATE_CONTEXTS = int(

    (
        CONTEXT_CANDIDATE_COUNTS_DF[
            "candidate_count"
        ]
        >= 2
    )
    .sum()

)


MAX_CANDIDATES_PER_CONTEXT = int(

    CONTEXT_CANDIDATE_COUNTS_DF[
        "candidate_count"
    ]
    .max()

)


print("\nCANDIDATE COVERAGE")
print("-" * 100)

print(
    f"Unique contexts              : "
    f"{UNIQUE_CONTEXTS:,}"
)


print(
    f"Contexts with >=2 candidates : "
    f"{MULTI_CANDIDATE_CONTEXTS:,}"
)


print(
    f"Maximum candidates/context   : "
    f"{MAX_CANDIDATES_PER_CONTEXT:,}"
)


if MULTI_CANDIDATE_CONTEXTS == 0:

    print(
        "\nWARNING: Current empirical evidence "
        "contains one evaluated candidate per context."
    )

    print(
        "This is recorded as an evidence-coverage "
        "limitation for downstream confidence analysis."
    )


# --------------------------------------------------------------------------------------------------
# 20. FINAL SCHEMA SUMMARY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("NOTEBOOK 07.3 INPUT SCHEMA NORMALIZATION : COMPLETE")
print("=" * 100)


print(
    f"Evaluation rows              : "
    f"{len(EVALUATION_DF):,}"
)


print(
    f"Successful evaluation rows   : "
    f"{len(SUCCESS_EVALUATION_DF):,}"
)


print(
    f"LLM recommendation coverage  : "
    f"{LLM_AVAILABLE_ROWS / len(EVALUATION_DF):.2%}"
)


print(
    f"Unique experimental contexts : "
    f"{UNIQUE_CONTEXTS:,}"
)


print(
    f"Multi-candidate contexts     : "
    f"{MULTI_CANDIDATE_CONTEXTS:,}"
)


print(
    "Schema validation             : PASS"
)


print("=" * 100)

AIR-LLM — NOTEBOOK 07.3
INPUT SCHEMA NORMALIZATION AND VALIDATION

EMPIRICAL SCHEMA
----------------------------------------------------------------------------------------------------
Required columns : PASS
Total columns    : 22
Total rows       : 5,775

DATASET VALIDATION
----------------------------------------------------------------------------------------------------
Dataset identifiers : PASS
Missingness mechanisms : PASS
Missingness rates : PASS
Evaluation status : PASS

LLM RECOMMENDATION COVERAGE
----------------------------------------------------------------------------------------------------
Rows with recommendation : 5,775
Rows without recommendation : 0

LLM recommendation distribution:


,llm_recommended,count
0,False,5775



Empirical strategy distribution:


,strategy_id,count
0,mode,3975
1,mean,1800



CANDIDATE COVERAGE
----------------------------------------------------------------------------------------------------
Unique contexts              : 5,775
Contexts with >=2 candidates : 0
Maximum candidates/context   : 1

This is recorded as an evidence-coverage limitation for downstream confidence analysis.

NOTEBOOK 07.3 INPUT SCHEMA NORMALIZATION : COMPLETE
Evaluation rows              : 5,775
Successful evaluation rows   : 5,775
LLM recommendation coverage  : 100.00%
Unique experimental contexts : 5,775
Multi-candidate contexts     : 0
Schema validation             : PASS


In [10]:
# ==================================================================================================
# AIR-LLM — NOTEBOOK 07.4
# EVALUATION SCHEMA ADAPTER
# ==================================================================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.4")
print("EVALUATION SCHEMA ADAPTER")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. NOTEBOOK 06.21 EMPIRICAL SCHEMA
# --------------------------------------------------------------------------------------------------

EXPECTED_0621_COLUMNS = [

    "evaluation_key",

    "dataset_id",

    "feature",

    "target",

    "feature_type",

    "strategy_id",

    "missingness_mechanism",

    "missingness_rate",

    "repetition",

    "seed",

    "llm_recommended",

    "status",

    "mae",

    "rmse",

    "r2",

    "accuracy",

    "macro_f1",

    "weighted_f1",

    "n",

    "runtime_seconds"

]


missing_0621_columns = [

    column

    for column
    in EXPECTED_0621_COLUMNS

    if column
    not in EVALUATION_DF.columns

]


if missing_0621_columns:

    raise RuntimeError(

        "Notebook 06.21 empirical schema validation failed.\n\n"

        "Missing columns:\n"

        +
        "\n".join(

            f" - {column}"

            for column
            in missing_0621_columns

        )

    )


print("\nNOTEBOOK 06.21 SCHEMA")
print("-" * 100)

print(
    "06.21 empirical schema : PASS"
)


print(
    f"Rows                  : "
    f"{len(EVALUATION_DF):,}"
)


print(
    f"Columns               : "
    f"{len(EVALUATION_DF.columns):,}"
)


# --------------------------------------------------------------------------------------------------
# 2. CREATE EXPLICIT AIR-LLM STANDARD COLUMN MAP
# --------------------------------------------------------------------------------------------------
#
# IMPORTANT:
#
# strategy_id is the actual strategy identifier in Notebook 06.
#
# The generic 07.4 adapter previously searched only for "strategy",
# which caused the failure:
#
#     strategy -> NOT FOUND
#
# We therefore map directly to the authoritative Notebook 06 schema.
#
# --------------------------------------------------------------------------------------------------

COLUMN_MAP = {

    "dataset_id":
        "dataset_id",

    "feature":
        "feature",

    "strategy":
        "strategy_id",

    "mechanism":
        "missingness_mechanism",

    "rate":
        "missingness_rate",

    "repetition":
        "repetition",

    "seed":
        "seed",

    "llm_recommended":
        "llm_recommended",

    "status":
        "status",

    "mae":
        "mae",

    "rmse":
        "rmse",

    "r2":
        "r2",

    "accuracy":
        "accuracy",

    "macro_f1":
        "macro_f1",

    "weighted_f1":
        "weighted_f1",

    "efficiency":
        "runtime_seconds"

}


print("\nAIR-LLM COLUMN MAPPING")
print("-" * 100)

for key, value in COLUMN_MAP.items():

    print(
        f"{key:20s} -> {value}"
    )


# --------------------------------------------------------------------------------------------------
# 3. CREATE STANDARDIZED VIEW
# --------------------------------------------------------------------------------------------------

EVALUATION_STANDARD_DF = EVALUATION_DF.rename(

    columns={

        source:
        target

        for target, source
        in COLUMN_MAP.items()

    }

).copy()


# --------------------------------------------------------------------------------------------------
# 4. VALIDATE STANDARDIZED SCHEMA
# --------------------------------------------------------------------------------------------------

REQUIRED_STANDARD_COLUMNS = [

    "dataset_id",

    "feature",

    "strategy",

    "mechanism",

    "rate",

    "repetition",

    "seed",

    "llm_recommended",

    "status",

    "mae",

    "rmse",

    "r2",

    "accuracy",

    "macro_f1",

    "weighted_f1",

    "efficiency"

]


missing_standard_columns = [

    column

    for column
    in REQUIRED_STANDARD_COLUMNS

    if column
    not in EVALUATION_STANDARD_DF.columns

]


if missing_standard_columns:

    raise RuntimeError(

        "Standardized AIR-LLM evaluation schema is incomplete:\n"

        +
        "\n".join(

            f" - {column}"

            for column
            in missing_standard_columns

        )

    )


print("\nSTANDARDIZED SCHEMA")
print("-" * 100)

print(
    "AIR-LLM standardized schema : PASS"
)


# --------------------------------------------------------------------------------------------------
# 5. NORMALIZE STRATEGY IDENTIFIER
# --------------------------------------------------------------------------------------------------

EVALUATION_STANDARD_DF[
    "strategy"
] = (

    EVALUATION_STANDARD_DF[
        "strategy"
    ]

    .astype("string")

    .str.strip()

)


# --------------------------------------------------------------------------------------------------
# 6. NORMALIZE LLM RECOMMENDATION
# --------------------------------------------------------------------------------------------------

EVALUATION_STANDARD_DF[
    "llm_recommended"
] = (

    EVALUATION_STANDARD_DF[
        "llm_recommended"
    ]

    .astype("string")

    .str.strip()

)


# --------------------------------------------------------------------------------------------------
# 7. NORMALIZE MECHANISM
# --------------------------------------------------------------------------------------------------

EVALUATION_STANDARD_DF[
    "mechanism"
] = (

    EVALUATION_STANDARD_DF[
        "mechanism"
    ]

    .astype("string")

    .str.strip()

    .str.upper()

)


# --------------------------------------------------------------------------------------------------
# 8. NUMERIC STANDARDIZATION
# --------------------------------------------------------------------------------------------------

STANDARD_NUMERIC_COLUMNS = [

    "rate",

    "repetition",

    "seed",

    "mae",

    "rmse",

    "r2",

    "accuracy",

    "macro_f1",

    "weighted_f1",

    "efficiency"

]


for column in STANDARD_NUMERIC_COLUMNS:

    EVALUATION_STANDARD_DF[
        column
    ] = pd.to_numeric(

        EVALUATION_STANDARD_DF[
            column
        ],

        errors="coerce"

    )


# --------------------------------------------------------------------------------------------------
# 9. SUCCESSFUL EVALUATIONS
# --------------------------------------------------------------------------------------------------

SUCCESS_STANDARD_DF = (

    EVALUATION_STANDARD_DF[

        EVALUATION_STANDARD_DF[
            "status"
        ]
        .astype("string")
        .str.upper()
        .eq("SUCCESS")

    ]

    .copy()

)


if SUCCESS_STANDARD_DF.empty:

    raise RuntimeError(

        "No successful empirical evaluations "
        "are available in the standardized dataset."

    )


# --------------------------------------------------------------------------------------------------
# 10. EXPERIMENTAL CONTEXT
# --------------------------------------------------------------------------------------------------

STANDARD_CONTEXT_KEYS = [

    "dataset_id",

    "feature",

    "mechanism",

    "rate",

    "seed"

]


# --------------------------------------------------------------------------------------------------
# 11. CANDIDATE COVERAGE
# --------------------------------------------------------------------------------------------------

CONTEXT_COVERAGE_DF = (

    SUCCESS_STANDARD_DF

    .groupby(

        STANDARD_CONTEXT_KEYS,

        dropna=False

    )

    ["strategy"]

    .nunique()

    .reset_index(

        name="candidate_count"

    )

)


UNIQUE_CONTEXT_COUNT = int(

    len(
        CONTEXT_COVERAGE_DF
    )

)


MULTI_CANDIDATE_CONTEXT_COUNT = int(

    (

        CONTEXT_COVERAGE_DF[
            "candidate_count"
        ]

        >= 2

    )

    .sum()

)


MAX_CANDIDATES_PER_CONTEXT = int(

    CONTEXT_COVERAGE_DF[
        "candidate_count"
    ]
    .max()

)


# --------------------------------------------------------------------------------------------------
# 12. STRATEGY REGISTRY
# --------------------------------------------------------------------------------------------------

STRATEGY_REGISTRY_0724 = sorted(

    SUCCESS_STANDARD_DF[
        "strategy"
    ]
    .dropna()
    .unique()
    .tolist()

)


LLM_RECOMMENDATION_REGISTRY_0724 = sorted(

    SUCCESS_STANDARD_DF[
        "llm_recommended"
    ]
    .dropna()
    .unique()
    .tolist()

)


# --------------------------------------------------------------------------------------------------
# 13. LLM COVERAGE
# --------------------------------------------------------------------------------------------------

LLM_COVERAGE_ROWS = int(

    SUCCESS_STANDARD_DF[
        "llm_recommended"
    ]
    .notna()
    .sum()

)


TOTAL_SUCCESS_ROWS = int(

    len(
        SUCCESS_STANDARD_DF
    )

)


LLM_COVERAGE_RATE = (

    LLM_COVERAGE_ROWS
    /
    TOTAL_SUCCESS_ROWS

    if TOTAL_SUCCESS_ROWS > 0

    else 0.0

)


# --------------------------------------------------------------------------------------------------
# 14. REPORT
# --------------------------------------------------------------------------------------------------

print("\nSTANDARDIZED EMPIRICAL EVIDENCE")
print("-" * 100)

print(
    f"Successful evaluations       : "
    f"{TOTAL_SUCCESS_ROWS:,}"
)


print(
    f"Unique experimental contexts : "
    f"{UNIQUE_CONTEXT_COUNT:,}"
)


print(
    f"Contexts with >=2 candidates : "
    f"{MULTI_CANDIDATE_CONTEXT_COUNT:,}"
)


print(
    f"Maximum candidates/context   : "
    f"{MAX_CANDIDATES_PER_CONTEXT:,}"
)


print(
    f"Unique empirical strategies  : "
    f"{len(STRATEGY_REGISTRY_0724):,}"
)


print(
    f"LLM recommendation coverage  : "
    f"{LLM_COVERAGE_RATE:.2%}"
)


print(
    "\nEmpirical strategies:"
)


print(
    ", ".join(
        STRATEGY_REGISTRY_0724
    )
)


print(
    "\nLLM recommendation strategies:"
)


print(
    ", ".join(
        LLM_RECOMMENDATION_REGISTRY_0724
    )
)


# --------------------------------------------------------------------------------------------------
# 15. EXPECTED AIR-LLM 06.21 COVERAGE CHECK
# --------------------------------------------------------------------------------------------------

EXPECTED_SUCCESSFUL_EVALUATIONS = 5775


if TOTAL_SUCCESS_ROWS != EXPECTED_SUCCESSFUL_EVALUATIONS:

    print(
        "\nWARNING:"
    )

    print(
        f"Expected approximately "
        f"{EXPECTED_SUCCESSFUL_EVALUATIONS:,} "
        f"successful evaluations, but found "
        f"{TOTAL_SUCCESS_ROWS:,}."
    )

else:

    print(
        "\n06.21 successful-evaluation count : PASS"
    )


# --------------------------------------------------------------------------------------------------
# 16. CREATE ADAPTER METADATA
# --------------------------------------------------------------------------------------------------

EVALUATION_SCHEMA_ADAPTER_METADATA = {

    "source_notebook":
        "06.21",

    "source_rows":
        int(len(EVALUATION_DF)),

    "successful_rows":
        TOTAL_SUCCESS_ROWS,

    "unique_contexts":
        UNIQUE_CONTEXT_COUNT,

    "multi_candidate_contexts":
        MULTI_CANDIDATE_CONTEXT_COUNT,

    "max_candidates_per_context":
        MAX_CANDIDATES_PER_CONTEXT,

    "strategy_count":
        len(STRATEGY_REGISTRY_0724),

    "llm_recommendation_coverage":
        float(LLM_COVERAGE_RATE),

    "strategy_column":
        "strategy_id",

    "standardized_strategy_column":
        "strategy",

    "recommendation_column":
        "llm_recommended",

    "context_keys":
        STANDARD_CONTEXT_KEYS

}


# --------------------------------------------------------------------------------------------------
# 17. FINAL STATUS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("NOTEBOOK 07.4 EVALUATION SCHEMA ADAPTER : COMPLETE")
print("=" * 100)

print(
    "Source schema                : Notebook 06.21"
)

print(
    "AIR-LLM standard schema      : READY"
)

print(
    "Empirical evidence           : READY"
)

print(
    "LLM recommendation linkage   : READY"
)

print(
    "Downstream selection input   : READY"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 07.4
EVALUATION SCHEMA ADAPTER

NOTEBOOK 06.21 SCHEMA
----------------------------------------------------------------------------------------------------
06.21 empirical schema : PASS
Rows                  : 5,775
Columns               : 22

AIR-LLM COLUMN MAPPING
----------------------------------------------------------------------------------------------------
dataset_id           -> dataset_id
feature              -> feature
strategy             -> strategy_id
mechanism            -> missingness_mechanism
rate                 -> missingness_rate
repetition           -> repetition
seed                 -> seed
llm_recommended      -> llm_recommended
status               -> status
mae                  -> mae
rmse                 -> rmse
r2                   -> r2
accuracy             -> accuracy
macro_f1             -> macro_f1
weighted_f1          -> weighted_f1
efficiency           -> runtime_seconds

STANDARDIZED SCHEMA
------------------------------------------

In [13]:
# ============================================================
# NOTEBOOK 07.5 — STANDARDIZE EMPIRICAL EVIDENCE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.5")
print("STANDARDIZING EMPIRICAL EVIDENCE")
print("=" * 100)


# ============================================================
# 1. RECOVER NOTEBOOK 06.26 UTILITY ARTIFACT
# ============================================================

UTILITY_ARTIFACT_PATH = (
    PROJECT_ROOT
    / "results"
    / "notebook_06_final"
    / "air_llm_06_26_candidate_utility_scores.csv"
)

ADAPTIVE_SELECTION_ARTIFACT_PATH = (
    PROJECT_ROOT
    / "results"
    / "notebook_06_final"
    / "air_llm_06_26_adaptive_utility_selection.csv"
)


print("\nNOTEBOOK 06.26 UTILITY ARTIFACT")
print("-" * 100)


if UTILITY_ARTIFACT_PATH.exists():

    print(
        f"FOUND : {UTILITY_ARTIFACT_PATH}"
    )

    UTILITY_DF = pd.read_csv(
        UTILITY_ARTIFACT_PATH
    )

else:

    raise FileNotFoundError(
        "Notebook 06.26 candidate utility artifact was not found:\n"
        f"{UTILITY_ARTIFACT_PATH}"
    )


# ============================================================
# 2. NORMALIZE UTILITY COLUMNS
# ============================================================

UTILITY_DF = normalize_columns(
    UTILITY_DF
)


print(
    f"Utility rows available : "
    f"{len(UTILITY_DF):,}"
)


# ============================================================
# 3. REQUIRED EXPERIMENTAL KEYS
# ============================================================

KEY_CANDIDATES = {

    "evaluation_key": [
        "evaluation_key"
    ],

    "dataset_id": [
        "dataset_id"
    ],

    "feature": [
        "feature"
    ],

    "strategy": [
        "strategy_id",
        "strategy"
    ],

    "mechanism": [
        "missingness_mechanism",
        "mechanism"
    ],

    "rate": [
        "missingness_rate",
        "rate"
    ],

    "seed": [
        "seed"
    ],

    "repetition": [
        "repetition",
        "repeat",
        "rep"
    ]
}


def resolve_column(
    df,
    candidates
):

    for candidate in candidates:

        if candidate in df.columns:
            return candidate

    return None


UTILITY_COLUMN_MAP = {
    key: resolve_column(
        UTILITY_DF,
        candidates
    )
    for key, candidates in KEY_CANDIDATES.items()
}


print("\nUTILITY KEY MAPPING")
print("-" * 100)

for key, source in UTILITY_COLUMN_MAP.items():

    print(
        f"{key:20s} -> "
        f"{source if source else 'NOT FOUND'}"
    )


# ============================================================
# 4. VALIDATE CORE KEYS
# ============================================================

required_utility_keys = [
    "dataset_id",
    "feature",
    "strategy",
    "mechanism",
    "rate",
    "seed"
]


missing_utility_keys = [
    key
    for key in required_utility_keys
    if UTILITY_COLUMN_MAP[key] is None
]


if missing_utility_keys:

    raise RuntimeError(
        "Notebook 06.26 utility artifact is missing required "
        "experimental keys:\n"
        + "\n".join(
            missing_utility_keys
        )
    )


# ============================================================
# 5. STANDARDIZE RAW EMPIRICAL EVIDENCE
# ============================================================

STANDARD_EVIDENCE = pd.DataFrame(
    index=EVALUATION_DF.index
)


# ------------------------------------------------------------
# Experimental identity
# ------------------------------------------------------------

for standard_name, source_name in {

    "evaluation_key": "evaluation_key",
    "dataset_id": "dataset_id",
    "feature": "feature",
    "target": "target",
    "feature_type": "feature_type",
    "strategy": "strategy_id",
    "mechanism": "missingness_mechanism",
    "rate": "missingness_rate",
    "repetition": "repetition",
    "seed": "seed"

}.items():

    if source_name in EVALUATION_DF.columns:

        STANDARD_EVIDENCE[
            standard_name
        ] = EVALUATION_DF[
            source_name
        ]

    else:

        STANDARD_EVIDENCE[
            standard_name
        ] = np.nan


# ============================================================
# 6. STANDARDIZE EMPIRICAL METRICS
# ============================================================

EMPIRICAL_METRICS = [

    "mae",
    "rmse",
    "r2",
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "n",
    "runtime_seconds"
]


for metric in EMPIRICAL_METRICS:

    if metric in EVALUATION_DF.columns:

        STANDARD_EVIDENCE[
            metric
        ] = pd.to_numeric(
            EVALUATION_DF[
                metric
            ],
            errors="coerce"
        )

    else:

        STANDARD_EVIDENCE[
            metric
        ] = np.nan


# ============================================================
# 7. VALIDATE EMPIRICAL STATUS
# ============================================================

if "status" in EVALUATION_DF.columns:

    STANDARD_EVIDENCE[
        "status"
    ] = EVALUATION_DF[
        "status"
    ].astype(str)

else:

    STANDARD_EVIDENCE[
        "status"
    ] = "UNKNOWN"


# ============================================================
# 8. ATTACH LLM RECOMMENDATION
# ============================================================

if "llm_recommended" in EVALUATION_DF.columns:

    STANDARD_EVIDENCE[
        "llm_recommended"
    ] = EVALUATION_DF[
        "llm_recommended"
    ].astype(str)

elif (
    "LLM_EVALUATION_DF" in globals()
    and
    "llm_recommended" in LLM_EVALUATION_DF.columns
):

    llm_lookup = (
        LLM_EVALUATION_DF[
            [
                "evaluation_key",
                "llm_recommended"
            ]
        ]
        .drop_duplicates(
            "evaluation_key"
        )
    )

    STANDARD_EVIDENCE = (
        STANDARD_EVIDENCE
        .merge(
            llm_lookup,
            on="evaluation_key",
            how="left",
            suffixes=("", "_llm")
        )
    )

else:

    STANDARD_EVIDENCE[
        "llm_recommended"
    ] = np.nan


# ============================================================
# 9. ATTACH NOTEBOOK 06.26 UTILITY DIMENSIONS
# ============================================================

UTILITY_COLUMNS = [

    "reconstruction_utility",
    "distribution_utility",
    "dependency_utility",
    "downstream_utility",
    "efficiency_utility",
    "adaptive_utility"
]


available_utility_columns = [
    column
    for column in UTILITY_COLUMNS
    if column in UTILITY_DF.columns
]


print("\nUTILITY DIMENSIONS FOUND")
print("-" * 100)

for column in UTILITY_COLUMNS:

    print(
        f"{column:25s} : "
        f"{'FOUND' if column in available_utility_columns else 'NOT AVAILABLE'}"
    )


# ------------------------------------------------------------
# Build utility join keys
# ------------------------------------------------------------

UTILITY_JOIN_KEYS = [
    "dataset_id",
    "feature",
    "strategy",
    "mechanism",
    "rate",
    "seed"
]


UTILITY_JOIN_DF = pd.DataFrame()


for key in UTILITY_JOIN_KEYS:

    source = UTILITY_COLUMN_MAP[
        key
    ]

    UTILITY_JOIN_DF[
        key
    ] = UTILITY_DF[
        source
    ]


for column in available_utility_columns:

    UTILITY_JOIN_DF[
        column
    ] = pd.to_numeric(
        UTILITY_DF[
            column
        ],
        errors="coerce"
    )


# ------------------------------------------------------------
# Normalize join-key types
# ------------------------------------------------------------

STANDARD_EVIDENCE[
    "dataset_id"
] = (
    STANDARD_EVIDENCE[
        "dataset_id"
    ]
    .astype(str)
    .str.strip()
)

STANDARD_EVIDENCE[
    "feature"
] = (
    STANDARD_EVIDENCE[
        "feature"
    ]
    .astype(str)
    .str.strip()
)

STANDARD_EVIDENCE[
    "strategy"
] = (
    STANDARD_EVIDENCE[
        "strategy"
    ]
    .astype(str)
    .str.strip()
)

STANDARD_EVIDENCE[
    "mechanism"
] = (
    STANDARD_EVIDENCE[
        "mechanism"
    ]
    .astype(str)
    .str.strip()
)

STANDARD_EVIDENCE[
    "rate"
] = pd.to_numeric(
    STANDARD_EVIDENCE[
        "rate"
    ],
    errors="coerce"
)

STANDARD_EVIDENCE[
    "seed"
] = pd.to_numeric(
    STANDARD_EVIDENCE[
        "seed"
    ],
    errors="coerce"
)


UTILITY_JOIN_DF[
    "dataset_id"
] = (
    UTILITY_JOIN_DF[
        "dataset_id"
    ]
    .astype(str)
    .str.strip()
)

UTILITY_JOIN_DF[
    "feature"
] = (
    UTILITY_JOIN_DF[
        "feature"
    ]
    .astype(str)
    .str.strip()
)

UTILITY_JOIN_DF[
    "strategy"
] = (
    UTILITY_JOIN_DF[
        "strategy"
    ]
    .astype(str)
    .str.strip()
)

UTILITY_JOIN_DF[
    "mechanism"
] = (
    UTILITY_JOIN_DF[
        "mechanism"
    ]
    .astype(str)
    .str.strip()
)

UTILITY_JOIN_DF[
    "rate"
] = pd.to_numeric(
    UTILITY_JOIN_DF[
        "rate"
    ],
    errors="coerce"
)

UTILITY_JOIN_DF[
    "seed"
] = pd.to_numeric(
    UTILITY_JOIN_DF[
        "seed"
    ],
    errors="coerce"
)


# ============================================================
# 10. REMOVE DUPLICATE UTILITY KEYS
# ============================================================

UTILITY_JOIN_DF = (
    UTILITY_JOIN_DF
    .drop_duplicates(
        subset=UTILITY_JOIN_KEYS,
        keep="first"
    )
)


# ============================================================
# 11. ATTACH UTILITY RESULTS
# ============================================================

print("\nATTACHING 06.26 UTILITY RESULTS")
print("-" * 100)

before_rows = len(
    STANDARD_EVIDENCE
)


STANDARD_EVIDENCE = (
    STANDARD_EVIDENCE
    .merge(
        UTILITY_JOIN_DF,
        on=UTILITY_JOIN_KEYS,
        how="left",
        validate="many_to_one"
    )
)


after_rows = len(
    STANDARD_EVIDENCE
)


if before_rows != after_rows:

    raise RuntimeError(
        "Utility attachment changed the number of "
        "empirical evidence rows."
    )


# ============================================================
# 12. UTILITY ATTACHMENT VALIDATION
# ============================================================

if available_utility_columns:

    utility_attachment_coverage = (
        STANDARD_EVIDENCE[
            available_utility_columns
        ]
        .notna()
        .any(
            axis=1
        )
        .mean()
    )

else:

    utility_attachment_coverage = 0.0


print(
    f"Empirical evidence rows : "
    f"{before_rows:,}"
)

print(
    f"Rows after utility join : "
    f"{after_rows:,}"
)

print(
    f"Utility attachment rate : "
    f"{utility_attachment_coverage:.2%}"
)


if (
    available_utility_columns
    and
    utility_attachment_coverage < 1.0
):

    raise RuntimeError(
        "06.26 utility results were not successfully "
        "attached to every empirical evaluation."
    )


# ============================================================
# 13. RAW EMPIRICAL METRIC COVERAGE
# ============================================================

print("\nRAW EMPIRICAL METRIC COVERAGE")
print("-" * 100)


coverage_rows = []


for metric in EMPIRICAL_METRICS:

    if metric in STANDARD_EVIDENCE.columns:

        valid_rows = int(
            STANDARD_EVIDENCE[
                metric
            ]
            .notna()
            .sum()
        )

    else:

        valid_rows = 0


    coverage_rows.append({

        "metric": metric,

        "valid_rows": valid_rows,

        "coverage_rate": (
            valid_rows
            /
            len(STANDARD_EVIDENCE)
        )

    })


RAW_METRIC_COVERAGE_DF = pd.DataFrame(
    coverage_rows
)


display(
    RAW_METRIC_COVERAGE_DF
)


# ============================================================
# 14. UTILITY DIMENSION COVERAGE
# ============================================================

print("\n06.26 UTILITY COVERAGE")
print("-" * 100)


utility_coverage_rows = []


for dimension in UTILITY_COLUMNS:

    if dimension in STANDARD_EVIDENCE.columns:

        valid_rows = int(
            STANDARD_EVIDENCE[
                dimension
            ]
            .notna()
            .sum()
        )

    else:

        valid_rows = 0


    utility_coverage_rows.append({

        "utility_dimension": dimension,

        "valid_rows": valid_rows,

        "coverage_rate": (
            valid_rows
            /
            len(STANDARD_EVIDENCE)
        )

    })


UTILITY_COVERAGE_DF = pd.DataFrame(
    utility_coverage_rows
)


display(
    UTILITY_COVERAGE_DF
)


# ============================================================
# 15. DO NOT TREAT UNAVAILABLE DIMENSIONS AS ZERO
# ============================================================

# Explicit availability flags are important for research
# reporting and downstream confidence estimation.

for dimension in [
    "reconstruction_utility",
    "distribution_utility",
    "dependency_utility",
    "downstream_utility",
    "efficiency_utility"
]:

    STANDARD_EVIDENCE[
        f"{dimension}_available"
    ] = (
        STANDARD_EVIDENCE[
            dimension
        ]
        .notna()
        if dimension in STANDARD_EVIDENCE.columns
        else False
    )


# ============================================================
# 16. UTILITY COVERAGE SCORE
# ============================================================

UTILITY_DIMENSION_NAMES = [
    "reconstruction_utility",
    "distribution_utility",
    "dependency_utility",
    "downstream_utility",
    "efficiency_utility"
]


available_dimension_flags = [
    f"{dimension}_available"
    for dimension in UTILITY_DIMENSION_NAMES
]


STANDARD_EVIDENCE[
    "utility_dimensions_available"
] = (
    STANDARD_EVIDENCE[
        available_dimension_flags
    ]
    .sum(
        axis=1
    )
)


STANDARD_EVIDENCE[
    "utility_coverage"
] = (
    STANDARD_EVIDENCE[
        "utility_dimensions_available"
    ]
    /
    len(
        UTILITY_DIMENSION_NAMES
    )
)


# ============================================================
# 17. EVIDENCE QUALITY CLASSIFICATION
# ============================================================

def classify_evidence_coverage(
    coverage
):

    if coverage >= 0.80:
        return "HIGH"

    if coverage >= 0.60:
        return "MODERATE"

    if coverage >= 0.40:
        return "LIMITED"

    return "LOW"


STANDARD_EVIDENCE[
    "utility_evidence_quality"
] = (
    STANDARD_EVIDENCE[
        "utility_coverage"
    ]
    .apply(
        classify_evidence_coverage
    )
)


# ============================================================
# 18. FINAL STANDARDIZATION SUMMARY
# ============================================================

print("\nSTANDARD EVIDENCE SUMMARY")
print("-" * 100)

print(
    f"Total evidence rows       : "
    f"{len(STANDARD_EVIDENCE):,}"
)

if "status" in STANDARD_EVIDENCE.columns:

    successful_rows = int(
        (
            STANDARD_EVIDENCE[
                "status"
            ]
            .str.upper()
            == "SUCCESS"
        )
        .sum()
    )

else:

    successful_rows = len(
        STANDARD_EVIDENCE
    )


print(
    f"Successful evidence rows  : "
    f"{successful_rows:,}"
)

print(
    f"Unique datasets            : "
    f"{STANDARD_EVIDENCE['dataset_id'].nunique():,}"
)

print(
    f"Unique features            : "
    f"{STANDARD_EVIDENCE['feature'].nunique():,}"
)

print(
    f"Unique strategies          : "
    f"{STANDARD_EVIDENCE['strategy'].nunique():,}"
)

print(
    f"LLM recommendations        : "
    f"{STANDARD_EVIDENCE['llm_recommended'].notna().sum():,}"
)


print("\nUTILITY DIMENSION AVAILABILITY")
print("-" * 100)

for dimension in UTILITY_DIMENSION_NAMES:

    available = int(
        STANDARD_EVIDENCE[
            f"{dimension}_available"
        ]
        .sum()
    )

    rate = (
        available
        /
        len(STANDARD_EVIDENCE)
    )

    print(
        f"{dimension:25s} : "
        f"{available:,} / "
        f"{len(STANDARD_EVIDENCE):,} "
        f"({rate:.2%})"
    )


print("\nSAMPLE STANDARDIZED EVIDENCE")
print("-" * 100)

display(
    STANDARD_EVIDENCE.head(
        10
    )
)


print(
    f"\nFinal standardized rows : "
    f"{len(STANDARD_EVIDENCE):,}"
)

print(
    f"Final standardized columns : "
    f"{len(STANDARD_EVIDENCE.columns):,}"
)


# ============================================================
# 19. SAVE STANDARDIZED EVIDENCE
# ============================================================

STANDARD_EVIDENCE_PATH = (
    OUTPUT_DIR
    / "standardized_empirical_evidence.csv"
)

RAW_METRIC_COVERAGE_PATH = (
    DIAGNOSTIC_DIR
    / "raw_metric_coverage.csv"
)

UTILITY_COVERAGE_PATH = (
    DIAGNOSTIC_DIR
    / "utility_dimension_coverage.csv"
)


STANDARD_EVIDENCE.to_csv(
    STANDARD_EVIDENCE_PATH,
    index=False
)

RAW_METRIC_COVERAGE_DF.to_csv(
    RAW_METRIC_COVERAGE_PATH,
    index=False
)

UTILITY_COVERAGE_DF.to_csv(
    UTILITY_COVERAGE_PATH,
    index=False
)


print("\nPERSISTENCE")
print("-" * 100)

print(
    f"Standardized evidence : "
    f"{STANDARD_EVIDENCE_PATH}"
)

print(
    f"Raw metric coverage   : "
    f"{RAW_METRIC_COVERAGE_PATH}"
)

print(
    f"Utility coverage      : "
    f"{UTILITY_COVERAGE_PATH}"
)


# ============================================================
# 20. DOWNSTREAM DATAFRAME
# ============================================================

AIR_LLM_0705_STANDARD_EVIDENCE_DF = (
    STANDARD_EVIDENCE.copy()
)

AIR_LLM_0705_RAW_METRIC_COVERAGE_DF = (
    RAW_METRIC_COVERAGE_DF.copy()
)

AIR_LLM_0705_UTILITY_COVERAGE_DF = (
    UTILITY_COVERAGE_DF.copy()
)


print("\n" + "=" * 100)
print("NOTEBOOK 07.5 STANDARDIZATION : COMPLETE")
print("=" * 100)

print(
    f"Standardized evidence rows : "
    f"{len(AIR_LLM_0705_STANDARD_EVIDENCE_DF):,}"
)

print(
    "Distribution/dependency metrics are explicitly marked "
    "as unavailable where absent; they are NOT treated as zero."
)

print("=" * 100)

AIR-LLM — NOTEBOOK 07.5
STANDARDIZING EMPIRICAL EVIDENCE

NOTEBOOK 06.26 UTILITY ARTIFACT
----------------------------------------------------------------------------------------------------
FOUND : /content/drive/MyDrive/AIR_LLM_Research/results/notebook_06_final/air_llm_06_26_candidate_utility_scores.csv
Utility rows available : 5,775

UTILITY KEY MAPPING
----------------------------------------------------------------------------------------------------
evaluation_key       -> evaluation_key
dataset_id           -> dataset_id
feature              -> feature
strategy             -> strategy_id
mechanism            -> missingness_mechanism
rate                 -> missingness_rate
seed                 -> seed
repetition           -> repetition

UTILITY DIMENSIONS FOUND
----------------------------------------------------------------------------------------------------
reconstruction_utility    : FOUND
distribution_utility      : FOUND
dependency_utility        : FOUND
downstream_utilit

,metric,valid_rows,coverage_rate
0,mae,1800,0.311688
1,rmse,1800,0.311688
2,r2,1800,0.311688
3,accuracy,3975,0.688312
4,macro_f1,3975,0.688312
5,weighted_f1,3975,0.688312
6,n,5775,1.000000
7,runtime_seconds,5775,1.000000



06.26 UTILITY COVERAGE
----------------------------------------------------------------------------------------------------


,utility_dimension,valid_rows,coverage_rate
0,reconstruction_utility,1800,0.311688
1,distribution_utility,0,0.000000
2,dependency_utility,0,0.000000
3,downstream_utility,5775,1.000000
4,efficiency_utility,5775,1.000000
5,adaptive_utility,5775,1.000000



STANDARD EVIDENCE SUMMARY
----------------------------------------------------------------------------------------------------
Total evidence rows       : 5,775
Successful evidence rows  : 5,775
Unique datasets            : 3
Unique features            : 73
Unique strategies          : 2
LLM recommendations        : 5,775

UTILITY DIMENSION AVAILABILITY
----------------------------------------------------------------------------------------------------
reconstruction_utility    : 1,800 / 5,775 (31.17%)
distribution_utility      : 0 / 5,775 (0.00%)
dependency_utility        : 0 / 5,775 (0.00%)
downstream_utility        : 5,775 / 5,775 (100.00%)
efficiency_utility        : 5,775 / 5,775 (100.00%)

SAMPLE STANDARDIZED EVIDENCE
----------------------------------------------------------------------------------------------------


,evaluation_key,dataset_id,feature,target,feature_type,strategy,mechanism,rate,repetition,seed,...,efficiency_utility,adaptive_utility,reconstruction_utility_available,distribution_utility_available,dependency_utility_available,downstream_utility_available,efficiency_utility_available,utility_dimensions_available,utility_coverage,utility_evidence_quality
0,8b8d979f3253c12107b63163,adult_income,age,income,numerical,mean,MAR,0.1,1,3393134027,...,1.0,1.0,True,False,False,True,True,3,0.6,MODERATE
1,54811d482230bd847f01f2b7,adult_income,age,income,numerical,mean,MAR,0.1,2,2074715863,...,1.0,1.0,True,False,False,True,True,3,0.6,MODERATE
2,e30a0e15815828c68cb864b6,adult_income,age,income,numerical,mean,MAR,0.1,3,345086045,...,1.0,1.0,True,False,False,True,True,3,0.6,MODERATE
3,89feffbaf84ff25e810b53d1,adult_income,age,income,numerical,mean,MAR,0.1,4,1182932783,...,1.0,1.0,True,False,False,True,True,3,0.6,MODERATE
4,7cb75e5870163cf2b564d213,adult_income,age,income,numerical,mean,MAR,0.1,5,3313111679,...,1.0,1.0,True,False,False,True,True,3,0.6,MODERATE
5,9009c0b4f43619f556271df8,adult_income,age,income,numerical,mean,MAR,0.2,1,2657548161,...,1.0,1.0,True,False,False,True,True,3,0.6,MODERATE
6,1574667a7f3ee1852cbb371f,adult_income,age,income,numerical,mean,MAR,0.2,2,1963370003,...,1.0,1.0,True,False,False,True,True,3,0.6,MODERATE
7,b9e47d23cca6d7000713c5ba,adult_income,age,income,numerical,mean,MAR,0.2,3,433268084,...,1.0,1.0,True,False,False,True,True,3,0.6,MODERATE
8,cbc838fa5b563135c3336ed7,adult_income,age,income,numerical,mean,MAR,0.2,4,510431391,...,1.0,1.0,True,False,False,True,True,3,0.6,MODERATE
9,04e3b63a4a716641bdf6d929,adult_income,age,income,numerical,mean,MAR,0.2,5,2220770356,...,1.0,1.0,True,False,False,True,True,3,0.6,MODERATE



Final standardized rows : 5,775
Final standardized columns : 34

PERSISTENCE
----------------------------------------------------------------------------------------------------
Standardized evidence : /content/drive/MyDrive/AIR_LLM_Research/data/notebook_07/standardized_empirical_evidence.csv
Raw metric coverage   : /content/drive/MyDrive/AIR_LLM_Research/data/notebook_07/diagnostics/raw_metric_coverage.csv
Utility coverage      : /content/drive/MyDrive/AIR_LLM_Research/data/notebook_07/diagnostics/utility_dimension_coverage.csv

NOTEBOOK 07.5 STANDARDIZATION : COMPLETE
Standardized evidence rows : 5,775
Distribution/dependency metrics are explicitly marked as unavailable where absent; they are NOT treated as zero.


In [15]:
# ============================================================
# NOTEBOOK 07.6 — METRIC DIRECTION HANDLING
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.6")
print("METRIC DIRECTION NORMALIZATION")
print("=" * 100)


# ------------------------------------------------------------
# Research-defined metric directions
# ------------------------------------------------------------
#
# AIR-LLM uses:
#
#   Reconstruction:
#       MAE       -> LOWER is better
#       RMSE      -> LOWER is better
#       R2        -> HIGHER is better
#       Accuracy  -> HIGHER is better
#       Macro-F1  -> HIGHER is better
#       Weighted-F1-> HIGHER is better
#
#   Distributional fidelity:
#       Wasserstein distance -> LOWER is better
#       JS divergence        -> LOWER is better
#       KL divergence        -> LOWER is better
#
#   Dependency preservation:
#       Pearson/Spearman/M.I. preservation -> HIGHER is better
#       Correlation error                   -> LOWER is better
#
#   Efficiency:
#       Runtime -> LOWER is better
# ------------------------------------------------------------


METRIC_DIRECTIONS = {

    # Reconstruction
    "mae": False,
    "rmse": False,
    "r2": True,
    "accuracy": True,
    "macro_f1": True,
    "weighted_f1": True,

    # Distributional fidelity
    "wasserstein": False,
    "wasserstein_distance": False,
    "js_divergence": False,
    "kl_divergence": False,
    "kl": False,

    # Dependency preservation
    "pearson": True,
    "spearman": True,
    "mutual_information": True,
    "mutual_info": True,
    "dependency_score": True,
    "correlation_error": False,
    "dependency_error": False,

    # Computational efficiency
    "runtime_seconds": False,
    "runtime": False,
    "time_seconds": False
}


# ------------------------------------------------------------
# Utility-dimension availability
# ------------------------------------------------------------

UTILITY_DIMENSION_METRICS = {

    "reconstruction": [
        "mae",
        "rmse",
        "r2",
        "accuracy",
        "macro_f1",
        "weighted_f1"
    ],

    "distribution": [
        "wasserstein",
        "wasserstein_distance",
        "js_divergence",
        "kl_divergence",
        "kl"
    ],

    "dependency": [
        "pearson",
        "spearman",
        "mutual_information",
        "mutual_info",
        "dependency_score",
        "correlation_error",
        "dependency_error"
    ],

    "predictive": [
        "r2",
        "accuracy",
        "macro_f1",
        "weighted_f1"
    ],

    "efficiency": [
        "runtime_seconds",
        "runtime",
        "time_seconds"
    ]
}


# ------------------------------------------------------------
# Detect metrics actually available in standardized evidence
# ------------------------------------------------------------

AVAILABLE_METRICS = [
    column
    for column in STANDARD_EVIDENCE.columns
    if column in METRIC_DIRECTIONS
]


print("\nAVAILABLE METRICS")
print("-" * 100)

if AVAILABLE_METRICS:

    for metric in AVAILABLE_METRICS:

        direction = (
            "HIGHER IS BETTER"
            if METRIC_DIRECTIONS[metric]
            else "LOWER IS BETTER"
        )

        valid_rows = int(
            STANDARD_EVIDENCE[metric]
            .notna()
            .sum()
        )

        print(
            f"{metric:25s} | "
            f"{direction:20s} | "
            f"valid rows: {valid_rows:,}"
        )

else:

    print("No recognized empirical metrics detected.")


# ------------------------------------------------------------
# Determine available metrics by utility dimension
# ------------------------------------------------------------

UTILITY_DIMENSION_STATUS = []


for dimension, metrics in UTILITY_DIMENSION_METRICS.items():

    available = [
        metric
        for metric in metrics
        if metric in STANDARD_EVIDENCE.columns
        and STANDARD_EVIDENCE[metric].notna().any()
    ]

    UTILITY_DIMENSION_STATUS.append({

        "utility_dimension": dimension,

        "available_metrics": (
            ", ".join(available)
            if available
            else ""
        ),

        "n_available_metrics": len(
            available
        ),

        "available": bool(
            available
        )
    })


UTILITY_DIMENSION_STATUS_DF = pd.DataFrame(
    UTILITY_DIMENSION_STATUS
)


print("\nUTILITY DIMENSION AVAILABILITY")
print("-" * 100)

display(
    UTILITY_DIMENSION_STATUS_DF
)


# ------------------------------------------------------------
# Build research-safe direction map
# ------------------------------------------------------------

RESEARCH_METRIC_DIRECTION_DF = pd.DataFrame({

    "metric": list(
        METRIC_DIRECTIONS.keys()
    ),

    "higher_is_better": list(
        METRIC_DIRECTIONS.values()
    )
})


RESEARCH_METRIC_DIRECTION_DF[
    "direction"
] = np.where(
    RESEARCH_METRIC_DIRECTION_DF[
        "higher_is_better"
    ],
    "HIGHER",
    "LOWER"
)


# ------------------------------------------------------------
# Save direction configuration
# ------------------------------------------------------------

METRIC_DIRECTION_PATH = (
    METADATA_DIR /
    "metric_direction_configuration.csv"
)

UTILITY_DIMENSION_STATUS_PATH = (
    METADATA_DIR /
    "utility_dimension_availability.csv"
)


RESEARCH_METRIC_DIRECTION_DF.to_csv(
    METRIC_DIRECTION_PATH,
    index=False
)

UTILITY_DIMENSION_STATUS_DF.to_csv(
    UTILITY_DIMENSION_STATUS_PATH,
    index=False
)


# ------------------------------------------------------------
# Final diagnostic
# ------------------------------------------------------------

print("\nRESEARCH METRIC DIRECTION SUMMARY")
print("-" * 100)

for _, row in (
    RESEARCH_METRIC_DIRECTION_DF[
        RESEARCH_METRIC_DIRECTION_DF[
            "metric"
        ].isin(
            AVAILABLE_METRICS
        )
    ].iterrows()
):

    print(
        f"{row['metric']:25s} : "
        f"{row['direction']} IS BETTER"
    )


print("\nCONFIGURATION SAVED")
print("-" * 100)

print(
    f"Metric directions :\n"
    f"{METRIC_DIRECTION_PATH}"
)

print(
    f"Utility dimension availability :\n"
    f"{UTILITY_DIMENSION_STATUS_PATH}"
)


print("=" * 100)
print("NOTEBOOK 07.6 METRIC DIRECTION HANDLING : COMPLETE")
print("=" * 100)

AIR-LLM — NOTEBOOK 07.6
METRIC DIRECTION NORMALIZATION

AVAILABLE METRICS
----------------------------------------------------------------------------------------------------
mae                       | LOWER IS BETTER      | valid rows: 1,800
rmse                      | LOWER IS BETTER      | valid rows: 1,800
r2                        | HIGHER IS BETTER     | valid rows: 1,800
accuracy                  | HIGHER IS BETTER     | valid rows: 3,975
macro_f1                  | HIGHER IS BETTER     | valid rows: 3,975
weighted_f1               | HIGHER IS BETTER     | valid rows: 3,975
runtime_seconds           | LOWER IS BETTER      | valid rows: 5,775

UTILITY DIMENSION AVAILABILITY
----------------------------------------------------------------------------------------------------


,utility_dimension,available_metrics,n_available_metrics,available
0,reconstruction,"mae, rmse, r2, accuracy, macro_f1, weighted_f1",6,True
1,distribution,,0,False
2,dependency,,0,False
3,predictive,"r2, accuracy, macro_f1, weighted_f1",4,True
4,efficiency,runtime_seconds,1,True



RESEARCH METRIC DIRECTION SUMMARY
----------------------------------------------------------------------------------------------------
mae                       : LOWER IS BETTER
rmse                      : LOWER IS BETTER
r2                        : HIGHER IS BETTER
accuracy                  : HIGHER IS BETTER
macro_f1                  : HIGHER IS BETTER
weighted_f1               : HIGHER IS BETTER
runtime_seconds           : LOWER IS BETTER

CONFIGURATION SAVED
----------------------------------------------------------------------------------------------------
Metric directions :
/content/drive/MyDrive/AIR_LLM_Research/data/notebook_07/metadata/metric_direction_configuration.csv
Utility dimension availability :
/content/drive/MyDrive/AIR_LLM_Research/data/notebook_07/metadata/utility_dimension_availability.csv
NOTEBOOK 07.6 METRIC DIRECTION HANDLING : COMPLETE


In [17]:
# ============================================================
# NOTEBOOK 07.7 — ROBUST METRIC NORMALIZATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.7")
print("ROBUST METRIC NORMALIZATION")
print("=" * 100)


# ------------------------------------------------------------
# Robust percentile-based normalization
# ------------------------------------------------------------

def robust_minmax(
    series,
    higher_is_better=True
):

    x = pd.to_numeric(
        series,
        errors="coerce"
    )

    valid = x.dropna()

    # No empirical observations
    if len(valid) == 0:

        return pd.Series(
            np.nan,
            index=series.index,
            dtype=float
        )

    # Robust 5th–95th percentile bounds
    q_low = valid.quantile(0.05)
    q_high = valid.quantile(0.95)

    if not np.isfinite(q_low):
        q_low = valid.min()

    if not np.isfinite(q_high):
        q_high = valid.max()

    # Constant metric within the normalization group
    if q_high <= q_low:

        result = pd.Series(
            0.5,
            index=series.index,
            dtype=float
        )

    else:

        clipped = x.clip(
            lower=q_low,
            upper=q_high
        )

        result = (
            clipped - q_low
        ) / (
            q_high - q_low
        )

    # Reverse error/distance/runtime metrics
    if not higher_is_better:

        result = 1.0 - result

    # Preserve missingness rather than inventing evidence
    result = result.where(
        x.notna(),
        np.nan
    )

    return result.clip(
        0.0,
        1.0
    )


# ------------------------------------------------------------
# Actual empirical metrics available in Notebook 06
# ------------------------------------------------------------

EMPIRICAL_METRICS = [

    "mae",
    "rmse",
    "r2",
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "runtime_seconds"
]


# ------------------------------------------------------------
# Validate metric availability
# ------------------------------------------------------------

AVAILABLE_EMPIRICAL_METRICS = [
    metric
    for metric in EMPIRICAL_METRICS
    if metric in STANDARD_EVIDENCE.columns
]


print("\nEMPIRICAL METRICS AVAILABLE")
print("-" * 100)

for metric in EMPIRICAL_METRICS:

    if metric in STANDARD_EVIDENCE.columns:

        valid_rows = int(
            STANDARD_EVIDENCE[
                metric
            ].notna().sum()
        )

        print(
            f"{metric:20s} : "
            f"AVAILABLE | "
            f"valid rows = {valid_rows:,}"
        )

    else:

        print(
            f"{metric:20s} : "
            f"NOT AVAILABLE"
        )


if not AVAILABLE_EMPIRICAL_METRICS:

    raise RuntimeError(
        "No empirical metrics are available "
        "for normalization."
    )


# ------------------------------------------------------------
# Normalize metrics within experimental context
# ------------------------------------------------------------
#
# Normalization is performed within:
#
#       dataset × feature
#
# This prevents large-scale differences between datasets
# and feature domains from dominating the utility score.
#
# Missing metric values remain NaN.
# They are NOT replaced by 0.5 because that would falsely
# imply neutral empirical evidence.
# ------------------------------------------------------------

NORMALIZATION_GROUPS = [
    "dataset_id",
    "feature"
]


for metric in AVAILABLE_EMPIRICAL_METRICS:

    direction = METRIC_DIRECTIONS.get(
        metric,
        None
    )

    if direction is None:

        print(
            f"WARNING: No direction defined for "
            f"{metric}; normalization skipped."
        )

        continue

    score_column = (
        f"{metric}_score"
    )

    STANDARD_EVIDENCE[
        score_column
    ] = (

        STANDARD_EVIDENCE
        .groupby(
            NORMALIZATION_GROUPS,
            dropna=False
        )[metric]
        .transform(

            lambda s, d=direction:
                robust_minmax(
                    s,
                    higher_is_better=d
                )
        )
    )


# ------------------------------------------------------------
# Report normalized metric coverage
# ------------------------------------------------------------

NORMALIZED_METRIC_ROWS = []


for metric in AVAILABLE_EMPIRICAL_METRICS:

    score_column = (
        f"{metric}_score"
    )

    if score_column not in STANDARD_EVIDENCE.columns:
        continue

    valid_rows = int(
        STANDARD_EVIDENCE[
            score_column
        ].notna().sum()
    )

    NORMALIZED_METRIC_ROWS.append({

        "metric": metric,

        "direction": (
            "HIGHER"
            if METRIC_DIRECTIONS[metric]
            else "LOWER"
        ),

        "valid_rows": valid_rows,

        "coverage_rate": (
            valid_rows /
            len(STANDARD_EVIDENCE)
            if len(STANDARD_EVIDENCE) > 0
            else 0.0
        )

    })


NORMALIZED_METRIC_COVERAGE_DF = pd.DataFrame(
    NORMALIZED_METRIC_ROWS
)


print("\nNORMALIZED METRIC COVERAGE")
print("-" * 100)

display(
    NORMALIZED_METRIC_COVERAGE_DF
)


# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

print("\nNORMALIZATION SANITY CHECK")
print("-" * 100)


for metric in AVAILABLE_EMPIRICAL_METRICS:

    score_column = (
        f"{metric}_score"
    )

    if score_column not in STANDARD_EVIDENCE.columns:
        continue

    valid_scores = STANDARD_EVIDENCE[
        score_column
    ].dropna()

    if len(valid_scores) == 0:

        print(
            f"{metric:20s} : "
            f"NO VALID NORMALIZED VALUES"
        )

        continue

    minimum = float(
        valid_scores.min()
    )

    maximum = float(
        valid_scores.max()
    )

    invalid = int(
        (
            (valid_scores < 0.0)
            |
            (valid_scores > 1.0)
        ).sum()
    )

    status = (
        "PASS"
        if invalid == 0
        else "FAIL"
    )

    print(
        f"{metric:20s} : "
        f"{status} | "
        f"range=[{minimum:.4f}, {maximum:.4f}] | "
        f"invalid={invalid}"
    )

    if invalid > 0:

        raise RuntimeError(
            f"Normalized {metric} contains "
            f"values outside [0,1]."
        )


# ------------------------------------------------------------
# Save normalization configuration
# ------------------------------------------------------------

NORMALIZATION_PATH = (
    METADATA_DIR /
    "normalized_metric_coverage.csv"
)

NORMALIZED_METRIC_COVERAGE_DF.to_csv(
    NORMALIZATION_PATH,
    index=False
)


print("\nNORMALIZATION ARTIFACT")
print("-" * 100)

print(
    f"Saved to:\n"
    f"{NORMALIZATION_PATH}"
)


# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print("\nSTANDARD EVIDENCE COLUMNS AFTER NORMALIZATION")
print("-" * 100)

normalized_columns = [
    column
    for column in STANDARD_EVIDENCE.columns
    if column.endswith("_score")
]

for column in normalized_columns:
    print(column)


print("=" * 100)
print("NOTEBOOK 07.7 ROBUST METRIC NORMALIZATION : COMPLETE")
print("=" * 100)

AIR-LLM — NOTEBOOK 07.7
ROBUST METRIC NORMALIZATION

EMPIRICAL METRICS AVAILABLE
----------------------------------------------------------------------------------------------------
mae                  : AVAILABLE | valid rows = 1,800
rmse                 : AVAILABLE | valid rows = 1,800
r2                   : AVAILABLE | valid rows = 1,800
accuracy             : AVAILABLE | valid rows = 3,975
macro_f1             : AVAILABLE | valid rows = 3,975
weighted_f1          : AVAILABLE | valid rows = 3,975
runtime_seconds      : AVAILABLE | valid rows = 5,775

NORMALIZED METRIC COVERAGE
----------------------------------------------------------------------------------------------------


,metric,direction,valid_rows,coverage_rate
0,mae,LOWER,1800,0.311688
1,rmse,LOWER,1800,0.311688
2,r2,HIGHER,1800,0.311688
3,accuracy,HIGHER,3975,0.688312
4,macro_f1,HIGHER,3975,0.688312
5,weighted_f1,HIGHER,3975,0.688312
6,runtime_seconds,LOWER,5775,1.000000



NORMALIZATION SANITY CHECK
----------------------------------------------------------------------------------------------------
mae                  : PASS | range=[0.0000, 1.0000] | invalid=0
rmse                 : PASS | range=[0.0000, 1.0000] | invalid=0
r2                   : PASS | range=[0.0000, 1.0000] | invalid=0
accuracy             : PASS | range=[0.0000, 1.0000] | invalid=0
macro_f1             : PASS | range=[0.0000, 1.0000] | invalid=0
weighted_f1          : PASS | range=[0.0000, 1.0000] | invalid=0
runtime_seconds      : PASS | range=[0.0000, 1.0000] | invalid=0

NORMALIZATION ARTIFACT
----------------------------------------------------------------------------------------------------
Saved to:
/content/drive/MyDrive/AIR_LLM_Research/data/notebook_07/metadata/normalized_metric_coverage.csv

STANDARD EVIDENCE COLUMNS AFTER NORMALIZATION
----------------------------------------------------------------------------------------------------
accuracy_score
mae_score
rmse_score


In [19]:
# ============================================================
# NOTEBOOK 07.8 — FEATURE CONTEXT
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.8")
print("LOADING FEATURE CONTEXT")
print("=" * 100)


# ------------------------------------------------------------
# Locate profile columns
# ------------------------------------------------------------

def find_profile_column(df, candidates):

    for candidate in candidates:

        candidate = str(candidate).strip().lower()

        if candidate in df.columns:
            return candidate

    return None


missing_dataset_col = find_profile_column(
    MISSINGNESS_PROFILE_DF,
    [
        "dataset_id",
        "dataset"
    ]
)

missing_feature_col = find_profile_column(
    MISSINGNESS_PROFILE_DF,
    [
        "feature",
        "feature_name",
        "column"
    ]
)

missing_rate_col = find_profile_column(
    MISSINGNESS_PROFILE_DF,
    [
        "missing_percentage",
        "missing_percent",
        "missing_rate",
        "missing_fraction"
    ]
)


print("\nPROFILE COLUMN MAPPING")
print("-" * 100)

print(
    f"Dataset column : "
    f"{missing_dataset_col}"
)

print(
    f"Feature column : "
    f"{missing_feature_col}"
)

print(
    f"Missing rate   : "
    f"{missing_rate_col}"
)


# ------------------------------------------------------------
# Build feature context
# ------------------------------------------------------------

if (
    missing_dataset_col
    and missing_feature_col
    and missing_rate_col
):

    CONTEXT_DF = (

        MISSINGNESS_PROFILE_DF[
            [
                missing_dataset_col,
                missing_feature_col,
                missing_rate_col
            ]
        ]
        .rename(
            columns={
                missing_dataset_col:
                    "dataset_id",

                missing_feature_col:
                    "feature",

                missing_rate_col:
                    "natural_missing_rate"
            }
        )
        .copy()
    )

else:

    print(
        "\nWARNING: Missingness profile does not "
        "contain the expected columns."
    )

    CONTEXT_DF = (
        STANDARD_EVIDENCE[
            [
                "dataset_id",
                "feature"
            ]
        ]
        .drop_duplicates()
        .copy()
    )

    CONTEXT_DF[
        "natural_missing_rate"
    ] = 0.0


# ------------------------------------------------------------
# Normalize identifiers
# ------------------------------------------------------------

CONTEXT_DF[
    "dataset_id"
] = (
    CONTEXT_DF[
        "dataset_id"
    ]
    .astype(str)
    .str.strip()
)

CONTEXT_DF[
    "feature"
] = (
    CONTEXT_DF[
        "feature"
    ]
    .astype(str)
    .str.strip()
)


# ------------------------------------------------------------
# Normalize natural missingness rate
# ------------------------------------------------------------

CONTEXT_DF[
    "natural_missing_rate"
] = pd.to_numeric(
    CONTEXT_DF[
        "natural_missing_rate"
    ],
    errors="coerce"
)


# ------------------------------------------------------------
# Handle percentage representation
# ------------------------------------------------------------
#
# If the profile stores values such as 10, 20, 30 rather
# than 0.10, 0.20, 0.30, convert them to fractions.
# ------------------------------------------------------------

percentage_mask = (
    CONTEXT_DF[
        "natural_missing_rate"
    ] > 1
)

CONTEXT_DF.loc[
    percentage_mask,
    "natural_missing_rate"
] = (
    CONTEXT_DF.loc[
        percentage_mask,
        "natural_missing_rate"
    ] / 100.0
)


CONTEXT_DF[
    "natural_missing_rate"
] = (
    CONTEXT_DF[
        "natural_missing_rate"
    ]
    .clip(0.0, 1.0)
)


# ------------------------------------------------------------
# Detect duplicate context keys
# ------------------------------------------------------------

duplicate_contexts = (
    CONTEXT_DF
    .duplicated(
        subset=[
            "dataset_id",
            "feature"
        ],
        keep=False
    )
)

duplicate_count = int(
    duplicate_contexts.sum()
)


print("\nFEATURE CONTEXT VALIDATION")
print("-" * 100)

print(
    f"Context rows                  : "
    f"{len(CONTEXT_DF):,}"
)

print(
    f"Unique dataset-feature pairs  : "
    f"{CONTEXT_DF[['dataset_id', 'feature']].drop_duplicates().shape[0]:,}"
)

print(
    f"Duplicate context rows        : "
    f"{duplicate_count:,}"
)


# ------------------------------------------------------------
# Resolve duplicate profile rows safely
# ------------------------------------------------------------

if duplicate_count > 0:

    print(
        "\nWARNING: Duplicate dataset-feature context "
        "records detected."
    )

    print(
        "Aggregating duplicate natural missingness "
        "rates using the mean."
    )

    CONTEXT_DF = (
        CONTEXT_DF
        .groupby(
            [
                "dataset_id",
                "feature"
            ],
            as_index=False
        )[
            "natural_missing_rate"
        ]
        .mean()
    )


# ------------------------------------------------------------
# Final uniqueness assertion
# ------------------------------------------------------------

if CONTEXT_DF.duplicated(
    subset=[
        "dataset_id",
        "feature"
    ]
).any():

    raise RuntimeError(
        "Feature context still contains duplicate "
        "dataset-feature keys after normalization."
    )


# ------------------------------------------------------------
# Preserve experimental missingness separately
# ------------------------------------------------------------
#
# IMPORTANT:
#
# natural_missing_rate
#     = missingness already present in the source profile
#
# rate
#     = controlled experimental missingness rate used
#       by Notebook 06 (e.g., 0.10–0.50)
#
# They must NOT be conflated.
# ------------------------------------------------------------

if "rate" in STANDARD_EVIDENCE.columns:

    STANDARD_EVIDENCE[
        "experimental_missingness_rate"
    ] = pd.to_numeric(
        STANDARD_EVIDENCE[
            "rate"
        ],
        errors="coerce"
    )


# ------------------------------------------------------------
# Merge feature context
# ------------------------------------------------------------

before_merge_rows = len(
    STANDARD_EVIDENCE
)


STANDARD_EVIDENCE = (
    STANDARD_EVIDENCE
    .drop(
        columns=[
            "natural_missing_rate"
        ],
        errors="ignore"
    )
    .merge(
        CONTEXT_DF,
        on=[
            "dataset_id",
            "feature"
        ],
        how="left",
        validate="many_to_one"
    )
)


after_merge_rows = len(
    STANDARD_EVIDENCE
)


# ------------------------------------------------------------
# Verify merge integrity
# ------------------------------------------------------------

if before_merge_rows != after_merge_rows:

    raise RuntimeError(
        "Feature-context merge changed the number "
        "of empirical evidence rows.\n"
        f"Before merge : {before_merge_rows:,}\n"
        f"After merge  : {after_merge_rows:,}"
    )


missing_context_rows = int(
    STANDARD_EVIDENCE[
        "natural_missing_rate"
    ].isna().sum()
)


print("\nMERGE VALIDATION")
print("-" * 100)

print(
    f"Evidence rows before merge : "
    f"{before_merge_rows:,}"
)

print(
    f"Evidence rows after merge  : "
    f"{after_merge_rows:,}"
)

print(
    f"Missing feature context    : "
    f"{missing_context_rows:,}"
)


if missing_context_rows > 0:

    print(
        "WARNING: Some experimental features do not "
        "have a matching natural missingness profile."
    )


# ------------------------------------------------------------
# Context summary
# ------------------------------------------------------------

print("\nFEATURE CONTEXT SUMMARY")
print("-" * 100)

print(
    f"Context rows               : "
    f"{len(CONTEXT_DF):,}"
)

print(
    f"Unique datasets             : "
    f"{CONTEXT_DF['dataset_id'].nunique():,}"
)

print(
    f"Unique features             : "
    f"{CONTEXT_DF['feature'].nunique():,}"
)

print(
    f"Natural missingness range   : "
    f"{CONTEXT_DF['natural_missing_rate'].min():.4f} "
    f"to "
    f"{CONTEXT_DF['natural_missing_rate'].max():.4f}"
)


# ------------------------------------------------------------
# Save feature context artifact
# ------------------------------------------------------------

FEATURE_CONTEXT_PATH = (
    METADATA_DIR /
    "feature_context.csv"
)

CONTEXT_DF.to_csv(
    FEATURE_CONTEXT_PATH,
    index=False
)


print("\nFEATURE CONTEXT ARTIFACT")
print("-" * 100)

print(
    f"Saved to:\n"
    f"{FEATURE_CONTEXT_PATH}"
)


print("=" * 100)
print("NOTEBOOK 07.8 FEATURE CONTEXT : COMPLETE")
print("=" * 100)

AIR-LLM — NOTEBOOK 07.8
LOADING FEATURE CONTEXT

PROFILE COLUMN MAPPING
----------------------------------------------------------------------------------------------------
Dataset column : dataset_id
Feature column : feature
Missing rate   : missing_percentage

FEATURE CONTEXT VALIDATION
----------------------------------------------------------------------------------------------------
Context rows                  : 80
Unique dataset-feature pairs  : 80
Duplicate context rows        : 0

MERGE VALIDATION
----------------------------------------------------------------------------------------------------
Evidence rows before merge : 5,775
Evidence rows after merge  : 5,775
Missing feature context    : 525

FEATURE CONTEXT SUMMARY
----------------------------------------------------------------------------------------------------
Context rows               : 80
Unique datasets             : 3
Unique features             : 74
Natural missingness range   : 0.0000 to 0.9684

FEATURE CONT

In [20]:
# ============================================================
# NOTEBOOK 07.9 — ADAPTIVE UTILITY WEIGHTS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.9")
print("ADAPTIVE UTILITY WEIGHTING")
print("=" * 100)


# ------------------------------------------------------------
# Research-validated baseline weights
# Consistent with Notebook 06.26
# ------------------------------------------------------------

BASE_WEIGHTS = {
    "reconstruction": 0.30,
    "distribution":   0.20,
    "dependency":     0.15,
    "downstream":     0.20,
    "efficiency":     0.15
}


# ------------------------------------------------------------
# Validate baseline weights
# ------------------------------------------------------------

BASE_WEIGHT_SUM = sum(
    BASE_WEIGHTS.values()
)

if not np.isclose(
    BASE_WEIGHT_SUM,
    1.0,
    atol=1e-10
):

    raise RuntimeError(
        "BASE_WEIGHTS must sum to 1.0."
    )


print("\nBASE UTILITY WEIGHTS")
print("-" * 100)

for key, value in BASE_WEIGHTS.items():

    print(
        f"{key:20s}: {value:.3f}"
    )


# ------------------------------------------------------------
# Adaptive weighting function
# ------------------------------------------------------------

def adaptive_weights(
    missing_rate,
    dependency_strength=0.5,
    task_relevance=0.5
):

    """
    Context-aware utility weighting.

    The Notebook 06.26 weights are used as the
    research baseline. Contextual adjustments are
    applied conservatively and renormalized to sum
    to one.
    """

    weights = BASE_WEIGHTS.copy()


    # --------------------------------------------------------
    # Normalize contextual variables
    # --------------------------------------------------------

    r = float(
        np.clip(
            pd.to_numeric(
                missing_rate,
                errors="coerce"
            )
            if pd.notna(missing_rate)
            else 0.0,
            0.0,
            1.0
        )
    )

    dep = float(
        np.clip(
            pd.to_numeric(
                dependency_strength,
                errors="coerce"
            )
            if pd.notna(dependency_strength)
            else 0.5,
            0.0,
            1.0
        )
    )

    task = float(
        np.clip(
            pd.to_numeric(
                task_relevance,
                errors="coerce"
            )
            if pd.notna(task_relevance)
            else 0.5,
            0.0,
            1.0
        )
    )


    # --------------------------------------------------------
    # Context-sensitive adjustments
    # --------------------------------------------------------

    # Higher missingness:
    # reconstruction becomes more important.
    weights["reconstruction"] += (
        0.10 * r
    )


    # Strong dependency:
    # dependency preservation becomes more important.
    weights["dependency"] += (
        0.10 * dep
    )


    # Strong downstream relevance:
    # downstream predictive utility becomes more important.
    weights["downstream"] += (
        0.10 * task
    )


    # Higher missingness generally increases computational
    # burden; efficiency remains relevant but receives a
    # modest relative reduction.
    weights["efficiency"] -= (
        0.05 * r
    )


    # --------------------------------------------------------
    # Prevent negative weights
    # --------------------------------------------------------

    weights = {
        key: max(
            float(value),
            0.0
        )
        for key, value in weights.items()
    }


    # --------------------------------------------------------
    # Renormalize
    # --------------------------------------------------------

    total = sum(
        weights.values()
    )

    if (
        not np.isfinite(total)
        or total <= 0
    ):

        return BASE_WEIGHTS.copy()


    weights = {
        key:
            value / total
        for key, value in weights.items()
    }


    return weights


# ------------------------------------------------------------
# Context extraction helper
# ------------------------------------------------------------

def get_profile_value(
    df,
    dataset_id,
    feature,
    candidate_columns,
    default=0.5
):

    if df is None or df.empty:
        return default


    dataset_col = find_profile_column(
        df,
        [
            "dataset_id",
            "dataset"
        ]
    )

    feature_col = find_profile_column(
        df,
        [
            "feature",
            "feature_name",
            "column"
        ]
    )


    if (
        dataset_col is None
        or feature_col is None
    ):

        return default


    subset = df[
        (
            df[dataset_col]
            .astype(str)
            ==
            str(dataset_id)
        )
        &
        (
            df[feature_col]
            .astype(str)
            ==
            str(feature)
        )
    ]


    if subset.empty:
        return default


    for column in candidate_columns:

        if column not in subset.columns:
            continue


        value = pd.to_numeric(
            subset.iloc[0][column],
            errors="coerce"
        )


        if pd.notna(value):

            return float(
                np.clip(
                    value,
                    0.0,
                    1.0
                )
            )


    return default


# ------------------------------------------------------------
# Identify available feature-profile information
# ------------------------------------------------------------

FEATURE_PROFILE_COLUMNS = set(
    FEATURE_PROFILE_DF.columns
)


DEPENDENCY_CANDIDATES = [
    "dependency_strength",
    "dependency_score",
    "correlation_strength",
    "correlation",
    "mean_correlation",
    "pearson_correlation",
    "spearman_correlation",
    "mutual_information",
    "mutual_info"
]


TASK_RELEVANCE_CANDIDATES = [
    "task_relevance",
    "target_relevance",
    "predictive_relevance",
    "target_correlation",
    "target_association",
    "target_dependency",
    "mutual_information_target",
    "target_mutual_information"
]


AVAILABLE_DEPENDENCY_COLUMNS = [
    column
    for column in DEPENDENCY_CANDIDATES
    if column in FEATURE_PROFILE_COLUMNS
]


AVAILABLE_TASK_COLUMNS = [
    column
    for column in TASK_RELEVANCE_CANDIDATES
    if column in FEATURE_PROFILE_COLUMNS
]


print("\nCONTEXT SIGNAL AVAILABILITY")
print("-" * 100)

print(
    "Dependency columns :",
    AVAILABLE_DEPENDENCY_COLUMNS
    if AVAILABLE_DEPENDENCY_COLUMNS
    else "NONE"
)

print(
    "Task relevance columns :",
    AVAILABLE_TASK_COLUMNS
    if AVAILABLE_TASK_COLUMNS
    else "NONE"
)


# ------------------------------------------------------------
# Create adaptive weight profiles
# ------------------------------------------------------------

WEIGHT_ROWS = []


FEATURE_CONTEXTS = (
    STANDARD_EVIDENCE[
        [
            "dataset_id",
            "feature"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "dataset_id",
            "feature"
        ]
    )
)


for row in FEATURE_CONTEXTS.itertuples(
    index=False
):

    dataset_id = row.dataset_id
    feature = row.feature


    # --------------------------------------------------------
    # Missingness context
    # --------------------------------------------------------

    context = CONTEXT_DF[
        (
            CONTEXT_DF["dataset_id"]
            .astype(str)
            ==
            str(dataset_id)
        )
        &
        (
            CONTEXT_DF["feature"]
            .astype(str)
            ==
            str(feature)
        )
    ]


    if context.empty:

        missing_rate = 0.0

    else:

        missing_rate = float(
            pd.to_numeric(
                context.iloc[0][
                    "natural_missing_rate"
                ],
                errors="coerce"
            )
        )

        if not np.isfinite(
            missing_rate
        ):

            missing_rate = 0.0


    # --------------------------------------------------------
    # Dependency context
    # --------------------------------------------------------

    if AVAILABLE_DEPENDENCY_COLUMNS:

        dependency_strength = (
            get_profile_value(
                FEATURE_PROFILE_DF,
                dataset_id,
                feature,
                AVAILABLE_DEPENDENCY_COLUMNS,
                default=0.5
            )
        )

    else:

        dependency_strength = 0.5


    # --------------------------------------------------------
    # Task relevance context
    # --------------------------------------------------------

    if AVAILABLE_TASK_COLUMNS:

        task_relevance = (
            get_profile_value(
                FEATURE_PROFILE_DF,
                dataset_id,
                feature,
                AVAILABLE_TASK_COLUMNS,
                default=0.5
            )
        )

    else:

        task_relevance = 0.5


    # --------------------------------------------------------
    # Calculate adaptive weights
    # --------------------------------------------------------

    weights = adaptive_weights(
        missing_rate=missing_rate,
        dependency_strength=dependency_strength,
        task_relevance=task_relevance
    )


    WEIGHT_ROWS.append({

        "dataset_id":
            dataset_id,

        "feature":
            feature,

        "missing_rate":
            missing_rate,

        "dependency_strength":
            dependency_strength,

        "task_relevance":
            task_relevance,

        "weight_reconstruction":
            weights["reconstruction"],

        "weight_distribution":
            weights["distribution"],

        "weight_dependency":
            weights["dependency"],

        "weight_downstream":
            weights["downstream"],

        "weight_efficiency":
            weights["efficiency"]
    })


WEIGHTS_DF = pd.DataFrame(
    WEIGHT_ROWS
)


# ------------------------------------------------------------
# Validate adaptive weights
# ------------------------------------------------------------

WEIGHT_COLUMNS = [
    "weight_reconstruction",
    "weight_distribution",
    "weight_dependency",
    "weight_downstream",
    "weight_efficiency"
]


if WEIGHTS_DF.empty:

    raise RuntimeError(
        "No adaptive weight profiles were generated."
    )


WEIGHTS_DF[
    "weight_sum"
] = WEIGHTS_DF[
    WEIGHT_COLUMNS
].sum(
    axis=1
)


WEIGHT_SUM_VALID = np.allclose(
    WEIGHTS_DF["weight_sum"],
    1.0,
    atol=1e-8
)


if not WEIGHT_SUM_VALID:

    raise RuntimeError(
        "Adaptive utility weights do not sum to 1.0."
    )


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\nADAPTIVE WEIGHT SUMMARY")
print("-" * 100)

print(
    f"Weight profiles generated : "
    f"{len(WEIGHTS_DF):,}"
)

print(
    f"Weight sums valid          : "
    f"{WEIGHT_SUM_VALID}"
)

print(
    f"Dependency signal          : "
    f"{'AVAILABLE' if AVAILABLE_DEPENDENCY_COLUMNS else 'NOT AVAILABLE — default 0.5'}"
)

print(
    f"Task relevance signal      : "
    f"{'AVAILABLE' if AVAILABLE_TASK_COLUMNS else 'NOT AVAILABLE — default 0.5'}"
)


display(
    WEIGHTS_DF.head(10)
)


print("\nNOTEBOOK 07.9 : COMPLETE")

AIR-LLM — NOTEBOOK 07.9
ADAPTIVE UTILITY WEIGHTING

BASE UTILITY WEIGHTS
----------------------------------------------------------------------------------------------------
reconstruction      : 0.300
distribution        : 0.200
dependency          : 0.150
downstream          : 0.200
efficiency          : 0.150

CONTEXT SIGNAL AVAILABILITY
----------------------------------------------------------------------------------------------------
Dependency columns : ['mutual_information']
Task relevance columns : NONE

ADAPTIVE WEIGHT SUMMARY
----------------------------------------------------------------------------------------------------
Weight profiles generated : 77
Weight sums valid          : True
Dependency signal          : AVAILABLE
Task relevance signal      : NOT AVAILABLE — default 0.5


,dataset_id,feature,missing_rate,dependency_strength,task_relevance,weight_reconstruction,weight_distribution,weight_dependency,weight_downstream,weight_efficiency,weight_sum
0,adult_income,age,0.000000,0.057178,0.5,0.284167,0.189445,0.147499,0.236806,0.142083,1.0
1,adult_income,capital_gain,0.000000,0.083629,0.5,0.283457,0.188971,0.149630,0.236214,0.141728,1.0
2,adult_income,capital_loss,0.000000,0.036198,0.5,0.284733,0.189822,0.145802,0.237277,0.142366,1.0
3,adult_income,education,0.000000,0.065856,0.5,0.283933,0.189289,0.148200,0.236611,0.141967,1.0
4,adult_income,education_num,0.000000,0.071534,0.5,0.283781,0.189187,0.148657,0.236484,0.141890,1.0
5,adult_income,fnlwgt,0.000000,0.013443,0.5,0.285349,0.190233,0.143953,0.237791,0.142674,1.0
6,adult_income,hours_per_week,0.000000,0.035510,0.5,0.284751,0.189834,0.145746,0.237293,0.142376,1.0
7,adult_income,marital_status,0.000000,0.104163,0.5,0.282908,0.188605,0.151277,0.235756,0.141454,1.0
8,adult_income,native_country,0.017857,0.007695,0.5,0.286961,0.190175,0.143363,0.237719,0.141782,1.0
9,adult_income,occupation,0.056204,0.061990,0.5,0.288591,0.188856,0.147495,0.236070,0.138988,1.0



NOTEBOOK 07.9 : COMPLETE


In [23]:
# ============================================================
# NOTEBOOK 07.10 — MULTI-OBJECTIVE UTILITY
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.10")
print("MULTI-OBJECTIVE UTILITY CALCULATION")
print("=" * 100)


# ------------------------------------------------------------
# Research-validated utility dimensions
# Consistent with Notebook 06.26
# ------------------------------------------------------------

UTILITY_DIMENSIONS = [
    "reconstruction",
    "distribution",
    "dependency",
    "downstream",
    "efficiency"
]


# ------------------------------------------------------------
# Ensure required empirical columns exist
# ------------------------------------------------------------

EMPIRICAL_METRICS = [
    "mae",
    "rmse",
    "r2",
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "runtime_seconds"
]


missing_empirical_metrics = [
    column
    for column in EMPIRICAL_METRICS
    if column not in STANDARD_EVIDENCE.columns
]


if missing_empirical_metrics:

    raise RuntimeError(
        "Required empirical metric columns are missing:\n"
        + "\n".join(
            missing_empirical_metrics
        )
    )


# ------------------------------------------------------------
# Merge adaptive weights
# ------------------------------------------------------------

UTILITY_DF = (
    STANDARD_EVIDENCE
    .merge(
        WEIGHTS_DF,
        on=[
            "dataset_id",
            "feature"
        ],
        how="left",
        validate="many_to_one"
    )
)


# ------------------------------------------------------------
# Robust normalization helper
# ------------------------------------------------------------

def normalize_metric_group(
    df,
    metric,
    group_columns,
    higher_is_better
):

    output = pd.Series(
        np.nan,
        index=df.index,
        dtype=float
    )


    for _, group_index in df.groupby(
        group_columns,
        dropna=False
    ).groups.items():

        values = pd.to_numeric(
            df.loc[
                group_index,
                metric
            ],
            errors="coerce"
        )


        valid = values.dropna()


        if valid.empty:
            continue


        q_low = valid.quantile(
            0.05
        )

        q_high = valid.quantile(
            0.95
        )


        if (
            not np.isfinite(q_low)
            or
            not np.isfinite(q_high)
            or
            q_high <= q_low
        ):

            normalized = pd.Series(
                0.5,
                index=values.index
            )

        else:

            clipped = values.clip(
                lower=q_low,
                upper=q_high
            )

            normalized = (
                clipped - q_low
            ) / (
                q_high - q_low
            )


        if not higher_is_better:

            normalized = (
                1.0 - normalized
            )


        output.loc[
            group_index
        ] = normalized


    return output.clip(
        0.0,
        1.0
    )


# ------------------------------------------------------------
# Reconstruction score
#
# Numerical features:
# MAE and RMSE
#
# Categorical features:
# Accuracy / F1 are downstream/task-oriented metrics,
# therefore reconstruction is left unavailable here.
# ------------------------------------------------------------

UTILITY_DF[
    "mae"
] = pd.to_numeric(
    UTILITY_DF["mae"],
    errors="coerce"
)

UTILITY_DF[
    "rmse"
] = pd.to_numeric(
    UTILITY_DF["rmse"],
    errors="coerce"
)


UTILITY_DF[
    "mae_score"
] = normalize_metric_group(
    UTILITY_DF,
    "mae",
    [
        "dataset_id",
        "feature"
    ],
    higher_is_better=False
)


UTILITY_DF[
    "rmse_score"
] = normalize_metric_group(
    UTILITY_DF,
    "rmse",
    [
        "dataset_id",
        "feature"
    ],
    higher_is_better=False
)


reconstruction_components = [
    column
    for column in [
        "mae_score",
        "rmse_score"
    ]
    if column in UTILITY_DF.columns
]


UTILITY_DF[
    "reconstruction_score"
] = (
    UTILITY_DF[
        reconstruction_components
    ]
    .mean(
        axis=1,
        skipna=True
    )
)


# Require at least one valid reconstruction metric
UTILITY_DF.loc[
    UTILITY_DF[
        reconstruction_components
    ]
    .notna()
    .sum(axis=1)
    == 0,
    "reconstruction_score"
] = np.nan


# ------------------------------------------------------------
# Distribution score
#
# Notebook 06.21 currently does not contain Wasserstein,
# JS divergence or another distributional metric.
# Therefore this dimension remains unavailable rather
# than being fabricated.
# ------------------------------------------------------------

UTILITY_DF[
    "distribution_score"
] = np.nan


# ------------------------------------------------------------
# Dependency score
#
# Notebook 06.21 currently does not contain dependency
# preservation metrics.
# Therefore this dimension remains unavailable.
# ------------------------------------------------------------

UTILITY_DF[
    "dependency_score"
] = np.nan


# ------------------------------------------------------------
# Downstream predictive score
#
# Numerical:
# R2
#
# Categorical:
# Accuracy, Macro-F1, Weighted-F1
# ------------------------------------------------------------

for metric in [
    "r2",
    "accuracy",
    "macro_f1",
    "weighted_f1"
]:

    UTILITY_DF[
        metric
    ] = pd.to_numeric(
        UTILITY_DF[metric],
        errors="coerce"
    )


predictive_components = []


for metric in [
    "r2",
    "accuracy",
    "macro_f1",
    "weighted_f1"
]:

    score_column = (
        f"{metric}_score"
    )


    UTILITY_DF[
        score_column
    ] = normalize_metric_group(
        UTILITY_DF,
        metric,
        [
            "dataset_id",
            "feature"
        ],
        higher_is_better=True
    )


    predictive_components.append(
        score_column
    )


UTILITY_DF[
    "downstream_score"
] = (
    UTILITY_DF[
        predictive_components
    ]
    .mean(
        axis=1,
        skipna=True
    )
)


UTILITY_DF.loc[
    UTILITY_DF[
        predictive_components
    ]
    .notna()
    .sum(axis=1)
    == 0,
    "downstream_score"
] = np.nan


# ------------------------------------------------------------
# Computational efficiency score
#
# Lower runtime is better.
# ------------------------------------------------------------

UTILITY_DF[
    "runtime_seconds"
] = pd.to_numeric(
    UTILITY_DF[
        "runtime_seconds"
    ],
    errors="coerce"
)


UTILITY_DF[
    "efficiency_score"
] = normalize_metric_group(
    UTILITY_DF,
    "runtime_seconds",
    [
        "dataset_id",
        "feature"
    ],
    higher_is_better=False
)


# ------------------------------------------------------------
# Verify adaptive weights
# ------------------------------------------------------------

WEIGHT_COLUMNS = [
    "weight_reconstruction",
    "weight_distribution",
    "weight_dependency",
    "weight_downstream",
    "weight_efficiency"
]


missing_weight_columns = [
    column
    for column in WEIGHT_COLUMNS
    if column not in UTILITY_DF.columns
]


if missing_weight_columns:

    raise RuntimeError(
        "Adaptive weight columns are missing:\n"
        + "\n".join(
            missing_weight_columns
        )
    )


# ------------------------------------------------------------
# Determine empirical availability
# ------------------------------------------------------------

for dimension in UTILITY_DIMENSIONS:

    UTILITY_DF[
        f"{dimension}_available"
    ] = (
        UTILITY_DF[
            f"{dimension}_score"
        ]
        .notna()
    )


# ------------------------------------------------------------
# Available weight
#
# Only dimensions supported by actual empirical evidence
# participate in the composite utility.
# ------------------------------------------------------------

UTILITY_DF[
    "available_weight_sum"
] = 0.0


for dimension in UTILITY_DIMENSIONS:

    mask = UTILITY_DF[
        f"{dimension}_available"
    ]


    UTILITY_DF.loc[
        mask,
        "available_weight_sum"
    ] += UTILITY_DF.loc[
        mask,
        f"weight_{dimension}"
    ]


# ------------------------------------------------------------
# Effective normalized weights
# ------------------------------------------------------------

for dimension in UTILITY_DIMENSIONS:

    UTILITY_DF[
        f"effective_weight_{dimension}"
    ] = 0.0


    mask = (
        UTILITY_DF[
            f"{dimension}_available"
        ]
        &
        (
            UTILITY_DF[
                "available_weight_sum"
            ] > 0
        )
    )


    UTILITY_DF.loc[
        mask,
        f"effective_weight_{dimension}"
    ] = (
        UTILITY_DF.loc[
            mask,
            f"weight_{dimension}"
        ]
        /
        UTILITY_DF.loc[
            mask,
            "available_weight_sum"
        ]
    )


# ------------------------------------------------------------
# Dimension-level utility contributions
# ------------------------------------------------------------

for dimension in UTILITY_DIMENSIONS:

    UTILITY_DF[
        f"utility_{dimension}"
    ] = (
        UTILITY_DF[
            f"{dimension}_score"
        ].fillna(0.0)
        *
        UTILITY_DF[
            f"effective_weight_{dimension}"
        ]
    )


# ------------------------------------------------------------
# Composite adaptive utility
# ------------------------------------------------------------

UTILITY_DF[
    "adaptive_utility"
] = 0.0


for dimension in UTILITY_DIMENSIONS:

    UTILITY_DF[
        "adaptive_utility"
    ] += UTILITY_DF[
        f"utility_{dimension}"
    ]


UTILITY_DF[
    "adaptive_utility"
] = (
    UTILITY_DF[
        "adaptive_utility"
    ]
    .clip(
        0.0,
        1.0
    )
)


# ------------------------------------------------------------
# Utility coverage
# ------------------------------------------------------------

UTILITY_DF[
    "utility_dimensions_available"
] = (
    UTILITY_DF[
        [
            f"{dimension}_available"
            for dimension in UTILITY_DIMENSIONS
        ]
    ]
    .sum(
        axis=1
    )
)


UTILITY_DF[
    "utility_coverage"
] = (
    UTILITY_DF[
        "utility_dimensions_available"
    ]
    /
    len(UTILITY_DIMENSIONS)
)


# ------------------------------------------------------------
# Utility validity
# ------------------------------------------------------------

UTILITY_DF[
    "utility_valid"
] = (
    UTILITY_DF[
        "available_weight_sum"
    ]
    > 0
)


# ------------------------------------------------------------
# Effective weight validation
# ------------------------------------------------------------

UTILITY_DF[
    "effective_weight_sum"
] = (
    UTILITY_DF[
        [
            f"effective_weight_{dimension}"
            for dimension in UTILITY_DIMENSIONS
        ]
    ]
    .sum(
        axis=1
    )
)


valid_rows = (
    UTILITY_DF[
        "utility_valid"
    ]
)


if not np.allclose(
    UTILITY_DF.loc[
        valid_rows,
        "effective_weight_sum"
    ],
    1.0,
    atol=1e-8
):

    raise RuntimeError(
        "Effective utility weights failed validation."
    )


# ------------------------------------------------------------
# Coverage report
# ------------------------------------------------------------

print("\nEMPIRICAL UTILITY COVERAGE")
print("-" * 100)

for dimension in UTILITY_DIMENSIONS:

    valid_count = int(
        UTILITY_DF[
            f"{dimension}_available"
        ].sum()
    )

    coverage = (
        valid_count /
        len(UTILITY_DF)
        if len(UTILITY_DF)
        else 0.0
    )

    print(
        f"{dimension:20s} : "
        f"{valid_count:>6,} / "
        f"{len(UTILITY_DF):,} "
        f"({coverage:.2%})"
    )


# ------------------------------------------------------------
# Composite utility summary
# ------------------------------------------------------------

print("\nCOMPOSITE UTILITY SUMMARY")
print("-" * 100)

print(
    f"Valid utility rows : "
    f"{UTILITY_DF['utility_valid'].sum():,}"
)

print(
    f"Mean utility       : "
    f"{UTILITY_DF['adaptive_utility'].mean():.6f}"
)

print(
    f"Median utility     : "
    f"{UTILITY_DF['adaptive_utility'].median():.6f}"
)

print(
    f"Minimum utility    : "
    f"{UTILITY_DF['adaptive_utility'].min():.6f}"
)

print(
    f"Maximum utility    : "
    f"{UTILITY_DF['adaptive_utility'].max():.6f}"
)


# ------------------------------------------------------------
# Sample output
# ------------------------------------------------------------

print("\nSAMPLE MULTI-OBJECTIVE UTILITY")
print("-" * 100)


DISPLAY_COLUMNS = [
    "evaluation_key",
    "dataset_id",
    "feature",
    "strategy",
    "mechanism",
    "rate",
    "repetition",
    "reconstruction_score",
    "distribution_score",
    "dependency_score",
    "downstream_score",
    "efficiency_score",
    "effective_weight_reconstruction",
    "effective_weight_distribution",
    "effective_weight_dependency",
    "effective_weight_downstream",
    "effective_weight_efficiency",
    "adaptive_utility",
    "utility_dimensions_available",
    "utility_coverage"
]


DISPLAY_COLUMNS = [
    column
    for column in DISPLAY_COLUMNS
    if column in UTILITY_DF.columns
]


display(
    UTILITY_DF[
        DISPLAY_COLUMNS
    ].head(15)
)


print("=" * 100)
print("NOTEBOOK 07.10 — MULTI-OBJECTIVE UTILITY : COMPLETE")
print("=" * 100)

AIR-LLM — NOTEBOOK 07.10
MULTI-OBJECTIVE UTILITY CALCULATION

EMPIRICAL UTILITY COVERAGE
----------------------------------------------------------------------------------------------------
reconstruction       :  1,800 / 5,775 (31.17%)
distribution         :      0 / 5,775 (0.00%)
dependency           :      0 / 5,775 (0.00%)
downstream           :  5,775 / 5,775 (100.00%)
efficiency           :  5,775 / 5,775 (100.00%)

COMPOSITE UTILITY SUMMARY
----------------------------------------------------------------------------------------------------
Valid utility rows : 5,775
Mean utility       : 0.638672
Median utility     : 0.687664
Minimum utility    : 0.000000
Maximum utility    : 1.000000

SAMPLE MULTI-OBJECTIVE UTILITY
----------------------------------------------------------------------------------------------------


,evaluation_key,dataset_id,feature,strategy,mechanism,rate,repetition,reconstruction_score,distribution_score,dependency_score,downstream_score,efficiency_score,effective_weight_reconstruction,effective_weight_distribution,effective_weight_dependency,effective_weight_downstream,effective_weight_efficiency,adaptive_utility,utility_dimensions_available,utility_coverage
0,8b8d979f3253c12107b63163,adult_income,age,mean,MAR,0.1,1,0.527843,NaN,NaN,0.783278,0.821487,0.428571,0.0,0.0,0.357143,0.214286,0.681993,3,0.6
1,54811d482230bd847f01f2b7,adult_income,age,mean,MAR,0.1,2,0.527843,NaN,NaN,0.783278,0.000000,0.428571,0.0,0.0,0.357143,0.214286,0.505961,3,0.6
2,e30a0e15815828c68cb864b6,adult_income,age,mean,MAR,0.1,3,0.527843,NaN,NaN,0.783278,0.925594,0.428571,0.0,0.0,0.357143,0.214286,0.704302,3,0.6
3,89feffbaf84ff25e810b53d1,adult_income,age,mean,MAR,0.1,4,0.527843,NaN,NaN,0.783278,0.968097,0.428571,0.0,0.0,0.357143,0.214286,0.713410,3,0.6
4,7cb75e5870163cf2b564d213,adult_income,age,mean,MAR,0.1,5,0.527843,NaN,NaN,0.783278,0.818830,0.428571,0.0,0.0,0.357143,0.214286,0.681424,3,0.6
5,9009c0b4f43619f556271df8,adult_income,age,mean,MAR,0.2,1,0.622286,NaN,NaN,0.862099,0.000000,0.428571,0.0,0.0,0.357143,0.214286,0.574587,3,0.6
6,1574667a7f3ee1852cbb371f,adult_income,age,mean,MAR,0.2,2,0.622286,NaN,NaN,0.862099,0.000000,0.428571,0.0,0.0,0.357143,0.214286,0.574587,3,0.6
7,b9e47d23cca6d7000713c5ba,adult_income,age,mean,MAR,0.2,3,0.622286,NaN,NaN,0.862099,0.901180,0.428571,0.0,0.0,0.357143,0.214286,0.767697,3,0.6
8,cbc838fa5b563135c3336ed7,adult_income,age,mean,MAR,0.2,4,0.622286,NaN,NaN,0.862099,0.861334,0.428571,0.0,0.0,0.357143,0.214286,0.759158,3,0.6
9,04e3b63a4a716641bdf6d929,adult_income,age,mean,MAR,0.2,5,0.622286,NaN,NaN,0.862099,0.056810,0.428571,0.0,0.0,0.357143,0.214286,0.586760,3,0.6


NOTEBOOK 07.10 — MULTI-OBJECTIVE UTILITY : COMPLETE


In [25]:
# ============================================================
# NOTEBOOK 07.11 — ROBUST SCENARIO AGGREGATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.11")
print("ROBUST AGGREGATION ACROSS EXPERIMENTAL CONDITIONS")
print("=" * 100)


# ------------------------------------------------------------
# Validate required columns
# ------------------------------------------------------------

REQUIRED_COLUMNS = [
    "dataset_id",
    "feature",
    "strategy",
    "adaptive_utility"
]

missing_columns = [
    column
    for column in REQUIRED_COLUMNS
    if column not in UTILITY_DF.columns
]

if missing_columns:

    raise RuntimeError(
        "Required columns missing from UTILITY_DF:\n"
        + "\n".join(missing_columns)
    )


# ------------------------------------------------------------
# Aggregation dimensions
# ------------------------------------------------------------

GROUP_COLUMNS = [
    "dataset_id",
    "feature",
    "strategy"
]


# ------------------------------------------------------------
# Robust scenario aggregation
# ------------------------------------------------------------

AGGREGATION_DF = (
    UTILITY_DF
    .groupby(
        GROUP_COLUMNS,
        dropna=False
    )
    .agg(

        mean_utility=(
            "adaptive_utility",
            "mean"
        ),

        median_utility=(
            "adaptive_utility",
            "median"
        ),

        utility_std=(
            "adaptive_utility",
            "std"
        ),

        min_utility=(
            "adaptive_utility",
            "min"
        ),

        max_utility=(
            "adaptive_utility",
            "max"
        ),

        evidence_count=(
            "adaptive_utility",
            "count"
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# Fill standard deviation for single observations
# ------------------------------------------------------------

AGGREGATION_DF[
    "utility_std"
] = (
    AGGREGATION_DF[
        "utility_std"
    ]
    .fillna(0.0)
)


# ------------------------------------------------------------
# Experimental coverage
# ------------------------------------------------------------

if "mechanism" in UTILITY_DF.columns:

    mechanism_counts = (
        UTILITY_DF
        .groupby(
            GROUP_COLUMNS,
            dropna=False
        )["mechanism"]
        .nunique()
        .reset_index(
            name="mechanism_count"
        )
    )

    AGGREGATION_DF = AGGREGATION_DF.merge(
        mechanism_counts,
        on=GROUP_COLUMNS,
        how="left"
    )

else:

    AGGREGATION_DF[
        "mechanism_count"
    ] = np.nan


if "rate" in UTILITY_DF.columns:

    rate_counts = (
        UTILITY_DF
        .groupby(
            GROUP_COLUMNS,
            dropna=False
        )["rate"]
        .nunique()
        .reset_index(
            name="missingness_rate_count"
        )
    )

    AGGREGATION_DF = AGGREGATION_DF.merge(
        rate_counts,
        on=GROUP_COLUMNS,
        how="left"
    )

else:

    AGGREGATION_DF[
        "missingness_rate_count"
    ] = np.nan


if "repetition" in UTILITY_DF.columns:

    repetition_counts = (
        UTILITY_DF
        .groupby(
            GROUP_COLUMNS,
            dropna=False
        )["repetition"]
        .nunique()
        .reset_index(
            name="repetition_count"
        )
    )

    AGGREGATION_DF = AGGREGATION_DF.merge(
        repetition_counts,
        on=GROUP_COLUMNS,
        how="left"
    )

else:

    AGGREGATION_DF[
        "repetition_count"
    ] = np.nan


# ------------------------------------------------------------
# Utility coverage
# ------------------------------------------------------------

AGGREGATION_DF[
    "utility_coverage"
] = (
    AGGREGATION_DF[
        "evidence_count"
    ]
    /
    UTILITY_DF
    .groupby(
        GROUP_COLUMNS,
        dropna=False
    )
    .size()
    .reindex(
        pd.MultiIndex.from_frame(
            AGGREGATION_DF[
                GROUP_COLUMNS
            ]
        )
    )
    .to_numpy()
)


AGGREGATION_DF[
    "utility_coverage"
] = (
    AGGREGATION_DF[
        "utility_coverage"
    ]
    .fillna(0.0)
    .clip(0.0, 1.0)
)


# ------------------------------------------------------------
# Robust utility
#
# Conservative aggregation:
#
# 50% mean performance
# 30% median performance
# 20% worst-case performance
# ------------------------------------------------------------

ROBUST_MEAN_WEIGHT = 0.50
ROBUST_MEDIAN_WEIGHT = 0.30
ROBUST_MIN_WEIGHT = 0.20


AGGREGATION_DF[
    "robust_utility"
] = (

    ROBUST_MEAN_WEIGHT
    *
    AGGREGATION_DF[
        "mean_utility"
    ]

    +

    ROBUST_MEDIAN_WEIGHT
    *
    AGGREGATION_DF[
        "median_utility"
    ]

    +

    ROBUST_MIN_WEIGHT
    *
    AGGREGATION_DF[
        "min_utility"
    ]
)


AGGREGATION_DF[
    "robust_utility"
] = (
    AGGREGATION_DF[
        "robust_utility"
    ]
    .clip(0.0, 1.0)
)


# ------------------------------------------------------------
# Stability indicator
# ------------------------------------------------------------

AGGREGATION_DF[
    "utility_stability"
] = (
    1.0
    -
    AGGREGATION_DF[
        "utility_std"
    ]
    .clip(0.0, 1.0)
).clip(
    0.0,
    1.0
)


# ------------------------------------------------------------
# Rank strategies within each dataset-feature context
# ------------------------------------------------------------

AGGREGATION_DF[
    "robust_rank"
] = (
    AGGREGATION_DF
    .groupby(
        [
            "dataset_id",
            "feature"
        ],
        dropna=False
    )[
        "robust_utility"
    ]
    .rank(
        ascending=False,
        method="min"
    )
)


# ------------------------------------------------------------
# Final ordering
# ------------------------------------------------------------

AGGREGATION_DF = (
    AGGREGATION_DF
    .sort_values(
        [
            "dataset_id",
            "feature",
            "robust_rank",
            "robust_utility"
        ],
        ascending=[
            True,
            True,
            True,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print(
    f"\nCandidate-feature combinations : "
    f"{len(AGGREGATION_DF):,}"
)

print(
    f"Datasets                       : "
    f"{AGGREGATION_DF['dataset_id'].nunique():,}"
)

print(
    f"Features                       : "
    f"{AGGREGATION_DF['feature'].nunique():,}"
)

print(
    f"Strategies                     : "
    f"{AGGREGATION_DF['strategy'].nunique():,}"
)


print("\nROBUST UTILITY SUMMARY")
print("-" * 100)

display(
    AGGREGATION_DF[
        [
            "dataset_id",
            "feature",
            "strategy",
            "mean_utility",
            "median_utility",
            "utility_std",
            "min_utility",
            "max_utility",
            "evidence_count",
            "utility_coverage",
            "utility_stability",
            "robust_utility",
            "robust_rank"
        ]
    ]
    .head(20)
)


# ------------------------------------------------------------
# Validate expected evidence structure
# ------------------------------------------------------------

print("\nEVIDENCE VALIDATION")
print("-" * 100)

if "mechanism_count" in AGGREGATION_DF.columns:

    print(
        "Mechanism coverage : "
        f"{AGGREGATION_DF['mechanism_count'].dropna().unique()}"
    )

if "missingness_rate_count" in AGGREGATION_DF.columns:

    print(
        "Rate coverage      : "
        f"{AGGREGATION_DF['missingness_rate_count'].dropna().unique()}"
    )

if "repetition_count" in AGGREGATION_DF.columns:

    print(
        "Repetition coverage: "
        f"{AGGREGATION_DF['repetition_count'].dropna().unique()}"
    )


print("\n" + "=" * 100)
print("NOTEBOOK 07.11 ROBUST AGGREGATION : COMPLETE")
print("=" * 100)

AIR-LLM — NOTEBOOK 07.11
ROBUST AGGREGATION ACROSS EXPERIMENTAL CONDITIONS

Candidate-feature combinations : 77
Datasets                       : 3
Features                       : 73
Strategies                     : 2

ROBUST UTILITY SUMMARY
----------------------------------------------------------------------------------------------------


,dataset_id,feature,strategy,mean_utility,median_utility,utility_std,min_utility,max_utility,evidence_count,utility_coverage,utility_stability,robust_utility,robust_rank
0,adult_income,age,mean,0.722343,0.790036,0.234720,0.000000,0.983488,75,1.0,0.765280,0.598182,1.0
1,adult_income,capital_gain,mean,0.740090,0.784632,0.210009,0.012780,0.992284,75,1.0,0.789991,0.607990,1.0
2,adult_income,capital_loss,mean,0.762644,0.819713,0.228779,0.000000,0.994707,75,1.0,0.771221,0.627236,1.0
3,adult_income,education,mode,0.620751,0.720256,0.229169,0.091886,0.891273,75,1.0,0.770831,0.544829,1.0
4,adult_income,education_num,mean,0.659110,0.703001,0.256796,0.049761,0.991037,75,1.0,0.743204,0.550408,1.0
5,adult_income,fnlwgt,mean,0.672078,0.695207,0.228531,0.099540,0.987668,75,1.0,0.771469,0.564509,1.0
6,adult_income,hours_per_week,mean,0.699846,0.736271,0.234175,0.085893,0.995343,75,1.0,0.765825,0.587983,1.0
7,adult_income,marital_status,mode,0.586039,0.652476,0.262264,0.000000,0.978921,75,1.0,0.737736,0.488762,1.0
8,adult_income,native_country,mode,0.603695,0.589465,0.224806,0.265097,1.000000,75,1.0,0.775194,0.531706,1.0
9,adult_income,occupation,mode,0.465131,0.357742,0.260096,0.000000,0.997336,75,1.0,0.739904,0.339888,1.0



EVIDENCE VALIDATION
----------------------------------------------------------------------------------------------------
Mechanism coverage : [3]
Rate coverage      : [5]
Repetition coverage: [5]

NOTEBOOK 07.11 ROBUST AGGREGATION : COMPLETE


In [26]:
# ============================================================
# NOTEBOOK 07.12 — CROSS-CONDITION STABILITY
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.12")
print("CROSS-CONDITION STABILITY")
print("=" * 100)


def calculate_stability(group):

    values = group[
        "adaptive_utility"
    ].dropna()

    if len(values) <= 1:
        return 1.0

    mean_value = values.mean()

    if mean_value <= 1e-12:
        return 0.0

    cv = (
        values.std(ddof=0)
        / mean_value
    )

    stability = 1.0 / (
        1.0 + cv
    )

    return float(
        np.clip(
            stability,
            0,
            1
        )
    )


STABILITY_ROWS = []

for keys, group in UTILITY_DF.groupby(
    GROUP_COLUMNS,
    dropna=False
):

    dataset_id, feature, strategy = keys

    STABILITY_ROWS.append({

        "dataset_id":
            dataset_id,

        "feature":
            feature,

        "strategy":
            strategy,

        "stability":
            calculate_stability(group),

        "condition_count":
            int(
                group[
                    "adaptive_utility"
                ].count()
            )
    })


STABILITY_DF = pd.DataFrame(
    STABILITY_ROWS
)


AGGREGATION_DF = AGGREGATION_DF.merge(
    STABILITY_DF,
    on=GROUP_COLUMNS,
    how="left"
)


print(
    f"Stability profiles: "
    f"{len(STABILITY_DF):,}"
)

AIR-LLM — NOTEBOOK 07.12
CROSS-CONDITION STABILITY
Stability profiles: 77


In [28]:
# ============================================================
# NOTEBOOK 07.13 — CANDIDATE RANKING
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.13")
print("EMPIRICAL CANDIDATE RANKING")
print("=" * 100)


# ------------------------------------------------------------
# Validate ranking input
# ------------------------------------------------------------

REQUIRED_RANKING_COLUMNS = [
    "dataset_id",
    "feature",
    "strategy",
    "robust_utility"
]

missing_columns = [
    c for c in REQUIRED_RANKING_COLUMNS
    if c not in AGGREGATION_DF.columns
]

if missing_columns:
    raise RuntimeError(
        "Candidate ranking cannot proceed.\n"
        "Missing columns:\n"
        + "\n".join(missing_columns)
    )


# ------------------------------------------------------------
# Remove stale ranking columns from previous executions
# ------------------------------------------------------------

for column in [
    "utility_rank",
    "utility_percentile"
]:
    if column in AGGREGATION_DF.columns:
        AGGREGATION_DF.drop(
            columns=[column],
            inplace=True
        )


# ------------------------------------------------------------
# Rank candidates within each dataset-feature context
# ------------------------------------------------------------

RANK_GROUPS = [
    "dataset_id",
    "feature"
]


AGGREGATION_DF[
    "utility_rank"
] = (
    AGGREGATION_DF
    .groupby(
        RANK_GROUPS,
        dropna=False
    )[
        "robust_utility"
    ]
    .rank(
        ascending=False,
        method="min"
    )
)


# ------------------------------------------------------------
# Candidate count per context
# ------------------------------------------------------------

AGGREGATION_DF[
    "candidate_count"
] = (
    AGGREGATION_DF
    .groupby(
        RANK_GROUPS,
        dropna=False
    )[
        "strategy"
    ]
    .transform("nunique")
)


# ------------------------------------------------------------
# Normalized candidate percentile
# ------------------------------------------------------------

AGGREGATION_DF[
    "utility_percentile"
] = np.where(

    AGGREGATION_DF["candidate_count"] > 1,

    1.0
    -
    (
        AGGREGATION_DF["utility_rank"] - 1
    )
    /
    (
        AGGREGATION_DF["candidate_count"] - 1
    ),

    1.0
)


AGGREGATION_DF[
    "utility_percentile"
] = (
    AGGREGATION_DF[
        "utility_percentile"
    ]
    .clip(0.0, 1.0)
)


# ------------------------------------------------------------
# Identify whether genuine candidate competition exists
# ------------------------------------------------------------

CONTEXT_COMPETITION_DF = (
    AGGREGATION_DF[
        RANK_GROUPS + ["candidate_count"]
    ]
    .drop_duplicates()
)


TOTAL_CONTEXTS = len(
    CONTEXT_COMPETITION_DF
)

COMPETING_CONTEXTS = int(
    (
        CONTEXT_COMPETITION_DF[
            "candidate_count"
        ] > 1
    ).sum()
)

SINGLE_CANDIDATE_CONTEXTS = (
    TOTAL_CONTEXTS
    - COMPETING_CONTEXTS
)


# ------------------------------------------------------------
# Ranking diagnostics
# ------------------------------------------------------------

print("\nRANKING DIAGNOSTICS")
print("-" * 100)

print(
    f"Candidate-feature combinations : "
    f"{len(AGGREGATION_DF):,}"
)

print(
    f"Experimental contexts          : "
    f"{TOTAL_CONTEXTS:,}"
)

print(
    f"Contexts with >1 candidate     : "
    f"{COMPETING_CONTEXTS:,}"
)

print(
    f"Contexts with 1 candidate      : "
    f"{SINGLE_CANDIDATE_CONTEXTS:,}"
)

print(
    f"Unique strategies               : "
    f"{AGGREGATION_DF['strategy'].nunique():,}"
)


# ------------------------------------------------------------
# Strategy ranking distribution
# ------------------------------------------------------------

print("\nSTRATEGY RANK DISTRIBUTION")
print("-" * 100)

RANK_SUMMARY_DF = (
    AGGREGATION_DF[
        [
            "strategy",
            "utility_rank"
        ]
    ]
    .groupby(
        "strategy",
        dropna=False
    )
    .agg(
        contexts=("utility_rank", "count"),
        mean_rank=("utility_rank", "mean"),
        best_rank=("utility_rank", "min"),
        worst_rank=("utility_rank", "max")
    )
    .reset_index()
    .sort_values(
        [
            "mean_rank",
            "best_rank"
        ]
    )
)

display(
    RANK_SUMMARY_DF
)


# ------------------------------------------------------------
# Ranked candidates
# ------------------------------------------------------------

print("\nRANKED CANDIDATES")
print("-" * 100)

display(
    AGGREGATION_DF.sort_values(
        [
            "dataset_id",
            "feature",
            "utility_rank",
            "strategy"
        ]
    ).head(20)
)


print("\nCandidate ranking completed.")

AIR-LLM — NOTEBOOK 07.13
EMPIRICAL CANDIDATE RANKING

RANKING DIAGNOSTICS
----------------------------------------------------------------------------------------------------
Candidate-feature combinations : 77
Experimental contexts          : 77
Contexts with >1 candidate     : 0
Contexts with 1 candidate      : 77
Unique strategies               : 2

STRATEGY RANK DISTRIBUTION
----------------------------------------------------------------------------------------------------


,strategy,contexts,mean_rank,best_rank,worst_rank
0,mean,24,1.0,1.0,1.0
1,mode,53,1.0,1.0,1.0



RANKED CANDIDATES
----------------------------------------------------------------------------------------------------


,dataset_id,feature,strategy,mean_utility,median_utility,utility_std,min_utility,max_utility,evidence_count,mechanism_count,...,repetition_count,utility_coverage,robust_utility,utility_stability,robust_rank,stability,condition_count,utility_rank,candidate_count,utility_percentile
0,adult_income,age,mean,0.722343,0.790036,0.234720,0.000000,0.983488,75,3,...,5,1.0,0.598182,0.765280,1.0,0.755990,75,1.0,1,1.0
1,adult_income,capital_gain,mean,0.740090,0.784632,0.210009,0.012780,0.992284,75,3,...,5,1.0,0.607990,0.789991,1.0,0.780114,75,1.0,1,1.0
2,adult_income,capital_loss,mean,0.762644,0.819713,0.228779,0.000000,0.994707,75,3,...,5,1.0,0.627236,0.771221,1.0,0.770431,75,1.0,1,1.0
3,adult_income,education,mode,0.620751,0.720256,0.229169,0.091886,0.891273,75,3,...,5,1.0,0.544829,0.770831,1.0,0.731684,75,1.0,1,1.0
4,adult_income,education_num,mean,0.659110,0.703001,0.256796,0.049761,0.991037,75,3,...,5,1.0,0.550408,0.743204,1.0,0.720978,75,1.0,1,1.0
5,adult_income,fnlwgt,mean,0.672078,0.695207,0.228531,0.099540,0.987668,75,3,...,5,1.0,0.564509,0.771469,1.0,0.747517,75,1.0,1,1.0
6,adult_income,hours_per_week,mean,0.699846,0.736271,0.234175,0.085893,0.995343,75,3,...,5,1.0,0.587983,0.765825,1.0,0.750542,75,1.0,1,1.0
7,adult_income,marital_status,mode,0.586039,0.652476,0.262264,0.000000,0.978921,75,3,...,5,1.0,0.488762,0.737736,1.0,0.692268,75,1.0,1,1.0
8,adult_income,native_country,mode,0.603695,0.589465,0.224806,0.265097,1.000000,75,3,...,5,1.0,0.531706,0.775194,1.0,0.729984,75,1.0,1,1.0
9,adult_income,occupation,mode,0.465131,0.357742,0.260096,0.000000,0.997336,75,3,...,5,1.0,0.339888,0.739904,1.0,0.642902,75,1.0,1,1.0



Candidate ranking completed.


In [31]:
# ============================================================
# NOTEBOOK 07.14 — SELECTION MARGIN
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.14")
print("BEST-VS-SECOND-BEST MARGIN")
print("=" * 100)


# ------------------------------------------------------------
# Validate required columns
# ------------------------------------------------------------

REQUIRED_MARGIN_COLUMNS = [
    "dataset_id",
    "feature",
    "strategy",
    "robust_utility"
]

missing_columns = [
    col
    for col in REQUIRED_MARGIN_COLUMNS
    if col not in AGGREGATION_DF.columns
]

if missing_columns:
    raise RuntimeError(
        "Selection-margin calculation cannot proceed.\n"
        "Missing columns:\n"
        + "\n".join(missing_columns)
    )


# ------------------------------------------------------------
# Remove invalid utility rows
# ------------------------------------------------------------

MARGIN_SOURCE_DF = AGGREGATION_DF[
    [
        "dataset_id",
        "feature",
        "strategy",
        "robust_utility"
    ]
].copy()

MARGIN_SOURCE_DF["robust_utility"] = pd.to_numeric(
    MARGIN_SOURCE_DF["robust_utility"],
    errors="coerce"
)

MARGIN_SOURCE_DF = MARGIN_SOURCE_DF.dropna(
    subset=["robust_utility"]
)


# ------------------------------------------------------------
# Detect duplicate candidate entries
# ------------------------------------------------------------

DUPLICATE_CANDIDATES = (
    MARGIN_SOURCE_DF
    .groupby(
        [
            "dataset_id",
            "feature",
            "strategy"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="count")
)

DUPLICATE_CANDIDATES = (
    DUPLICATE_CANDIDATES[
        DUPLICATE_CANDIDATES["count"] > 1
    ]
)

if not DUPLICATE_CANDIDATES.empty:

    raise RuntimeError(
        "Duplicate dataset-feature-strategy combinations "
        "detected before margin calculation.\n"
        "Resolve duplicate candidate rows first."
    )


# ------------------------------------------------------------
# Calculate best and second-best candidates
# ------------------------------------------------------------

MARGIN_ROWS = []

for (
    dataset_id,
    feature
), group in MARGIN_SOURCE_DF.groupby(
    [
        "dataset_id",
        "feature"
    ],
    dropna=False
):

    ranked = (
        group
        .sort_values(
            [
                "robust_utility",
                "strategy"
            ],
            ascending=[
                False,
                True
            ]
        )
        .reset_index(drop=True)
    )


    candidate_count = len(ranked)


    # --------------------------------------------------------
    # Best candidate
    # --------------------------------------------------------

    if candidate_count >= 1:

        best_strategy = str(
            ranked.loc[
                0,
                "strategy"
            ]
        )

        best_utility = float(
            ranked.loc[
                0,
                "robust_utility"
            ]
        )

    else:

        best_strategy = None
        best_utility = np.nan


    # --------------------------------------------------------
    # Second-best candidate
    # --------------------------------------------------------

    if candidate_count >= 2:

        second_best_strategy = str(
            ranked.loc[
                1,
                "strategy"
            ]
        )

        second_best_utility = float(
            ranked.loc[
                1,
                "robust_utility"
            ]
        )

        selection_margin = (
            best_utility
            - second_best_utility
        )

        margin_available = True

    else:

        second_best_strategy = None
        second_best_utility = np.nan
        selection_margin = np.nan
        margin_available = False


    # --------------------------------------------------------
    # Store result
    # --------------------------------------------------------

    MARGIN_ROWS.append({

        "dataset_id":
            dataset_id,

        "feature":
            feature,

        "best_strategy":
            best_strategy,

        "best_utility":
            best_utility,

        "second_best_strategy":
            second_best_strategy,

        "second_best_utility":
            second_best_utility,

        "selection_margin":
            selection_margin,

        "candidate_count":
            candidate_count,

        "margin_available":
            margin_available
    })


MARGIN_DF = pd.DataFrame(
    MARGIN_ROWS
)


# ------------------------------------------------------------
# Merge margin information
# ------------------------------------------------------------

# Remove previous margin columns if this cell is re-run.
OLD_MARGIN_COLUMNS = [
    "best_strategy",
    "best_utility",
    "second_best_strategy",
    "second_best_utility",
    "selection_margin",
    "candidate_count",
    "margin_available"
]

AGGREGATION_DF = AGGREGATION_DF.drop(
    columns=[
        c
        for c in OLD_MARGIN_COLUMNS
        if c in AGGREGATION_DF.columns
    ],
    errors="ignore"
)


AGGREGATION_DF = AGGREGATION_DF.merge(
    MARGIN_DF,
    on=[
        "dataset_id",
        "feature"
    ],
    how="left"
)


# ------------------------------------------------------------
# Summary diagnostics
# ------------------------------------------------------------

TOTAL_FEATURE_DECISIONS = len(
    MARGIN_DF
)

DECISIONS_WITH_MARGIN = int(
    MARGIN_DF[
        "margin_available"
    ].sum()
)

SINGLE_CANDIDATE_DECISIONS = int(
    (
        MARGIN_DF[
            "candidate_count"
        ] == 1
    ).sum()
)

MULTI_CANDIDATE_DECISIONS = int(
    (
        MARGIN_DF[
            "candidate_count"
        ] >= 2
    ).sum()
)


print("\nSELECTION MARGIN SUMMARY")
print("-" * 100)

print(
    f"Feature decisions              : "
    f"{TOTAL_FEATURE_DECISIONS:,}"
)

print(
    f"Decisions with >=2 candidates  : "
    f"{MULTI_CANDIDATE_DECISIONS:,}"
)

print(
    f"Decisions with valid margin    : "
    f"{DECISIONS_WITH_MARGIN:,}"
)

print(
    f"Single-candidate decisions     : "
    f"{SINGLE_CANDIDATE_DECISIONS:,}"
)


# ------------------------------------------------------------
# Candidate-count distribution
# ------------------------------------------------------------

print("\nCANDIDATE COUNT DISTRIBUTION")
print("-" * 100)

display(
    MARGIN_DF[
        "candidate_count"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("candidate_count")
    .reset_index(name="feature_decisions")
)


# ------------------------------------------------------------
# Sample results
# ------------------------------------------------------------

print("\nSAMPLE MARGIN RESULTS")
print("-" * 100)

display(
    MARGIN_DF[
        [
            "dataset_id",
            "feature",
            "best_strategy",
            "best_utility",
            "second_best_strategy",
            "second_best_utility",
            "selection_margin",
            "candidate_count",
            "margin_available"
        ]
    ]
    .head(20)
)


# ------------------------------------------------------------
# Research-integrity warning
# ------------------------------------------------------------

if MULTI_CANDIDATE_DECISIONS == 0:

    print("\n" + "!" * 100)
    print("RESEARCH INTEGRITY WARNING")
    print("!" * 100)

    print(
        "No dataset-feature combination contains two or more "
        "empirically evaluated candidate strategies."
    )

    print(
        "\nTherefore, a best-vs-second-best selection margin "
        "cannot be computed from the current empirical evidence."
    )

    print(
        "\nThe current result represents a single-candidate "
        "empirical evaluation, not comparative adaptive selection."
    )

    print(
        "\nDo NOT assign a numerical margin or confidence value "
        "to these single-candidate decisions."
    )


print("\n" + "=" * 100)
print("NOTEBOOK 07.14 SELECTION-MARGIN CALCULATION : COMPLETE")
print("=" * 100)

AIR-LLM — NOTEBOOK 07.14
BEST-VS-SECOND-BEST MARGIN

SELECTION MARGIN SUMMARY
----------------------------------------------------------------------------------------------------
Feature decisions              : 77
Decisions with >=2 candidates  : 0
Decisions with valid margin    : 0
Single-candidate decisions     : 77

CANDIDATE COUNT DISTRIBUTION
----------------------------------------------------------------------------------------------------


,candidate_count,feature_decisions
0,1,77



SAMPLE MARGIN RESULTS
----------------------------------------------------------------------------------------------------


,dataset_id,feature,best_strategy,best_utility,second_best_strategy,second_best_utility,selection_margin,candidate_count,margin_available
0,adult_income,age,mean,0.598182,None,NaN,NaN,1,False
1,adult_income,capital_gain,mean,0.607990,None,NaN,NaN,1,False
2,adult_income,capital_loss,mean,0.627236,None,NaN,NaN,1,False
3,adult_income,education,mode,0.544829,None,NaN,NaN,1,False
4,adult_income,education_num,mean,0.550408,None,NaN,NaN,1,False
5,adult_income,fnlwgt,mean,0.564509,None,NaN,NaN,1,False
6,adult_income,hours_per_week,mean,0.587983,None,NaN,NaN,1,False
7,adult_income,marital_status,mode,0.488762,None,NaN,NaN,1,False
8,adult_income,native_country,mode,0.531706,None,NaN,NaN,1,False
9,adult_income,occupation,mode,0.339888,None,NaN,NaN,1,False



!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
RESEARCH INTEGRITY WARNING
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
No dataset-feature combination contains two or more empirically evaluated candidate strategies.

Therefore, a best-vs-second-best selection margin cannot be computed from the current empirical evidence.

The current result represents a single-candidate empirical evaluation, not comparative adaptive selection.

Do NOT assign a numerical margin or confidence value to these single-candidate decisions.

NOTEBOOK 07.14 SELECTION-MARGIN CALCULATION : COMPLETE


In [39]:
# ============================================================
# NOTEBOOK 07.15 — EMPIRICAL CONFIDENCE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.15")
print("EMPIRICAL CONFIDENCE ESTIMATION")
print("=" * 100)


# ------------------------------------------------------------
# 1. Confidence from selection margin
# ------------------------------------------------------------

def confidence_from_margin(margin):

    if pd.isna(margin):
        return 0.0

    return float(
        np.clip(
            margin / 0.25,
            0.0,
            1.0
        )
    )


# ------------------------------------------------------------
# 2. Combine empirical confidence components
# ------------------------------------------------------------

def combine_confidence(
    margin_confidence,
    stability,
    evidence_count,
    candidate_count
):

    margin_confidence = float(
        np.clip(
            0.0 if pd.isna(margin_confidence)
            else margin_confidence,
            0.0,
            1.0
        )
    )

    stability = float(
        np.clip(
            0.0 if pd.isna(stability)
            else stability,
            0.0,
            1.0
        )
    )

    evidence_factor = float(
        np.clip(
            evidence_count / 5.0,
            0.0,
            1.0
        )
    )

    # --------------------------------------------------------
    # Multiple candidates
    # --------------------------------------------------------

    if candidate_count >= 2:

        confidence = (
            0.45 * margin_confidence
            +
            0.40 * stability
            +
            0.15 * evidence_factor
        )

    # --------------------------------------------------------
    # Single candidate
    #
    # No second candidate exists, therefore selection-margin
    # confidence is unavailable and is NOT treated as zero
    # evidence of poor performance.
    # --------------------------------------------------------

    elif candidate_count == 1:

        confidence = (
            0.70 * stability
            +
            0.30 * evidence_factor
        )

    else:

        confidence = 0.0


    return float(
        np.clip(
            confidence,
            0.0,
            1.0
        )
    )


# ------------------------------------------------------------
# 3. Build empirical selection table
# ------------------------------------------------------------

BEST_ROWS = []


for keys, group in AGGREGATION_DF.groupby(
    [
        "dataset_id",
        "feature"
    ],
    dropna=False
):

    dataset_id, feature = keys


    # --------------------------------------------------------
    # Rank candidates
    # --------------------------------------------------------

    ranked = (
        group
        .sort_values(
            [
                "robust_utility",
                "stability"
            ],
            ascending=[
                False,
                False
            ]
        )
        .reset_index(drop=True)
    )


    if ranked.empty:
        continue


    # --------------------------------------------------------
    # Best candidate
    # --------------------------------------------------------

    best = ranked.iloc[0]


    # --------------------------------------------------------
    # Candidate count
    # --------------------------------------------------------

    candidate_count = int(
        len(ranked)
    )


    # --------------------------------------------------------
    # Best-vs-second-best margin
    # --------------------------------------------------------

    if candidate_count >= 2:

        second_best = ranked.iloc[1]

        selection_margin = float(
            best["robust_utility"]
            -
            second_best["robust_utility"]
        )

        margin_available = True

    else:

        selection_margin = np.nan

        margin_available = False


    # --------------------------------------------------------
    # Margin confidence
    # --------------------------------------------------------

    margin_confidence = (
        confidence_from_margin(
            selection_margin
        )
        if margin_available
        else 0.0
    )


    # --------------------------------------------------------
    # Stability
    # --------------------------------------------------------

    stability_value = best.get(
        "stability",
        np.nan
    )

    if pd.isna(stability_value):

        stability_value = 0.0

    stability_value = float(
        np.clip(
            stability_value,
            0.0,
            1.0
        )
    )


    # --------------------------------------------------------
    # Evidence count
    # --------------------------------------------------------

    evidence_value = best.get(
        "evidence_count",
        0
    )

    if pd.isna(evidence_value):

        evidence_value = 0

    evidence_count = int(
        evidence_value
    )


    # --------------------------------------------------------
    # Empirical confidence
    # --------------------------------------------------------

    empirical_confidence = combine_confidence(

        margin_confidence=
            margin_confidence,

        stability=
            stability_value,

        evidence_count=
            evidence_count,

        candidate_count=
            candidate_count
    )


    # --------------------------------------------------------
    # Selection status
    # --------------------------------------------------------

    if candidate_count >= 2:

        selection_status = (
            "competitive_selection"
        )

    elif candidate_count == 1:

        selection_status = (
            "single_candidate"
        )

    else:

        selection_status = (
            "no_candidate"
        )


    # --------------------------------------------------------
    # Store result
    # --------------------------------------------------------

    BEST_ROWS.append({

        "dataset_id":
            dataset_id,

        "feature":
            feature,

        "selected_strategy":
            str(
                best["strategy"]
            ),

        "selected_utility":
            float(
                best["robust_utility"]
            ),

        "selection_margin":
            selection_margin,

        "margin_available":
            margin_available,

        "candidate_count":
            candidate_count,

        "stability":
            stability_value,

        "evidence_count":
            evidence_count,

        "selection_status":
            selection_status,

        "empirical_confidence":
            empirical_confidence
    })


# ------------------------------------------------------------
# 4. Create final empirical selection dataframe
# ------------------------------------------------------------

EMPIRICAL_SELECTION_DF = pd.DataFrame(
    BEST_ROWS
)


# ------------------------------------------------------------
# 5. Validate output schema
# ------------------------------------------------------------

REQUIRED_SELECTION_COLUMNS = [

    "dataset_id",
    "feature",
    "selected_strategy",
    "selected_utility",
    "selection_margin",
    "margin_available",
    "candidate_count",
    "stability",
    "evidence_count",
    "selection_status",
    "empirical_confidence"
]


missing_selection_columns = [
    column
    for column in REQUIRED_SELECTION_COLUMNS
    if column not in EMPIRICAL_SELECTION_DF.columns
]


if missing_selection_columns:

    raise RuntimeError(
        "Empirical selection table is incomplete.\n"
        "Missing columns:\n"
        +
        "\n".join(
            missing_selection_columns
        )
    )


# ------------------------------------------------------------
# 6. Validate candidate counts
# ------------------------------------------------------------

if len(EMPIRICAL_SELECTION_DF) > 0:

    invalid_candidate_counts = (
        EMPIRICAL_SELECTION_DF[
            "candidate_count"
        ]
        < 1
    ).sum()

    if invalid_candidate_counts > 0:

        raise RuntimeError(
            "Invalid empirical selection state detected: "
            "one or more feature decisions have zero candidates."
        )


# ------------------------------------------------------------
# 7. Summary
# ------------------------------------------------------------

competitive_count = int(
    (
        EMPIRICAL_SELECTION_DF[
            "candidate_count"
        ]
        >= 2
    ).sum()
)


single_candidate_count = int(
    (
        EMPIRICAL_SELECTION_DF[
            "candidate_count"
        ]
        == 1
    ).sum()
)


print(
    f"Empirical selections        : "
    f"{len(EMPIRICAL_SELECTION_DF):,}"
)

print(
    f"Decisions with >=2 candidates: "
    f"{competitive_count:,}"
)

print(
    f"Single-candidate decisions   : "
    f"{single_candidate_count:,}"
)


print(
    f"Mean empirical confidence   : "
    f"{EMPIRICAL_SELECTION_DF[
        'empirical_confidence'
    ].mean():.4f}"
)


# ------------------------------------------------------------
# 8. Sample output
# ------------------------------------------------------------

print("\nEMPIRICAL SELECTION SAMPLE")
print("-" * 100)


display(
    EMPIRICAL_SELECTION_DF[
        [
            "dataset_id",
            "feature",
            "selected_strategy",
            "selected_utility",
            "selection_margin",
            "margin_available",
            "candidate_count",
            "stability",
            "evidence_count",
            "selection_status",
            "empirical_confidence"
        ]
    ]
    .head(20)
)


print("\n" + "=" * 100)
print("NOTEBOOK 07.15 EMPIRICAL CONFIDENCE : COMPLETE")
print("=" * 100)

AIR-LLM — NOTEBOOK 07.15
EMPIRICAL CONFIDENCE ESTIMATION
Empirical selections        : 77
Decisions with >=2 candidates: 0
Single-candidate decisions   : 77
Mean empirical confidence   : 0.8107

EMPIRICAL SELECTION SAMPLE
----------------------------------------------------------------------------------------------------


,dataset_id,feature,selected_strategy,selected_utility,selection_margin,margin_available,candidate_count,stability,evidence_count,selection_status,empirical_confidence
0,adult_income,age,mean,0.598182,NaN,False,1,0.755990,75,single_candidate,0.829193
1,adult_income,capital_gain,mean,0.607990,NaN,False,1,0.780114,75,single_candidate,0.846080
2,adult_income,capital_loss,mean,0.627236,NaN,False,1,0.770431,75,single_candidate,0.839302
3,adult_income,education,mode,0.544829,NaN,False,1,0.731684,75,single_candidate,0.812178
4,adult_income,education_num,mean,0.550408,NaN,False,1,0.720978,75,single_candidate,0.804685
5,adult_income,fnlwgt,mean,0.564509,NaN,False,1,0.747517,75,single_candidate,0.823262
6,adult_income,hours_per_week,mean,0.587983,NaN,False,1,0.750542,75,single_candidate,0.825379
7,adult_income,marital_status,mode,0.488762,NaN,False,1,0.692268,75,single_candidate,0.784588
8,adult_income,native_country,mode,0.531706,NaN,False,1,0.729984,75,single_candidate,0.810989
9,adult_income,occupation,mode,0.339888,NaN,False,1,0.642902,75,single_candidate,0.750031



NOTEBOOK 07.15 EMPIRICAL CONFIDENCE : COMPLETE


In [42]:
# ============================================================
# DIAGNOSTIC — LLM RECOMMENDATION SCHEMA
# ============================================================

print("=" * 100)
print("AIR-LLM — LLM RECOMMENDATION SCHEMA DIAGNOSTIC")
print("=" * 100)

print("\nLLM_RECOMMENDATION_DF SHAPE")
print("-" * 100)

print(
    f"Rows    : {len(LLM_RECOMMENDATION_DF):,}"
)

print(
    f"Columns : {len(LLM_RECOMMENDATION_DF.columns):,}"
)


print("\nLLM_RECOMMENDATION_DF COLUMNS")
print("-" * 100)

for i, column in enumerate(
    LLM_RECOMMENDATION_DF.columns,
    start=1
):

    print(
        f"{i:3d}. {column}"
    )


print("\nLLM_RECOMMENDATION_DF SAMPLE")
print("-" * 100)

display(
    LLM_RECOMMENDATION_DF.head(20)
)

AIR-LLM — LLM RECOMMENDATION SCHEMA DIAGNOSTIC

LLM_RECOMMENDATION_DF SHAPE
----------------------------------------------------------------------------------------------------
Rows    : 55
Columns : 23

LLM_RECOMMENDATION_DF COLUMNS
----------------------------------------------------------------------------------------------------
  1. dataset_id
  2. target
  3. feature
  4. feature_type
  5. missing_rate
  6. dataset_rows
  7. dataset_columns
  8. feature_cardinality
  9. target_relevance
 10. rank
 11. llm_recommended
 12. family
 13. subfamily
 14. prior_score
 15. rationale
 16. strengths
 17. limitations
 18. applicability
 19. confidence
 20. estimated_cost
 21. recommendation_source
 22. recommendation_version
 23. random_state

LLM_RECOMMENDATION_DF SAMPLE
----------------------------------------------------------------------------------------------------


,dataset_id,target,feature,feature_type,missing_rate,dataset_rows,dataset_columns,feature_cardinality,target_relevance,rank,...,prior_score,rationale,strengths,limitations,applicability,confidence,estimated_cost,recommendation_source,recommendation_version,random_state
0,adult_income,income,native_country,categorical,0.017857,32561.0,15.0,41.0,0.095968,1,...,5.00,Categorical feature with missingness rate 0.01...,Gradient-boosted predictive imputation using n...,Actual effectiveness must be determined throug...,high,0.90,medium,LLM_guided_structured_recommendation,AIR-LLM-1.1,42
1,adult_income,income,native_country,categorical,0.017857,32561.0,15.0,41.0,0.095968,2,...,5.00,Categorical feature with missingness rate 0.01...,Random-forest predictive imputation using nonl...,Actual effectiveness must be determined throug...,high,0.82,medium,LLM_guided_structured_recommendation,AIR-LLM-1.1,42
2,adult_income,income,native_country,categorical,0.017857,32561.0,15.0,41.0,0.095968,3,...,4.75,Categorical feature with missingness rate 0.01...,Stochastic marginal imputation by sampling fro...,Actual effectiveness must be determined throug...,medium,0.74,low,LLM_guided_structured_recommendation,AIR-LLM-1.1,42
3,adult_income,income,native_country,categorical,0.017857,32561.0,15.0,41.0,0.095968,4,...,4.45,Categorical feature with missingness rate 0.01...,Iterative random-forest imputation for mixed-t...,Actual effectiveness must be determined throug...,medium,0.66,medium,LLM_guided_structured_recommendation,AIR-LLM-1.1,42
4,adult_income,income,native_country,categorical,0.017857,32561.0,15.0,41.0,0.095968,5,...,4.00,Categorical feature with missingness rate 0.01...,Mode imputation using the most frequent observ...,Actual effectiveness must be determined throug...,low,0.58,low,LLM_guided_structured_recommendation,AIR-LLM-1.1,42
5,adult_income,income,occupation,categorical,0.056204,32561.0,15.0,14.0,0.349804,1,...,4.50,Categorical feature with missingness rate 0.05...,Gradient-boosted predictive imputation using n...,Actual effectiveness must be determined throug...,high,0.90,medium,LLM_guided_structured_recommendation,AIR-LLM-1.1,42
6,adult_income,income,occupation,categorical,0.056204,32561.0,15.0,14.0,0.349804,2,...,4.50,Categorical feature with missingness rate 0.05...,Random-forest predictive imputation using nonl...,Actual effectiveness must be determined throug...,high,0.82,medium,LLM_guided_structured_recommendation,AIR-LLM-1.1,42
7,adult_income,income,occupation,categorical,0.056204,32561.0,15.0,14.0,0.349804,3,...,4.25,Categorical feature with missingness rate 0.05...,Stochastic marginal imputation by sampling fro...,Actual effectiveness must be determined throug...,medium,0.74,low,LLM_guided_structured_recommendation,AIR-LLM-1.1,42
8,adult_income,income,occupation,categorical,0.056204,32561.0,15.0,14.0,0.349804,4,...,4.00,Categorical feature with missingness rate 0.05...,Mode imputation using the most frequent observ...,Actual effectiveness must be determined throug...,medium,0.66,low,LLM_guided_structured_recommendation,AIR-LLM-1.1,42
9,adult_income,income,occupation,categorical,0.056204,32561.0,15.0,14.0,0.349804,5,...,3.95,Categorical feature with missingness rate 0.05...,Multiple imputation by chained equations using...,Actual effectiveness must be determined throug...,low,0.58,medium,LLM_guided_structured_recommendation,AIR-LLM-1.1,42


In [44]:
# ============================================================
# NOTEBOOK 07.16 — LLM RECOMMENDATION ALIGNMENT
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.16")
print("LLM RECOMMENDATION ALIGNMENT")
print("=" * 100)


# ============================================================
# 1. VALIDATE EMPIRICAL SELECTION TABLE
# ============================================================

REQUIRED_SELECTION_COLUMNS = [
    "dataset_id",
    "feature",
    "selected_strategy",
    "candidate_count",
    "selection_margin",
    "stability",
    "evidence_count"
]

missing_columns = [
    col
    for col in REQUIRED_SELECTION_COLUMNS
    if col not in EMPIRICAL_SELECTION_DF.columns
]

if missing_columns:
    raise RuntimeError(
        "LLM alignment cannot proceed.\n"
        "Missing empirical-selection columns:\n"
        + "\n".join(missing_columns)
    )


# ============================================================
# 2. INITIALIZE ALIGNMENT TABLE
# ============================================================

LLM_ALIGNMENT_DF = (
    EMPIRICAL_SELECTION_DF.copy()
)

LLM_ALIGNMENT_DF[
    "llm_recommended_strategy"
] = pd.NA

LLM_ALIGNMENT_DF[
    "llm_recommendation_rank"
] = pd.NA

LLM_ALIGNMENT_DF[
    "llm_alignment"
] = np.nan

LLM_ALIGNMENT_DF[
    "alignment_status"
] = "NO_LLM_RECOMMENDATION"


# ============================================================
# 3. VALIDATE LLM RECOMMENDATION DATA
# ============================================================

if LLM_RECOMMENDATION_DF is None:

    raise RuntimeError(
        "LLM_RECOMMENDATION_DF is not defined."
    )


if LLM_RECOMMENDATION_DF.empty:

    print("\nWARNING: LLM_RECOMMENDATION_DF is empty.")

else:

    print(
        f"\nLLM recommendation rows : "
        f"{len(LLM_RECOMMENDATION_DF):,}"
    )


# ============================================================
# 4. DETECT ACTUAL LLM SCHEMA
# ============================================================

if not LLM_RECOMMENDATION_DF.empty:

    # --------------------------------------------------------
    # Dataset
    # --------------------------------------------------------

    llm_dataset_col = find_profile_column(
        LLM_RECOMMENDATION_DF,
        [
            "dataset_id",
            "dataset"
        ]
    )

    # --------------------------------------------------------
    # Feature
    # --------------------------------------------------------

    llm_feature_col = find_profile_column(
        LLM_RECOMMENDATION_DF,
        [
            "feature",
            "feature_name",
            "column"
        ]
    )

    # --------------------------------------------------------
    # IMPORTANT:
    # Actual AIR-LLM column is "llm_recommended"
    # --------------------------------------------------------

    llm_strategy_col = find_profile_column(
        LLM_RECOMMENDATION_DF,
        [
            "llm_recommended"
        ]
    )

    # --------------------------------------------------------
    # Actual ranking column
    # --------------------------------------------------------

    llm_rank_col = find_profile_column(
        LLM_RECOMMENDATION_DF,
        [
            "rank"
        ]
    )


    print("\nDETECTED LLM SCHEMA")
    print("-" * 100)

    print(
        f"Dataset column      : "
        f"{llm_dataset_col}"
    )

    print(
        f"Feature column      : "
        f"{llm_feature_col}"
    )

    print(
        f"Recommendation      : "
        f"{llm_strategy_col}"
    )

    print(
        f"Rank column         : "
        f"{llm_rank_col}"
    )


    # ========================================================
    # 5. SCHEMA VALIDATION
    # ========================================================

    if (
        llm_dataset_col is None
        or llm_feature_col is None
        or llm_strategy_col is None
    ):

        raise RuntimeError(
            "\nLLM recommendation schema is incomplete.\n\n"
            "Required columns:\n"
            "  dataset_id\n"
            "  feature\n"
            "  llm_recommended\n"
        )


    # ========================================================
    # 6. EXTRACT LLM RECOMMENDATIONS
    # ========================================================

    extraction_columns = [
        llm_dataset_col,
        llm_feature_col,
        llm_strategy_col
    ]

    if llm_rank_col is not None:

        extraction_columns.append(
            llm_rank_col
        )


    llm_small = (
        LLM_RECOMMENDATION_DF[
            extraction_columns
        ]
        .copy()
    )


    rename_map = {

        llm_dataset_col:
            "dataset_id",

        llm_feature_col:
            "feature",

        llm_strategy_col:
            "llm_recommended_strategy"
    }


    if llm_rank_col is not None:

        rename_map[
            llm_rank_col
        ] = "llm_recommendation_rank"


    llm_small = (
        llm_small
        .rename(
            columns=rename_map
        )
    )


    # ========================================================
    # 7. NORMALIZE DATASET / FEATURE IDENTIFIERS
    # ========================================================

    llm_small[
        "dataset_id"
    ] = (
        llm_small[
            "dataset_id"
        ]
        .astype(str)
        .str.strip()
    )


    llm_small[
        "feature"
    ] = (
        llm_small[
            "feature"
        ]
        .astype(str)
        .str.strip()
    )


    llm_small[
        "llm_recommended_strategy"
    ] = (
        llm_small[
            "llm_recommended_strategy"
        ]
        .astype(str)
        .str.strip()
    )


    # ========================================================
    # 8. NORMALIZE RANK
    # ========================================================

    if (
        "llm_recommendation_rank"
        in llm_small.columns
    ):

        llm_small[
            "llm_recommendation_rank"
        ] = pd.to_numeric(
            llm_small[
                "llm_recommendation_rank"
            ],
            errors="coerce"
        )

    else:

        # Deterministic fallback
        llm_small[
            "llm_recommendation_rank"
        ] = 1


    # ========================================================
    # 9. REMOVE INVALID LLM RECOMMENDATIONS
    # ========================================================

    invalid_mask = (
        llm_small[
            "llm_recommended_strategy"
        ].isna()
        |
        (
            llm_small[
                "llm_recommended_strategy"
            ]
            .astype(str)
            .str.strip()
            .str.lower()
            .isin(
                [
                    "",
                    "nan",
                    "none",
                    "<na>"
                ]
            )
        )
    )


    llm_small = (
        llm_small[
            ~invalid_mask
        ]
        .copy()
    )


    # ========================================================
    # 10. SORT BY RANK
    # ========================================================

    llm_small = (
        llm_small
        .sort_values(
            [
                "dataset_id",
                "feature",
                "llm_recommendation_rank"
            ],
            na_position="last"
        )
    )


    # ========================================================
    # 11. SELECT LLM RANK-1 RECOMMENDATION
    # ========================================================

    llm_top = (
        llm_small
        .drop_duplicates(
            [
                "dataset_id",
                "feature"
            ],
            keep="first"
        )
        .copy()
    )


    print(
        f"\nUsable LLM recommendation rows : "
        f"{len(llm_small):,}"
    )

    print(
        f"Unique LLM feature decisions    : "
        f"{len(llm_top):,}"
    )


    # ========================================================
    # 12. NORMALIZE EMPIRICAL IDENTIFIERS
    # ========================================================

    LLM_ALIGNMENT_DF[
        "dataset_id"
    ] = (
        LLM_ALIGNMENT_DF[
            "dataset_id"
        ]
        .astype(str)
        .str.strip()
    )

    LLM_ALIGNMENT_DF[
        "feature"
    ] = (
        LLM_ALIGNMENT_DF[
            "feature"
        ]
        .astype(str)
        .str.strip()
    )


    # ========================================================
    # 13. MERGE LLM RANK-1 RECOMMENDATIONS
    # ========================================================

    LLM_ALIGNMENT_DF = (
        LLM_ALIGNMENT_DF
        .drop(
            columns=[
                "llm_recommended_strategy",
                "llm_recommendation_rank"
            ],
            errors="ignore"
        )
        .merge(
            llm_top[
                [
                    "dataset_id",
                    "feature",
                    "llm_recommended_strategy",
                    "llm_recommendation_rank"
                ]
            ],
            on=[
                "dataset_id",
                "feature"
            ],
            how="left"
        )
    )


    # ========================================================
    # 14. NORMALIZE STRATEGY NAMES
    # ========================================================

    def normalize_strategy(value):

        if pd.isna(value):
            return None

        value = str(value).strip().lower()

        # Normalize common formatting differences
        value = value.replace("-", "_")
        value = value.replace(" ", "_")

        return value


    LLM_ALIGNMENT_DF[
        "_selected_strategy_normalized"
    ] = (
        LLM_ALIGNMENT_DF[
            "selected_strategy"
        ]
        .apply(normalize_strategy)
    )


    LLM_ALIGNMENT_DF[
        "_llm_strategy_normalized"
    ] = (
        LLM_ALIGNMENT_DF[
            "llm_recommended_strategy"
        ]
        .apply(normalize_strategy)
    )


    # ========================================================
    # 15. IDENTIFY LLM-COVERED FEATURES
    # ========================================================

    has_llm = (
        LLM_ALIGNMENT_DF[
            "_llm_strategy_normalized"
        ].notna()
    )


    matches = (
        LLM_ALIGNMENT_DF[
            "_selected_strategy_normalized"
        ]
        ==
        LLM_ALIGNMENT_DF[
            "_llm_strategy_normalized"
        ]
    )


    # ========================================================
    # 16. CALCULATE ALIGNMENT
    # ========================================================

    LLM_ALIGNMENT_DF[
        "llm_alignment"
    ] = np.where(
        has_llm,
        matches.astype(float),
        np.nan
    )


    # ========================================================
    # 17. NUMERIC CANDIDATE COUNT
    # ========================================================

    candidate_count_numeric = pd.to_numeric(
        LLM_ALIGNMENT_DF[
            "candidate_count"
        ],
        errors="coerce"
    )


    # ========================================================
    # 18. CLASSIFY ALIGNMENT STATUS
    # ========================================================

    LLM_ALIGNMENT_DF[
        "alignment_status"
    ] = "NO_LLM_RECOMMENDATION"


    # --------------------------------------------------------
    # Comparative evidence
    # --------------------------------------------------------

    comparative_mask = (
        has_llm
        &
        (
            candidate_count_numeric >= 2
        )
    )


    # --------------------------------------------------------
    # Single-candidate evidence
    # --------------------------------------------------------

    single_candidate_mask = (
        has_llm
        &
        (
            candidate_count_numeric < 2
        )
    )


    # --------------------------------------------------------
    # Comparative aligned
    # --------------------------------------------------------

    LLM_ALIGNMENT_DF.loc[
        comparative_mask
        & matches,
        "alignment_status"
    ] = "ALIGNED"


    # --------------------------------------------------------
    # Comparative misaligned
    # --------------------------------------------------------

    LLM_ALIGNMENT_DF.loc[
        comparative_mask
        & ~matches,
        "alignment_status"
    ] = "MISALIGNED"


    # --------------------------------------------------------
    # Single candidate match
    # --------------------------------------------------------

    LLM_ALIGNMENT_DF.loc[
        single_candidate_mask
        & matches,
        "alignment_status"
    ] = "SINGLE_CANDIDATE_MATCH"


    # --------------------------------------------------------
    # Single candidate mismatch
    # --------------------------------------------------------

    LLM_ALIGNMENT_DF.loc[
        single_candidate_mask
        & ~matches,
        "alignment_status"
    ] = "SINGLE_CANDIDATE_MISMATCH"


# ============================================================
# 19. COVERAGE STATISTICS
# ============================================================

total_empirical = len(
    LLM_ALIGNMENT_DF
)

llm_covered = int(
    LLM_ALIGNMENT_DF[
        "llm_recommended_strategy"
    ].notna().sum()
)

llm_uncovered = (
    total_empirical
    - llm_covered
)


coverage_rate = (
    llm_covered / total_empirical
    if total_empirical > 0
    else np.nan
)


print(
    "\nLLM COVERAGE SUMMARY"
)

print("-" * 100)

print(
    f"Empirical feature decisions : "
    f"{total_empirical:,}"
)

print(
    f"LLM-covered features        : "
    f"{llm_covered:,}"
)

print(
    f"LLM-uncovered features      : "
    f"{llm_uncovered:,}"
)

print(
    f"LLM coverage rate           : "
    f"{coverage_rate:.4f}"
    if pd.notna(coverage_rate)
    else
    "LLM coverage rate           : NOT AVAILABLE"
)


# ============================================================
# 20. ALIGNMENT SUMMARY
# ============================================================

print(
    "\nLLM ALIGNMENT SUMMARY"
)

print("-" * 100)


status_counts = (
    LLM_ALIGNMENT_DF[
        "alignment_status"
    ]
    .value_counts()
    .sort_index()
)


for status, count in status_counts.items():

    print(
        f"{status:32s}: "
        f"{int(count):,}"
    )


# ============================================================
# 21. COMPARATIVE ALIGNMENT
# ============================================================

comparative_alignment = (
    LLM_ALIGNMENT_DF[
        LLM_ALIGNMENT_DF[
            "alignment_status"
        ].isin(
            [
                "ALIGNED",
                "MISALIGNED"
            ]
        )
    ]
)


comparative_count = len(
    comparative_alignment
)


if comparative_count > 0:

    comparative_alignment_rate = (
        comparative_alignment[
            "llm_alignment"
        ]
        .mean()
    )

else:

    comparative_alignment_rate = np.nan


print(
    "\nCOMPARATIVE LLM ALIGNMENT"
)

print("-" * 100)

print(
    f"Comparative cases           : "
    f"{comparative_count:,}"
)


if pd.notna(
    comparative_alignment_rate
):

    print(
        f"Alignment rate              : "
        f"{comparative_alignment_rate:.4f}"
    )

else:

    print(
        "Alignment rate              : "
        "NOT AVAILABLE"
    )

    print(
        "Reason                      : "
        "No feature has >=2 empirically "
        "evaluated candidates."
    )


# ============================================================
# 22. DESCRIPTIVE SINGLE-CANDIDATE AGREEMENT
# ============================================================

single_candidate_alignment = (
    LLM_ALIGNMENT_DF[
        LLM_ALIGNMENT_DF[
            "alignment_status"
        ].isin(
            [
                "SINGLE_CANDIDATE_MATCH",
                "SINGLE_CANDIDATE_MISMATCH"
            ]
        )
    ]
)


single_candidate_count = len(
    single_candidate_alignment
)


if single_candidate_count > 0:

    descriptive_agreement_rate = (
        single_candidate_alignment[
            "llm_alignment"
        ]
        .mean()
    )

else:

    descriptive_agreement_rate = np.nan


print(
    "\nDESCRIPTIVE SINGLE-CANDIDATE AGREEMENT"
)

print("-" * 100)

print(
    f"Single-candidate comparisons : "
    f"{single_candidate_count:,}"
)


if pd.notna(
    descriptive_agreement_rate
):

    print(
        f"Agreement rate              : "
        f"{descriptive_agreement_rate:.4f}"
    )

else:

    print(
        "Agreement rate              : "
        "NOT AVAILABLE"
    )


# ============================================================
# 23. OVERALL LLM AGREEMENT
# ============================================================

usable_alignment = (
    LLM_ALIGNMENT_DF[
        LLM_ALIGNMENT_DF[
            "alignment_status"
        ].isin(
            [
                "ALIGNED",
                "MISALIGNED",
                "SINGLE_CANDIDATE_MATCH",
                "SINGLE_CANDIDATE_MISMATCH"
            ]
        )
    ]
)


if len(usable_alignment) > 0:

    overall_agreement_rate = (
        usable_alignment[
            "llm_alignment"
        ]
        .mean()
    )

else:

    overall_agreement_rate = np.nan


print(
    "\nOVERALL LLM–EMPIRICAL AGREEMENT"
)

print("-" * 100)


if pd.notna(
    overall_agreement_rate
):

    print(
        f"Agreement rate              : "
        f"{overall_agreement_rate:.4f}"
    )

else:

    print(
        "Agreement rate              : "
        "NOT AVAILABLE"
    )


# ============================================================
# 24. SAMPLE RESULTS
# ============================================================

print(
    "\nSAMPLE LLM ALIGNMENT RESULTS"
)

print("-" * 100)


display_columns = [
    "dataset_id",
    "feature",
    "selected_strategy",
    "llm_recommended_strategy",
    "llm_recommendation_rank",
    "candidate_count",
    "selection_margin",
    "stability",
    "llm_alignment",
    "alignment_status"
]


display(
    LLM_ALIGNMENT_DF[
        display_columns
    ]
    .head(20)
)


# ============================================================
# 25. LLM-COVERED RESULTS ONLY
# ============================================================

if llm_covered > 0:

    print(
        "\nLLM-COVERED FEATURE RESULTS"
    )

    print("-" * 100)

    display(
        LLM_ALIGNMENT_DF[
            has_llm
        ][
            display_columns
        ]
        .head(20)
    )


# ============================================================
# 26. RESEARCH-INTEGRITY INTERPRETATION
# ============================================================

print(
    "\n" + "!" * 100
)

print(
    "RESEARCH INTERPRETATION"
)

print(
    "!" * 100
)


if llm_covered == 0:

    print(
        "No usable LLM recommendations were matched "
        "to the empirical-selection table."
    )

    print(
        "\nLLM recommendation alignment cannot currently "
        "be evaluated."
    )

elif comparative_count == 0:

    print(
        f"LLM recommendations were successfully matched "
        f"for {llm_covered:,} of {total_empirical:,} "
        f"empirical feature decisions."
    )

    print(
        "\nHowever, no feature currently has two or more "
        "empirically evaluated candidate strategies."
    )

    print(
        "\nTherefore, the current LLM–empirical agreement "
        "is descriptive only."
    )

    print(
        "\nIt must NOT be interpreted as comparative "
        "validation of the AIR-LLM recommendation engine."
    )

    print(
        "\nComparative validation requires multiple "
        "empirically evaluated candidate strategies "
        "for the same dataset-feature condition."
    )

else:

    print(
        f"Comparative LLM alignment was evaluated for "
        f"{comparative_count:,} feature conditions."
    )

    print(
        f"\nComparative alignment rate: "
        f"{comparative_alignment_rate:.4f}"
    )


# ============================================================
# 27. REMOVE INTERNAL NORMALIZATION COLUMNS
# ============================================================

LLM_ALIGNMENT_DF = (
    LLM_ALIGNMENT_DF.drop(
        columns=[
            "_selected_strategy_normalized",
            "_llm_strategy_normalized"
        ],
        errors="ignore"
    )
)


# ============================================================
# 28. FINAL VALIDATION
# ============================================================

required_output_columns = [
    "dataset_id",
    "feature",
    "selected_strategy",
    "llm_recommended_strategy",
    "llm_recommendation_rank",
    "candidate_count",
    "selection_margin",
    "stability",
    "evidence_count",
    "llm_alignment",
    "alignment_status"
]


missing_output_columns = [
    col
    for col in required_output_columns
    if col not in LLM_ALIGNMENT_DF.columns
]


if missing_output_columns:

    raise RuntimeError(
        "Notebook 07.16 output validation failed.\n"
        "Missing columns:\n"
        + "\n".join(
            missing_output_columns
        )
    )


print(
    "\nOutput validation : PASS"
)


# ============================================================
# 29. COMPLETION
# ============================================================

print(
    "\n" + "=" * 100
)

print(
    "NOTEBOOK 07.16 LLM ALIGNMENT : COMPLETE"
)

print(
    "=" * 100
)

AIR-LLM — NOTEBOOK 07.16
LLM RECOMMENDATION ALIGNMENT

LLM recommendation rows : 55

DETECTED LLM SCHEMA
----------------------------------------------------------------------------------------------------
Dataset column      : dataset_id
Feature column      : feature
Recommendation      : llm_recommended
Rank column         : rank

Usable LLM recommendation rows : 55
Unique LLM feature decisions    : 11

LLM COVERAGE SUMMARY
----------------------------------------------------------------------------------------------------
Empirical feature decisions : 77
LLM-covered features        : 11
LLM-uncovered features      : 66
LLM coverage rate           : 0.1429

LLM ALIGNMENT SUMMARY
----------------------------------------------------------------------------------------------------
NO_LLM_RECOMMENDATION           : 66
SINGLE_CANDIDATE_MISMATCH       : 11

COMPARATIVE LLM ALIGNMENT
----------------------------------------------------------------------------------------------------
Compara

,dataset_id,feature,selected_strategy,llm_recommended_strategy,llm_recommendation_rank,candidate_count,selection_margin,stability,llm_alignment,alignment_status
0,adult_income,age,mean,NaN,NaN,1,NaN,0.755990,NaN,NO_LLM_RECOMMENDATION
1,adult_income,capital_gain,mean,NaN,NaN,1,NaN,0.780114,NaN,NO_LLM_RECOMMENDATION
2,adult_income,capital_loss,mean,NaN,NaN,1,NaN,0.770431,NaN,NO_LLM_RECOMMENDATION
3,adult_income,education,mode,NaN,NaN,1,NaN,0.731684,NaN,NO_LLM_RECOMMENDATION
4,adult_income,education_num,mean,NaN,NaN,1,NaN,0.720978,NaN,NO_LLM_RECOMMENDATION
5,adult_income,fnlwgt,mean,NaN,NaN,1,NaN,0.747517,NaN,NO_LLM_RECOMMENDATION
6,adult_income,hours_per_week,mean,NaN,NaN,1,NaN,0.750542,NaN,NO_LLM_RECOMMENDATION
7,adult_income,marital_status,mode,NaN,NaN,1,NaN,0.692268,NaN,NO_LLM_RECOMMENDATION
8,adult_income,native_country,mode,gradient_boosting,1.0,1,NaN,0.729984,0.0,SINGLE_CANDIDATE_MISMATCH
9,adult_income,occupation,mode,gradient_boosting,1.0,1,NaN,0.642902,0.0,SINGLE_CANDIDATE_MISMATCH



LLM-COVERED FEATURE RESULTS
----------------------------------------------------------------------------------------------------


,dataset_id,feature,selected_strategy,llm_recommended_strategy,llm_recommendation_rank,candidate_count,selection_margin,stability,llm_alignment,alignment_status
8,adult_income,native_country,mode,gradient_boosting,1.0,1,NaN,0.729984,0.0,SINGLE_CANDIDATE_MISMATCH
9,adult_income,occupation,mode,gradient_boosting,1.0,1,NaN,0.642902,0.0,SINGLE_CANDIDATE_MISMATCH
13,adult_income,workclass,mode,gradient_boosting,1.0,1,NaN,0.762614,0.0,SINGLE_CANDIDATE_MISMATCH
40,diabetes_130us,diag_1,mode,gradient_boosting,1.0,1,NaN,0.656362,0.0,SINGLE_CANDIDATE_MISMATCH
41,diabetes_130us,diag_2,mode,gradient_boosting,1.0,1,NaN,0.631662,0.0,SINGLE_CANDIDATE_MISMATCH
42,diabetes_130us,diag_3,mode,gradient_boosting,1.0,1,NaN,0.624810,0.0,SINGLE_CANDIDATE_MISMATCH
53,diabetes_130us,max_glu_serum,mode,gradient_boosting,1.0,1,NaN,0.604319,0.0,SINGLE_CANDIDATE_MISMATCH
54,diabetes_130us,medical_specialty,mode,gradient_boosting,1.0,1,NaN,0.650576,0.0,SINGLE_CANDIDATE_MISMATCH
67,diabetes_130us,payer_code,mode,gradient_boosting,1.0,1,NaN,0.715964,0.0,SINGLE_CANDIDATE_MISMATCH
69,diabetes_130us,race,mode,gradient_boosting,1.0,1,NaN,0.717717,0.0,SINGLE_CANDIDATE_MISMATCH



!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
RESEARCH INTERPRETATION
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
LLM recommendations were successfully matched for 11 of 77 empirical feature decisions.

However, no feature currently has two or more empirically evaluated candidate strategies.

Therefore, the current LLM–empirical agreement is descriptive only.

It must NOT be interpreted as comparative validation of the AIR-LLM recommendation engine.

Comparative validation requires multiple empirically evaluated candidate strategies for the same dataset-feature condition.

Output validation : PASS

NOTEBOOK 07.16 LLM ALIGNMENT : COMPLETE


In [46]:
# ============================================================
# NOTEBOOK 07.17 — CONFIDENCE AND ABSTENTION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.17")
print("CONFIDENCE AND ABSTENTION")
print("=" * 100)


# ------------------------------------------------------------
# Create final selection table
# ------------------------------------------------------------

FINAL_SELECTION_DF = (
    LLM_ALIGNMENT_DF.copy()
)


# ------------------------------------------------------------
# Ensure required columns exist
# ------------------------------------------------------------

REQUIRED_COLUMNS = [
    "dataset_id",
    "feature",
    "selected_strategy",
    "selected_utility",
    "selection_margin",
    "stability",
    "empirical_confidence",
    "candidate_count"
]

missing_columns = [
    col
    for col in REQUIRED_COLUMNS
    if col not in FINAL_SELECTION_DF.columns
]

if missing_columns:

    raise RuntimeError(
        "Notebook 07.17 cannot proceed.\n"
        "Missing required columns:\n"
        + "\n".join(missing_columns)
    )


# ------------------------------------------------------------
# Comparative evidence availability
# ------------------------------------------------------------

FINAL_SELECTION_DF[
    "comparative_evidence"
] = (
    pd.to_numeric(
        FINAL_SELECTION_DF[
            "candidate_count"
        ],
        errors="coerce"
    )
    >= 2
)


# ------------------------------------------------------------
# Confidence pass
# ------------------------------------------------------------

FINAL_SELECTION_DF[
    "confidence_pass"
] = (
    pd.to_numeric(
        FINAL_SELECTION_DF[
            "empirical_confidence"
        ],
        errors="coerce"
    )
    >= CONFIDENCE_THRESHOLD
)


# ------------------------------------------------------------
# Margin availability and pass
# ------------------------------------------------------------

FINAL_SELECTION_DF[
    "margin_available"
] = (
    pd.to_numeric(
        FINAL_SELECTION_DF[
            "selection_margin"
        ],
        errors="coerce"
    )
    .notna()
)


FINAL_SELECTION_DF[
    "margin_pass"
] = np.where(
    FINAL_SELECTION_DF[
        "margin_available"
    ],
    (
        pd.to_numeric(
            FINAL_SELECTION_DF[
                "selection_margin"
            ],
            errors="coerce"
        )
        >= MARGIN_THRESHOLD
    ),
    False
)


# ------------------------------------------------------------
# Stability pass
# ------------------------------------------------------------

FINAL_SELECTION_DF[
    "stability_pass"
] = (
    pd.to_numeric(
        FINAL_SELECTION_DF[
            "stability"
        ],
        errors="coerce"
    )
    >= STABILITY_THRESHOLD
)


# ------------------------------------------------------------
# Evidence-aware decision logic
#
# CASE 1:
# At least two candidates exist.
# -> Full comparative selection criteria apply.
#
# CASE 2:
# Only one candidate exists.
# -> No comparative margin exists.
# -> Do NOT call this a failed model.
# -> Mark as insufficient comparative evidence.
# ------------------------------------------------------------

comparative_ready = (
    FINAL_SELECTION_DF[
        "comparative_evidence"
    ]
)


single_candidate = (
    ~FINAL_SELECTION_DF[
        "comparative_evidence"
    ]
)


# ------------------------------------------------------------
# Abstention logic
# ------------------------------------------------------------

FINAL_SELECTION_DF[
    "abstain"
] = False


# Comparative cases:
# abstain if any required criterion fails.

FINAL_SELECTION_DF.loc[
    comparative_ready,
    "abstain"
] = ~(
    FINAL_SELECTION_DF.loc[
        comparative_ready,
        "confidence_pass"
    ]
    &
    FINAL_SELECTION_DF.loc[
        comparative_ready,
        "margin_pass"
    ]
    &
    FINAL_SELECTION_DF.loc[
        comparative_ready,
        "stability_pass"
    ]
)


# Single-candidate cases:
# cannot claim comparative confidence.

FINAL_SELECTION_DF.loc[
    single_candidate,
    "abstain"
] = True


# ------------------------------------------------------------
# Decision status
# ------------------------------------------------------------

FINAL_SELECTION_DF[
    "decision_status"
] = "ABSTAIN_ADDITIONAL_EVALUATION"


# Comparative selections satisfying all criteria

comparative_select = (
    comparative_ready
    &
    ~FINAL_SELECTION_DF[
        "abstain"
    ]
)


FINAL_SELECTION_DF.loc[
    comparative_select,
    "decision_status"
] = "SELECT"


# Single-candidate condition

FINAL_SELECTION_DF.loc[
    single_candidate,
    "decision_status"
] = (
    "ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE"
)


# Comparative but confidence/margin/stability failure

comparative_abstain = (
    comparative_ready
    &
    FINAL_SELECTION_DF[
        "abstain"
    ]
)


FINAL_SELECTION_DF.loc[
    comparative_abstain,
    "decision_status"
] = (
    "ABSTAIN_LOW_EMPIRICAL_CONFIDENCE"
)


# ------------------------------------------------------------
# Evidence status
# ------------------------------------------------------------

FINAL_SELECTION_DF[
    "evidence_status"
] = np.where(
    FINAL_SELECTION_DF[
        "candidate_count"
    ] >= 2,
    "COMPARATIVE_EVIDENCE_AVAILABLE",
    "SINGLE_CANDIDATE_ONLY"
)


# ------------------------------------------------------------
# Decision summary
# ------------------------------------------------------------

print("\nDECISION SUMMARY")
print("-" * 100)

decision_counts = (
    FINAL_SELECTION_DF[
        "decision_status"
    ]
    .value_counts()
)

for status, count in decision_counts.items():

    print(
        f"{status:45s}: {int(count):,}"
    )


# ------------------------------------------------------------
# Evidence summary
# ------------------------------------------------------------

print("\nEVIDENCE SUMMARY")
print("-" * 100)

total_features = len(
    FINAL_SELECTION_DF
)

comparative_count = int(
    FINAL_SELECTION_DF[
        "comparative_evidence"
    ].sum()
)

single_candidate_count = int(
    FINAL_SELECTION_DF[
        "comparative_evidence"
    ].eq(False)
    .sum()
)

print(
    f"Total feature decisions              : "
    f"{total_features:,}"
)

print(
    f"Features with >=2 candidates         : "
    f"{comparative_count:,}"
)

print(
    f"Single-candidate features            : "
    f"{single_candidate_count:,}"
)


# ------------------------------------------------------------
# Selection summary
# ------------------------------------------------------------

selected_count = int(
    (
        FINAL_SELECTION_DF[
            "decision_status"
        ]
        == "SELECT"
    )
    .sum()
)

abstained_count = int(
    FINAL_SELECTION_DF[
        "abstain"
    ].sum()
)


print("\nSELECTION SUMMARY")
print("-" * 100)

print(
    f"Selected                            : "
    f"{selected_count:,}"
)

print(
    f"Abstained                           : "
    f"{abstained_count:,}"
)


# ------------------------------------------------------------
# Comparative selection rate
# ------------------------------------------------------------

if comparative_count > 0:

    comparative_selection_rate = (
        (
            FINAL_SELECTION_DF.loc[
                comparative_ready,
                "decision_status"
            ]
            == "SELECT"
        )
        .mean()
    )

else:

    comparative_selection_rate = np.nan


print(
    "\nCOMPARATIVE SELECTION RATE"
)
print("-" * 100)

if pd.notna(
    comparative_selection_rate
):

    print(
        f"Comparative selection rate         : "
        f"{comparative_selection_rate:.4f}"
    )

else:

    print(
        "Comparative selection rate         : "
        "NOT AVAILABLE"
    )

    print(
        "Reason                             : "
        "No feature has >=2 evaluated candidates."
    )


# ------------------------------------------------------------
# Sample results
# ------------------------------------------------------------

print(
    "\nSAMPLE FINAL SELECTION RESULTS"
)

print("-" * 100)

display(
    FINAL_SELECTION_DF[
        [
            "dataset_id",
            "feature",
            "selected_strategy",
            "selected_utility",
            "candidate_count",
            "selection_margin",
            "stability",
            "empirical_confidence",
            "comparative_evidence",
            "decision_status"
        ]
    ]
    .head(20)
)


# ------------------------------------------------------------
# Research-integrity interpretation
# ------------------------------------------------------------

print(
    "\n" + "!" * 100
)

print(
    "RESEARCH INTERPRETATION"
)

print(
    "!" * 100
)


if comparative_count == 0:

    print(
        "No feature currently has two or more "
        "empirically evaluated candidate strategies."
    )

    print(
        "\nTherefore, comparative confidence, "
        "selection margins, and comparative "
        "strategy superiority cannot yet be established."
    )

    print(
        "\nThe selected_strategy values represent "
        "the best available empirical candidate, "
        "not a conclusively validated winner."
    )

    print(
        "\nAll single-candidate decisions are therefore "
        "marked as ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE."
    )

else:

    print(
        f"{comparative_count:,} feature decisions have "
        "sufficient comparative evidence."
    )

    print(
        "\nFor these cases, confidence, margin and "
        "stability thresholds are applied."
    )


# ------------------------------------------------------------
# Final integrity checks
# ------------------------------------------------------------

assert (
    len(FINAL_SELECTION_DF)
    == len(EMPIRICAL_SELECTION_DF)
)


assert (
    FINAL_SELECTION_DF[
        "decision_status"
    ].notna().all()
)


print(
    "\n" + "=" * 100
)

print(
    "NOTEBOOK 07.17 CONFIDENCE AND ABSTENTION : COMPLETE"
)

print(
    "=" * 100
)

AIR-LLM — NOTEBOOK 07.17
CONFIDENCE AND ABSTENTION

DECISION SUMMARY
----------------------------------------------------------------------------------------------------
ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE    : 77

EVIDENCE SUMMARY
----------------------------------------------------------------------------------------------------
Total feature decisions              : 77
Features with >=2 candidates         : 0
Single-candidate features            : 77

SELECTION SUMMARY
----------------------------------------------------------------------------------------------------
Selected                            : 0
Abstained                           : 77

COMPARATIVE SELECTION RATE
----------------------------------------------------------------------------------------------------
Comparative selection rate         : NOT AVAILABLE
Reason                             : No feature has >=2 evaluated candidates.

SAMPLE FINAL SELECTION RESULTS
---------------------------------------------

,dataset_id,feature,selected_strategy,selected_utility,candidate_count,selection_margin,stability,empirical_confidence,comparative_evidence,decision_status
0,adult_income,age,mean,0.598182,1,NaN,0.755990,0.829193,False,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
1,adult_income,capital_gain,mean,0.607990,1,NaN,0.780114,0.846080,False,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
2,adult_income,capital_loss,mean,0.627236,1,NaN,0.770431,0.839302,False,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
3,adult_income,education,mode,0.544829,1,NaN,0.731684,0.812178,False,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
4,adult_income,education_num,mean,0.550408,1,NaN,0.720978,0.804685,False,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
5,adult_income,fnlwgt,mean,0.564509,1,NaN,0.747517,0.823262,False,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
6,adult_income,hours_per_week,mean,0.587983,1,NaN,0.750542,0.825379,False,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
7,adult_income,marital_status,mode,0.488762,1,NaN,0.692268,0.784588,False,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
8,adult_income,native_country,mode,0.531706,1,NaN,0.729984,0.810989,False,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
9,adult_income,occupation,mode,0.339888,1,NaN,0.642902,0.750031,False,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE



!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
RESEARCH INTERPRETATION
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
No feature currently has two or more empirically evaluated candidate strategies.

Therefore, comparative confidence, selection margins, and comparative strategy superiority cannot yet be established.

The selected_strategy values represent the best available empirical candidate, not a conclusively validated winner.

All single-candidate decisions are therefore marked as ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE.

NOTEBOOK 07.17 CONFIDENCE AND ABSTENTION : COMPLETE


In [48]:
# ============================================================
# NOTEBOOK 07.18 — ABSTENTION CANDIDATE EXPANSION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.18")
print("ABSTENTION CANDIDATE EXPANSION")
print("=" * 100)


# ------------------------------------------------------------
# Validate required tables
# ------------------------------------------------------------

REQUIRED_FINAL_COLUMNS = [
    "dataset_id",
    "feature",
    "decision_status"
]

missing_final = [
    col
    for col in REQUIRED_FINAL_COLUMNS
    if col not in FINAL_SELECTION_DF.columns
]

if missing_final:

    raise RuntimeError(
        "Notebook 07.18 cannot proceed.\n"
        "Missing FINAL_SELECTION_DF columns:\n"
        + "\n".join(missing_final)
    )


REQUIRED_AGG_COLUMNS = [
    "dataset_id",
    "feature",
    "strategy",
    "robust_utility",
    "stability"
]

missing_agg = [
    col
    for col in REQUIRED_AGG_COLUMNS
    if col not in AGGREGATION_DF.columns
]

if missing_agg:

    raise RuntimeError(
        "Notebook 07.18 cannot proceed.\n"
        "Missing AGGREGATION_DF columns:\n"
        + "\n".join(missing_agg)
    )


# ------------------------------------------------------------
# Identify all abstained decisions
# ------------------------------------------------------------

ABSTENTION_STATUSES = [
    "ABSTAIN_ADDITIONAL_EVALUATION",
    "ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE",
    "ABSTAIN_LOW_EMPIRICAL_CONFIDENCE"
]


ABSTAINED_SELECTIONS_DF = (
    FINAL_SELECTION_DF[
        FINAL_SELECTION_DF[
            "decision_status"
        ].isin(
            ABSTENTION_STATUSES
        )
    ]
    .copy()
)


print(
    "\nABSTENTION SUMMARY"
)

print("-" * 100)

print(
    f"Total final decisions       : "
    f"{len(FINAL_SELECTION_DF):,}"
)

print(
    f"Abstained decisions         : "
    f"{len(ABSTAINED_SELECTIONS_DF):,}"
)


# ------------------------------------------------------------
# Expand candidate pool for abstained features
# ------------------------------------------------------------

ABSTENTION_ROWS = []


for _, selected in (
    ABSTAINED_SELECTIONS_DF.iterrows()
):

    dataset_id = selected[
        "dataset_id"
    ]

    feature = selected[
        "feature"
    ]


    # --------------------------------------------------------
    # Retrieve all empirically available candidates
    # --------------------------------------------------------

    subset = AGGREGATION_DF[
        (
            AGGREGATION_DF[
                "dataset_id"
            ]
            == dataset_id
        )
        &
        (
            AGGREGATION_DF[
                "feature"
            ]
            == feature
        )
    ].copy()


    if subset.empty:
        continue


    # --------------------------------------------------------
    # Remove invalid utility rows
    # --------------------------------------------------------

    subset[
        "robust_utility"
    ] = pd.to_numeric(
        subset[
            "robust_utility"
        ],
        errors="coerce"
    )


    subset = subset[
        subset[
            "robust_utility"
        ].notna()
    ].copy()


    if subset.empty:
        continue


    # --------------------------------------------------------
    # Rank available candidates
    # --------------------------------------------------------

    subset = subset.sort_values(
        [
            "robust_utility",
            "stability"
        ],
        ascending=[
            False,
            False
        ],
        kind="mergesort"
    )


    # --------------------------------------------------------
    # Keep top 3 empirically available candidates
    # --------------------------------------------------------

    top_candidates = subset.head(3)


    for candidate_rank, (
        _, candidate
    ) in enumerate(
        top_candidates.iterrows(),
        start=1
    ):

        ABSTENTION_ROWS.append({

            "dataset_id":
                dataset_id,

            "feature":
                feature,

            "candidate_rank":
                candidate_rank,

            "candidate_strategy":
                candidate[
                    "strategy"
                ],

            "robust_utility":
                float(
                    candidate[
                        "robust_utility"
                    ]
                ),

            "stability":
                float(
                    candidate[
                        "stability"
                    ]
                )
                if pd.notna(
                    candidate[
                        "stability"
                    ]
                )
                else np.nan,

            "source":
                "EMPIRICALLY_AVAILABLE",

            "evaluation_status":
                "ALREADY_EVALUATED",

            "reason":
                selected[
                    "decision_status"
                ]
        })


# ------------------------------------------------------------
# Build abstention candidate table
# ------------------------------------------------------------

ABSTENTION_DF = pd.DataFrame(
    ABSTENTION_ROWS
)


# ------------------------------------------------------------
# Output summary
# ------------------------------------------------------------

print(
    "\nABSTENTION CANDIDATE EXPANSION"
)

print("-" * 100)

print(
    f"Abstained feature decisions : "
    f"{len(ABSTAINED_SELECTIONS_DF):,}"
)

print(
    f"Candidate rows generated    : "
    f"{len(ABSTENTION_DF):,}"
)


if not ABSTENTION_DF.empty:

    print(
        f"Unique abstained features   : "
        f"{ABSTENTION_DF[
            ['dataset_id', 'feature']
        ].drop_duplicates().shape[0]:,}"
    )

    print(
        f"Unique candidate strategies : "
        f"{ABSTENTION_DF[
            'candidate_strategy'
        ].nunique():,}"
    )


# ------------------------------------------------------------
# Candidate-count diagnostic
# ------------------------------------------------------------

if not ABSTENTION_DF.empty:

    candidate_counts = (
        ABSTENTION_DF
        .groupby(
            [
                "dataset_id",
                "feature"
            ]
        )[
            "candidate_strategy"
        ]
        .nunique()
    )


    print(
        "\nCANDIDATE COVERAGE"
    )

    print("-" * 100)

    print(
        f"Features with 1 candidate : "
        f"{int((candidate_counts == 1).sum()):,}"
    )

    print(
        f"Features with 2 candidates: "
        f"{int((candidate_counts == 2).sum()):,}"
    )

    print(
        f"Features with 3 candidates: "
        f"{int((candidate_counts >= 3).sum()):,}"
    )


# ------------------------------------------------------------
# Sample output
# ------------------------------------------------------------

print(
    "\nSAMPLE ABSTENTION CANDIDATES"
)

print("-" * 100)


if not ABSTENTION_DF.empty:

    display(
        ABSTENTION_DF[
            [
                "dataset_id",
                "feature",
                "candidate_rank",
                "candidate_strategy",
                "robust_utility",
                "stability",
                "source",
                "evaluation_status",
                "reason"
            ]
        ]
        .head(30)
    )

else:

    print(
        "No empirically available candidates "
        "were found for the abstained decisions."
    )


# ------------------------------------------------------------
# Research-integrity interpretation
# ------------------------------------------------------------

print(
    "\n" + "!" * 100
)

print(
    "RESEARCH INTERPRETATION"
)

print(
    "!" * 100
)


if len(ABSTAINED_SELECTIONS_DF) == 0:

    print(
        "No decisions were abstained."
    )

elif ABSTENTION_DF.empty:

    print(
        "Abstained feature decisions exist, but "
        "no empirically evaluated candidate rows "
        "are available in AGGREGATION_DF."
    )

    print(
        "\nAdditional candidate evaluation is therefore "
        "required before comparative selection."
    )

else:

    print(
        f"{len(ABSTENTION_DF):,} empirically available "
        "candidate rows were identified for abstained "
        "feature decisions."
    )

    print(
        "\nThese rows represent candidates that have "
        "already been evaluated and can be compared "
        "using their existing empirical utility."
    )

    print(
        "\nThey should NOT be interpreted as newly "
        "evaluated candidates. If fewer than two "
        "candidates are available for a feature, "
        "comparative validation remains unresolved."
    )


# ------------------------------------------------------------
# Final checks
# ------------------------------------------------------------

assert isinstance(
    ABSTENTION_DF,
    pd.DataFrame
)


print(
    "\n" + "=" * 100
)

print(
    "NOTEBOOK 07.18 ABSTENTION CANDIDATE EXPANSION : COMPLETE"
)

print(
    "=" * 100
)

AIR-LLM — NOTEBOOK 07.18
ABSTENTION CANDIDATE EXPANSION

ABSTENTION SUMMARY
----------------------------------------------------------------------------------------------------
Total final decisions       : 77
Abstained decisions         : 77

ABSTENTION CANDIDATE EXPANSION
----------------------------------------------------------------------------------------------------
Abstained feature decisions : 77
Candidate rows generated    : 77
Unique abstained features   : 77
Unique candidate strategies : 2

CANDIDATE COVERAGE
----------------------------------------------------------------------------------------------------
Features with 1 candidate : 77
Features with 2 candidates: 0
Features with 3 candidates: 0

SAMPLE ABSTENTION CANDIDATES
----------------------------------------------------------------------------------------------------


,dataset_id,feature,candidate_rank,candidate_strategy,robust_utility,stability,source,evaluation_status,reason
0,adult_income,age,1,mean,0.598182,0.755990,EMPIRICALLY_AVAILABLE,ALREADY_EVALUATED,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
1,adult_income,capital_gain,1,mean,0.607990,0.780114,EMPIRICALLY_AVAILABLE,ALREADY_EVALUATED,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
2,adult_income,capital_loss,1,mean,0.627236,0.770431,EMPIRICALLY_AVAILABLE,ALREADY_EVALUATED,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
3,adult_income,education,1,mode,0.544829,0.731684,EMPIRICALLY_AVAILABLE,ALREADY_EVALUATED,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
4,adult_income,education_num,1,mean,0.550408,0.720978,EMPIRICALLY_AVAILABLE,ALREADY_EVALUATED,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
5,adult_income,fnlwgt,1,mean,0.564509,0.747517,EMPIRICALLY_AVAILABLE,ALREADY_EVALUATED,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
6,adult_income,hours_per_week,1,mean,0.587983,0.750542,EMPIRICALLY_AVAILABLE,ALREADY_EVALUATED,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
7,adult_income,marital_status,1,mode,0.488762,0.692268,EMPIRICALLY_AVAILABLE,ALREADY_EVALUATED,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
8,adult_income,native_country,1,mode,0.531706,0.729984,EMPIRICALLY_AVAILABLE,ALREADY_EVALUATED,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
9,adult_income,occupation,1,mode,0.339888,0.642902,EMPIRICALLY_AVAILABLE,ALREADY_EVALUATED,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE



!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
RESEARCH INTERPRETATION
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
77 empirically available candidate rows were identified for abstained feature decisions.

These rows represent candidates that have already been evaluated and can be compared using their existing empirical utility.

They should NOT be interpreted as newly evaluated candidates. If fewer than two candidates are available for a feature, comparative validation remains unresolved.

NOTEBOOK 07.18 ABSTENTION CANDIDATE EXPANSION : COMPLETE


In [50]:
# ============================================================
# NOTEBOOK 07.19 — FINAL DECISION MAP
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.19")
print("FINAL STRATEGY DECISION MAP")
print("=" * 100)


# ------------------------------------------------------------
# Helper: convert values to JSON-safe values
# ------------------------------------------------------------

def json_safe_value(value):

    # Missing / non-finite numeric values -> JSON null
    if value is None:
        return None

    if isinstance(value, (float, np.floating)):

        if not np.isfinite(value):
            return None

        return float(value)

    # NumPy integers -> Python int
    if isinstance(value, (int, np.integer)):
        return int(value)

    # NumPy booleans -> Python bool
    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    # Pandas NA / NaT
    try:

        if pd.isna(value):
            return None

    except (TypeError, ValueError):
        pass

    return value


# ------------------------------------------------------------
# Build final decision map
# ------------------------------------------------------------

DECISION_MAP = {}


for _, row in FINAL_SELECTION_DF.iterrows():

    key = (
        f"{row['dataset_id']}::"
        f"{row['feature']}"
    )


    DECISION_MAP[key] = {

        "dataset_id":
            json_safe_value(
                row["dataset_id"]
            ),

        "feature":
            json_safe_value(
                row["feature"]
            ),

        "selected_strategy":
            json_safe_value(
                row["selected_strategy"]
            ),

        "utility":
            json_safe_value(
                row["selected_utility"]
            ),

        "empirical_confidence":
            json_safe_value(
                row["empirical_confidence"]
            ),

        "selection_margin":
            json_safe_value(
                row["selection_margin"]
            ),

        "stability":
            json_safe_value(
                row["stability"]
            ),

        "llm_recommended_strategy":
            json_safe_value(
                row[
                    "llm_recommended_strategy"
                ]
            ),

        "llm_alignment":
            json_safe_value(
                row["llm_alignment"]
            ),

        "status":
            json_safe_value(
                row["decision_status"]
            )
    }


# ------------------------------------------------------------
# Validate JSON safety before writing
# ------------------------------------------------------------

def validate_json_safe(obj):

    if isinstance(obj, dict):

        for value in obj.values():

            validate_json_safe(value)

    elif isinstance(obj, list):

        for value in obj:

            validate_json_safe(value)

    elif isinstance(obj, float):

        if not np.isfinite(obj):

            raise RuntimeError(
                "Non-finite numeric value detected "
                "before JSON export."
            )


validate_json_safe(
    DECISION_MAP
)


# ------------------------------------------------------------
# Output path
# ------------------------------------------------------------

DECISION_MAP_PATH = (
    METADATA_DIR
    / "decision_map.json"
)


# ------------------------------------------------------------
# Save decision map
# ------------------------------------------------------------

with open(
    DECISION_MAP_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        DECISION_MAP,
        f,
        indent=2,
        ensure_ascii=False,
        allow_nan=False
    )


# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------

decision_status_counts = (
    FINAL_SELECTION_DF[
        "decision_status"
    ]
    .value_counts()
    .to_dict()
)


null_llm_count = int(
    FINAL_SELECTION_DF[
        "llm_recommended_strategy"
    ]
    .isna()
    .sum()
)


null_alignment_count = int(
    FINAL_SELECTION_DF[
        "llm_alignment"
    ]
    .isna()
    .sum()
)


print("\nDECISION MAP SUMMARY")
print("-" * 100)

print(
    f"Decision records              : "
    f"{len(DECISION_MAP):,}"
)

print(
    f"SELECT decisions              : "
    f"{decision_status_counts.get('SELECT', 0):,}"
)

print(
    f"ABSTAIN decisions             : "
    f"{decision_status_counts.get('ABSTAIN_ADDITIONAL_EVALUATION', 0):,}"
)

print(
    f"Missing LLM recommendations   : "
    f"{null_llm_count:,}"
)

print(
    f"Missing LLM alignment values  : "
    f"{null_alignment_count:,}"
)


print("\nJSON EXPORT")
print("-" * 100)

print(
    f"Decision map saved:\n"
    f"{DECISION_MAP_PATH}"
)


# ------------------------------------------------------------
# Research-integrity diagnostic
# ------------------------------------------------------------

print("\nRESEARCH STATUS")
print("-" * 100)

if (
    FINAL_SELECTION_DF["decision_status"]
    .eq("ABSTAIN_ADDITIONAL_EVALUATION")
    .all()
):

    print(
        "All current feature decisions remain "
        "ABSTAIN_ADDITIONAL_EVALUATION."
    )

    print(
        "The decision map preserves these abstentions "
        "rather than converting them into unsupported "
        "final selections."
    )

else:

    print(
        "The decision map contains both accepted "
        "and abstained feature decisions."
    )


print("\n" + "=" * 100)
print("NOTEBOOK 07.19 FINAL DECISION MAP : COMPLETE")
print("=" * 100)

AIR-LLM — NOTEBOOK 07.19
FINAL STRATEGY DECISION MAP

DECISION MAP SUMMARY
----------------------------------------------------------------------------------------------------
Decision records              : 77
SELECT decisions              : 0
ABSTAIN decisions             : 0
Missing LLM recommendations   : 66
Missing LLM alignment values  : 66

JSON EXPORT
----------------------------------------------------------------------------------------------------
Decision map saved:
/content/drive/MyDrive/AIR_LLM_Research/data/notebook_07/metadata/decision_map.json

RESEARCH STATUS
----------------------------------------------------------------------------------------------------
The decision map contains both accepted and abstained feature decisions.

NOTEBOOK 07.19 FINAL DECISION MAP : COMPLETE


In [51]:
# ============================================================
# NOTEBOOK 07.20 — SELECTION AUDIT
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.20")
print("SELECTION AUDIT")
print("=" * 100)


AUDIT_COLUMNS = [

    "dataset_id",
    "feature",
    "selected_strategy",

    "selected_utility",

    "selection_margin",

    "stability",

    "evidence_count",

    "empirical_confidence",

    "llm_recommended_strategy",

    "llm_alignment",

    "confidence_pass",

    "margin_pass",

    "stability_pass",

    "abstain",

    "decision_status"
]


SELECTION_AUDIT_DF = (
    FINAL_SELECTION_DF[
        [
            c
            for c in AUDIT_COLUMNS
            if c in FINAL_SELECTION_DF.columns
        ]
    ]
    .sort_values(
        [
            "dataset_id",
            "feature"
        ]
    )
    .reset_index(drop=True)
)


AUDIT_PATH = (
    AUDIT_DIR
    / "selection_audit.csv"
)


SELECTION_AUDIT_DF.to_csv(
    AUDIT_PATH,
    index=False
)


print(
    f"Audit rows: "
    f"{len(SELECTION_AUDIT_DF):,}"
)

print(
    f"Saved to:\n{AUDIT_PATH}"
)

display(
    SELECTION_AUDIT_DF.head(20)
)

AIR-LLM — NOTEBOOK 07.20
SELECTION AUDIT
Audit rows: 77
Saved to:
/content/drive/MyDrive/AIR_LLM_Research/data/notebook_07/audit/selection_audit.csv


,dataset_id,feature,selected_strategy,selected_utility,selection_margin,stability,evidence_count,empirical_confidence,llm_recommended_strategy,llm_alignment,confidence_pass,margin_pass,stability_pass,abstain,decision_status
0,adult_income,age,mean,0.598182,NaN,0.755990,75,0.829193,NaN,NaN,True,False,True,True,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
1,adult_income,capital_gain,mean,0.607990,NaN,0.780114,75,0.846080,NaN,NaN,True,False,True,True,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
2,adult_income,capital_loss,mean,0.627236,NaN,0.770431,75,0.839302,NaN,NaN,True,False,True,True,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
3,adult_income,education,mode,0.544829,NaN,0.731684,75,0.812178,NaN,NaN,True,False,True,True,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
4,adult_income,education_num,mean,0.550408,NaN,0.720978,75,0.804685,NaN,NaN,True,False,True,True,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
5,adult_income,fnlwgt,mean,0.564509,NaN,0.747517,75,0.823262,NaN,NaN,True,False,True,True,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
6,adult_income,hours_per_week,mean,0.587983,NaN,0.750542,75,0.825379,NaN,NaN,True,False,True,True,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
7,adult_income,marital_status,mode,0.488762,NaN,0.692268,75,0.784588,NaN,NaN,True,False,True,True,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
8,adult_income,native_country,mode,0.531706,NaN,0.729984,75,0.810989,gradient_boosting,0.0,True,False,True,True,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
9,adult_income,occupation,mode,0.339888,NaN,0.642902,75,0.750031,gradient_boosting,0.0,True,False,True,True,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE


In [53]:
# ============================================================
# NOTEBOOK 07.21 — ADAPTIVE SELECTION OUTPUT
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.21")
print("ADAPTIVE SELECTION RESULTS")
print("=" * 100)


# ------------------------------------------------------------
# Create adaptive-selection output
# ------------------------------------------------------------

ADAPTIVE_SELECTION_COLUMNS = [

    "dataset_id",
    "feature",

    # Empirical winner
    "selected_strategy",
    "selected_utility",

    # Evidence quality
    "candidate_count",
    "evidence_count",
    "selection_margin",
    "stability",
    "empirical_confidence",

    # LLM information
    "llm_recommended_strategy",
    "llm_recommendation_rank",
    "llm_alignment",

    # Decision information
    "confidence_pass",
    "margin_pass",
    "stability_pass",
    "abstain",
    "decision_status"
]


available_columns = [
    c
    for c in ADAPTIVE_SELECTION_COLUMNS
    if c in FINAL_SELECTION_DF.columns
]


ADAPTIVE_SELECTION_DF = (
    FINAL_SELECTION_DF[
        available_columns
    ]
    .sort_values(
        [
            "dataset_id",
            "feature"
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Add explicit interpretation fields
# ------------------------------------------------------------

# The selected_strategy is the empirically highest-ranked
# strategy in the currently available evaluation evidence.
#
# It is NOT necessarily an accepted final strategy when
# comparative evidence is insufficient.

ADAPTIVE_SELECTION_DF[
    "provisional_empirical_winner"
] = (
    ADAPTIVE_SELECTION_DF[
        "selected_strategy"
    ]
)


ADAPTIVE_SELECTION_DF[
    "final_accepted_strategy"
] = np.where(
    ADAPTIVE_SELECTION_DF[
        "decision_status"
    ].astype(str)
    == "SELECT",
    ADAPTIVE_SELECTION_DF[
        "selected_strategy"
    ].astype(str),
    pd.NA
)


ADAPTIVE_SELECTION_DF[
    "selection_interpretation"
] = np.where(

    ADAPTIVE_SELECTION_DF[
        "decision_status"
    ].astype(str)
    == "SELECT",

    "EMPIRICALLY_ACCEPTED",

    np.where(

        ADAPTIVE_SELECTION_DF[
            "candidate_count"
        ].fillna(0)
        < 2,

        "PROVISIONAL_WINNER_INSUFFICIENT_COMPARATIVE_EVIDENCE",

        "ABSTAIN_REQUIRES_ADDITIONAL_EVALUATION"
    )
)


# ------------------------------------------------------------
# Save adaptive selection
# ------------------------------------------------------------

ADAPTIVE_SELECTION_PATH = (
    SELECTION_DIR
    / "adaptive_selection.csv"
)


ADAPTIVE_SELECTION_DF.to_csv(
    ADAPTIVE_SELECTION_PATH,
    index=False
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\nADAPTIVE SELECTION SUMMARY")
print("-" * 100)

print(
    f"Total feature conditions      : "
    f"{len(ADAPTIVE_SELECTION_DF):,}"
)


if "decision_status" in ADAPTIVE_SELECTION_DF.columns:

    decision_counts = (
        ADAPTIVE_SELECTION_DF[
            "decision_status"
        ]
        .value_counts()
    )

    for status, count in decision_counts.items():

        print(
            f"{str(status):45s}: {int(count):,}"
        )


# ------------------------------------------------------------
# Candidate evidence summary
# ------------------------------------------------------------

if "candidate_count" in ADAPTIVE_SELECTION_DF.columns:

    candidate_counts = pd.to_numeric(
        ADAPTIVE_SELECTION_DF[
            "candidate_count"
        ],
        errors="coerce"
    )

    print(
        "\nCANDIDATE EVIDENCE"
    )
    print("-" * 100)

    print(
        f"Mean candidates per feature : "
        f"{candidate_counts.mean():.2f}"
    )

    print(
        f"Minimum candidates          : "
        f"{candidate_counts.min():.0f}"
    )

    print(
        f"Maximum candidates          : "
        f"{candidate_counts.max():.0f}"
    )

    print(
        f"Features with >=2 candidates: "
        f"{int((candidate_counts >= 2).sum()):,}"
    )


# ------------------------------------------------------------
# Accepted vs provisional decisions
# ------------------------------------------------------------

accepted_mask = (
    ADAPTIVE_SELECTION_DF[
        "decision_status"
    ].astype(str)
    == "SELECT"
)


provisional_mask = (
    ADAPTIVE_SELECTION_DF[
        "selection_interpretation"
    ].astype(str)
    ==
    "PROVISIONAL_WINNER_INSUFFICIENT_COMPARATIVE_EVIDENCE"
)


print(
    "\nFINAL DECISION INTERPRETATION"
)
print("-" * 100)

print(
    f"Accepted adaptive selections : "
    f"{int(accepted_mask.sum()):,}"
)

print(
    f"Provisional empirical winners: "
    f"{int(provisional_mask.sum()):,}"
)

print(
    f"Abstained decisions           : "
    f"{int((~accepted_mask).sum()):,}"
)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print(
    "\nSAMPLE ADAPTIVE SELECTION RESULTS"
)
print("-" * 100)


display(
    ADAPTIVE_SELECTION_DF.head(20)
)


# ------------------------------------------------------------
# Research-integrity interpretation
# ------------------------------------------------------------

print(
    "\n" + "!" * 100
)
print(
    "RESEARCH INTERPRETATION"
)
print(
    "!" * 100
)


if int(accepted_mask.sum()) == 0:

    print(
        "No feature condition currently satisfies the "
        "requirements for final adaptive acceptance."
    )

    print(
        "\nThe selected_strategy field represents the "
        "provisional empirical winner from the available "
        "evaluation evidence."
    )

    print(
        "\nBecause the current evaluation contains only one "
        "candidate strategy for each feature condition, "
        "selection margins and stability estimates cannot "
        "support comparative strategy selection."
    )

    print(
        "\nTherefore, the AIR-LLM framework correctly abstains "
        "from declaring these provisional winners as final "
        "adaptive selections."
    )

    print(
        "\nThis is methodologically preferable to forcing a "
        "selection because it prevents unsupported claims of "
        "comparative empirical superiority."
    )

else:

    print(
        f"{int(accepted_mask.sum()):,} feature conditions "
        "satisfy the current adaptive-selection criteria."
    )

    print(
        f"{int(provisional_mask.sum()):,} additional feature "
        "conditions remain provisional because comparative "
        "evidence is insufficient."
    )


print(
    "\n" + "=" * 100
)
print(
    "NOTEBOOK 07.21 ADAPTIVE SELECTION : COMPLETE"
)
print(
    "=" * 100
)

AIR-LLM — NOTEBOOK 07.21
ADAPTIVE SELECTION RESULTS

ADAPTIVE SELECTION SUMMARY
----------------------------------------------------------------------------------------------------
Total feature conditions      : 77
ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE    : 77

CANDIDATE EVIDENCE
----------------------------------------------------------------------------------------------------
Mean candidates per feature : 1.00
Minimum candidates          : 1
Maximum candidates          : 1
Features with >=2 candidates: 0

FINAL DECISION INTERPRETATION
----------------------------------------------------------------------------------------------------
Accepted adaptive selections : 0
Provisional empirical winners: 77
Abstained decisions           : 77

SAMPLE ADAPTIVE SELECTION RESULTS
----------------------------------------------------------------------------------------------------


,dataset_id,feature,selected_strategy,selected_utility,candidate_count,evidence_count,selection_margin,stability,empirical_confidence,llm_recommended_strategy,llm_recommendation_rank,llm_alignment,confidence_pass,margin_pass,stability_pass,abstain,decision_status,provisional_empirical_winner,final_accepted_strategy,selection_interpretation
0,adult_income,age,mean,0.598182,1,75,NaN,0.755990,0.829193,NaN,NaN,NaN,True,False,True,True,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE,mean,<NA>,PROVISIONAL_WINNER_INSUFFICIENT_COMPARATIVE_EV...
1,adult_income,capital_gain,mean,0.607990,1,75,NaN,0.780114,0.846080,NaN,NaN,NaN,True,False,True,True,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE,mean,<NA>,PROVISIONAL_WINNER_INSUFFICIENT_COMPARATIVE_EV...
2,adult_income,capital_loss,mean,0.627236,1,75,NaN,0.770431,0.839302,NaN,NaN,NaN,True,False,True,True,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE,mean,<NA>,PROVISIONAL_WINNER_INSUFFICIENT_COMPARATIVE_EV...
3,adult_income,education,mode,0.544829,1,75,NaN,0.731684,0.812178,NaN,NaN,NaN,True,False,True,True,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE,mode,<NA>,PROVISIONAL_WINNER_INSUFFICIENT_COMPARATIVE_EV...
4,adult_income,education_num,mean,0.550408,1,75,NaN,0.720978,0.804685,NaN,NaN,NaN,True,False,True,True,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE,mean,<NA>,PROVISIONAL_WINNER_INSUFFICIENT_COMPARATIVE_EV...
5,adult_income,fnlwgt,mean,0.564509,1,75,NaN,0.747517,0.823262,NaN,NaN,NaN,True,False,True,True,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE,mean,<NA>,PROVISIONAL_WINNER_INSUFFICIENT_COMPARATIVE_EV...
6,adult_income,hours_per_week,mean,0.587983,1,75,NaN,0.750542,0.825379,NaN,NaN,NaN,True,False,True,True,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE,mean,<NA>,PROVISIONAL_WINNER_INSUFFICIENT_COMPARATIVE_EV...
7,adult_income,marital_status,mode,0.488762,1,75,NaN,0.692268,0.784588,NaN,NaN,NaN,True,False,True,True,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE,mode,<NA>,PROVISIONAL_WINNER_INSUFFICIENT_COMPARATIVE_EV...
8,adult_income,native_country,mode,0.531706,1,75,NaN,0.729984,0.810989,gradient_boosting,1.0,0.0,True,False,True,True,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE,mode,<NA>,PROVISIONAL_WINNER_INSUFFICIENT_COMPARATIVE_EV...
9,adult_income,occupation,mode,0.339888,1,75,NaN,0.642902,0.750031,gradient_boosting,1.0,0.0,True,False,True,True,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE,mode,<NA>,PROVISIONAL_WINNER_INSUFFICIENT_COMPARATIVE_EV...



!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
RESEARCH INTERPRETATION
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
No feature condition currently satisfies the requirements for final adaptive acceptance.

The selected_strategy field represents the provisional empirical winner from the available evaluation evidence.

Because the current evaluation contains only one candidate strategy for each feature condition, selection margins and stability estimates cannot support comparative strategy selection.

Therefore, the AIR-LLM framework correctly abstains from declaring these provisional winners as final adaptive selections.

This is methodologically preferable to forcing a selection because it prevents unsupported claims of comparative empirical superiority.

NOTEBOOK 07.21 ADAPTIVE SELECTION : COMPLETE


In [56]:
# ============================================================
# NOTEBOOK 07.22 — FINAL STRATEGY SELECTION
# RESEARCH-INTEGRITY CONTROLLED VERSION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.22")
print("FINAL STRATEGY SELECTION")
print("=" * 100)


# ------------------------------------------------------------
# 1. COPY REQUIRED SELECTION INFORMATION
# ------------------------------------------------------------

FINAL_STRATEGY_DF = FINAL_SELECTION_DF.copy()


# ------------------------------------------------------------
# 2. DETERMINE COMPARATIVE EVIDENCE
# ------------------------------------------------------------
# A final adaptive selection is permitted only when at least
# two candidate strategies were empirically evaluated for the
# same dataset-feature condition.

if "candidate_count" in FINAL_STRATEGY_DF.columns:

    FINAL_STRATEGY_DF["candidate_count"] = pd.to_numeric(
        FINAL_STRATEGY_DF["candidate_count"],
        errors="coerce"
    ).fillna(0).astype(int)

else:

    FINAL_STRATEGY_DF["candidate_count"] = 0


FINAL_STRATEGY_DF["has_comparative_evidence"] = (
    FINAL_STRATEGY_DF["candidate_count"] >= 2
)


# ------------------------------------------------------------
# 3. DEFINE FINAL SELECTION
# ------------------------------------------------------------
# IMPORTANT:
# selected_strategy may contain the provisional empirical
# winner even when comparative evidence is unavailable.
#
# final_selection is therefore intentionally NA in such cases.

FINAL_STRATEGY_DF["final_selection"] = np.where(

    FINAL_STRATEGY_DF["has_comparative_evidence"]
    &
    (FINAL_STRATEGY_DF["decision_status"] == "SELECT"),

    FINAL_STRATEGY_DF["selected_strategy"],

    pd.NA
)


# ------------------------------------------------------------
# 4. STANDARDISE DECISION STATUS
# ------------------------------------------------------------

FINAL_STRATEGY_DF["decision_status"] = np.where(

    FINAL_STRATEGY_DF["has_comparative_evidence"],

    FINAL_STRATEGY_DF["decision_status"],

    "ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE"
)


# ------------------------------------------------------------
# 5. DEFINE SELECTION BASIS
# ------------------------------------------------------------

FINAL_STRATEGY_DF["selection_basis"] = np.where(

    FINAL_STRATEGY_DF["final_selection"].notna(),

    "EMPIRICAL_COMPARATIVE_UTILITY",

    "PROVISIONAL_EMPIRICAL_WINNER_ONLY"
)


# ------------------------------------------------------------
# 6. RESEARCH INTERPRETATION
# ------------------------------------------------------------

FINAL_STRATEGY_DF["research_interpretation"] = np.where(

    FINAL_STRATEGY_DF["has_comparative_evidence"],

    "COMPARATIVE_EMPIRICAL_SELECTION",

    "PROVISIONAL_WINNER_SINGLE_CANDIDATE"
)


# ------------------------------------------------------------
# 7. FINAL OUTPUT COLUMNS
# ------------------------------------------------------------

FINAL_STRATEGY_COLUMNS = [

    "dataset_id",
    "feature",

    # Provisional empirical winner
    "selected_strategy",

    # True final selection
    "final_selection",

    "candidate_count",

    "selected_utility",
    "empirical_confidence",
    "selection_margin",
    "stability",

    "decision_status",

    "selection_basis",

    "research_interpretation"
]


FINAL_STRATEGY_DF = (

    FINAL_STRATEGY_DF[
        [
            c
            for c in FINAL_STRATEGY_COLUMNS
            if c in FINAL_STRATEGY_DF.columns
        ]
    ]

    .sort_values(
        [
            "dataset_id",
            "feature"
        ]
    )

    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 8. SUMMARY
# ------------------------------------------------------------

TOTAL = len(FINAL_STRATEGY_DF)

FINAL_SELECT_COUNT = int(
    (
        FINAL_STRATEGY_DF["final_selection"]
        .notna()
    ).sum()
)

ABSTAIN_COUNT = int(
    (
        FINAL_STRATEGY_DF["final_selection"]
        .isna()
    ).sum()
)

COMPARATIVE_COUNT = int(
    (
        FINAL_STRATEGY_DF["candidate_count"] >= 2
    ).sum()
)

SINGLE_COUNT = int(
    (
        FINAL_STRATEGY_DF["candidate_count"] == 1
    ).sum()
)

NO_EVIDENCE_COUNT = int(
    (
        FINAL_STRATEGY_DF["candidate_count"] == 0
    ).sum()
)


# ------------------------------------------------------------
# 9. PRINT RESEARCH-GRADE SUMMARY
# ------------------------------------------------------------

print("\nFINAL STRATEGY SELECTION SUMMARY")
print("-" * 100)

print(
    f"Total feature conditions : {TOTAL:,}"
)

print(
    f"Final SELECT decisions   : {FINAL_SELECT_COUNT:,}"
)

print(
    f"Abstained decisions      : {ABSTAIN_COUNT:,}"
)

print(
    f"With >=2 candidates      : {COMPARATIVE_COUNT:,}"
)

print(
    f"Single-candidate cases   : {SINGLE_COUNT:,}"
)

print(
    f"No evaluated candidates  : {NO_EVIDENCE_COUNT:,}"
)


print("\nDECISION STATUS DISTRIBUTION")
print("-" * 100)

status_counts = (
    FINAL_STRATEGY_DF["decision_status"]
    .value_counts(dropna=False)
)

for status, count in status_counts.items():

    print(
        f"{str(status):<55} : {int(count):,}"
    )


# ------------------------------------------------------------
# 10. PROVISIONAL WINNER SUMMARY
# ------------------------------------------------------------

PROVISIONAL_COUNT = int(
    FINAL_STRATEGY_DF["selected_strategy"]
    .notna()
    .sum()
)

print("\nPROVISIONAL EMPIRICAL WINNERS")
print("-" * 100)

print(
    f"Provisional winners available : "
    f"{PROVISIONAL_COUNT:,}"
)

print(
    f"Final adaptive selections     : "
    f"{FINAL_SELECT_COUNT:,}"
)


# ------------------------------------------------------------
# 11. SAMPLE RESULTS
# ------------------------------------------------------------

print("\nSAMPLE FINAL STRATEGY RESULTS")
print("-" * 100)

display(
    FINAL_STRATEGY_DF.head(20)
)


# ------------------------------------------------------------
# 12. RESEARCH-INTEGRITY INTERPRETATION
# ------------------------------------------------------------

print("\n" + "!" * 100)
print("RESEARCH-INTEGRITY INTERPRETATION")
print("!" * 100)

if COMPARATIVE_COUNT == 0:

    print(
        "No feature condition currently satisfies the minimum "
        "requirement for final adaptive selection."
    )

    print()

    print(
        "The selected_strategy column represents the provisional "
        "empirical winner obtained from the available evaluation evidence."
    )

    print()

    print(
        "Because each feature currently has fewer than two "
        "empirically evaluated candidate strategies, comparative "
        "selection_margin and stability cannot establish "
        "relative superiority."
    )

    print()

    print(
        "Therefore, final_selection remains undefined (NA) and "
        "the framework abstains from declaring these strategies "
        "as final adaptive selections."
    )

    print()

    print(
        "This preserves research integrity and prevents unsupported "
        "claims of comparative superiority."
    )

else:

    print(
        f"{COMPARATIVE_COUNT:,} feature conditions have sufficient "
        "comparative candidate evidence."
    )

    print(
        f"{FINAL_SELECT_COUNT:,} feature conditions satisfy the "
        "current final-selection criteria."
    )

    print(
        f"{ABSTAIN_COUNT:,} feature conditions remain abstained."
    )


# ------------------------------------------------------------
# 13. EXPORT
# ------------------------------------------------------------

FINAL_STRATEGY_PATH = (
    SELECTION_DIR
    / "final_strategy_selection.csv"
)


FINAL_STRATEGY_DF.to_csv(
    FINAL_STRATEGY_PATH,
    index=False
)


print("\nFINAL STRATEGY EXPORT")
print("-" * 100)

print(
    f"Final strategy table saved:\n"
    f"{FINAL_STRATEGY_PATH}"
)


print("\n" + "=" * 100)
print("NOTEBOOK 07.22 FINAL STRATEGY SELECTION : COMPLETE")
print("=" * 100)

AIR-LLM — NOTEBOOK 07.22
FINAL STRATEGY SELECTION

FINAL STRATEGY SELECTION SUMMARY
----------------------------------------------------------------------------------------------------
Total feature conditions : 77
Final SELECT decisions   : 0
Abstained decisions      : 77
With >=2 candidates      : 0
Single-candidate cases   : 77
No evaluated candidates  : 0

DECISION STATUS DISTRIBUTION
----------------------------------------------------------------------------------------------------
ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE               : 77

PROVISIONAL EMPIRICAL WINNERS
----------------------------------------------------------------------------------------------------
Provisional winners available : 77
Final adaptive selections     : 0

SAMPLE FINAL STRATEGY RESULTS
----------------------------------------------------------------------------------------------------


,dataset_id,feature,selected_strategy,final_selection,candidate_count,selected_utility,empirical_confidence,selection_margin,stability,decision_status,selection_basis,research_interpretation
0,adult_income,age,mean,<NA>,1,0.598182,0.829193,NaN,0.755990,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE,PROVISIONAL_EMPIRICAL_WINNER_ONLY,PROVISIONAL_WINNER_SINGLE_CANDIDATE
1,adult_income,capital_gain,mean,<NA>,1,0.607990,0.846080,NaN,0.780114,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE,PROVISIONAL_EMPIRICAL_WINNER_ONLY,PROVISIONAL_WINNER_SINGLE_CANDIDATE
2,adult_income,capital_loss,mean,<NA>,1,0.627236,0.839302,NaN,0.770431,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE,PROVISIONAL_EMPIRICAL_WINNER_ONLY,PROVISIONAL_WINNER_SINGLE_CANDIDATE
3,adult_income,education,mode,<NA>,1,0.544829,0.812178,NaN,0.731684,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE,PROVISIONAL_EMPIRICAL_WINNER_ONLY,PROVISIONAL_WINNER_SINGLE_CANDIDATE
4,adult_income,education_num,mean,<NA>,1,0.550408,0.804685,NaN,0.720978,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE,PROVISIONAL_EMPIRICAL_WINNER_ONLY,PROVISIONAL_WINNER_SINGLE_CANDIDATE
5,adult_income,fnlwgt,mean,<NA>,1,0.564509,0.823262,NaN,0.747517,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE,PROVISIONAL_EMPIRICAL_WINNER_ONLY,PROVISIONAL_WINNER_SINGLE_CANDIDATE
6,adult_income,hours_per_week,mean,<NA>,1,0.587983,0.825379,NaN,0.750542,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE,PROVISIONAL_EMPIRICAL_WINNER_ONLY,PROVISIONAL_WINNER_SINGLE_CANDIDATE
7,adult_income,marital_status,mode,<NA>,1,0.488762,0.784588,NaN,0.692268,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE,PROVISIONAL_EMPIRICAL_WINNER_ONLY,PROVISIONAL_WINNER_SINGLE_CANDIDATE
8,adult_income,native_country,mode,<NA>,1,0.531706,0.810989,NaN,0.729984,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE,PROVISIONAL_EMPIRICAL_WINNER_ONLY,PROVISIONAL_WINNER_SINGLE_CANDIDATE
9,adult_income,occupation,mode,<NA>,1,0.339888,0.750031,NaN,0.642902,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE,PROVISIONAL_EMPIRICAL_WINNER_ONLY,PROVISIONAL_WINNER_SINGLE_CANDIDATE



!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
RESEARCH-INTEGRITY INTERPRETATION
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
No feature condition currently satisfies the minimum requirement for final adaptive selection.

The selected_strategy column represents the provisional empirical winner obtained from the available evaluation evidence.

Because each feature currently has fewer than two empirically evaluated candidate strategies, comparative selection_margin and stability cannot establish relative superiority.

Therefore, final_selection remains undefined (NA) and the framework abstains from declaring these strategies as final adaptive selections.

This preserves research integrity and prevents unsupported claims of comparative superiority.

FINAL STRATEGY EXPORT
----------------------------------------------------------------------------------------------------
Final str

In [58]:
# ============================================================
# NOTEBOOK 07.23 — UTILITY COMPONENT ANALYSIS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.23")
print("UTILITY COMPONENT ANALYSIS")
print("=" * 100)


# ------------------------------------------------------------
# 1. VALIDATE REQUIRED INPUTS
# ------------------------------------------------------------

required_final_cols = {
    "dataset_id",
    "feature",
    "selected_strategy",
    "decision_status"
}

missing_final_cols = (
    required_final_cols
    - set(FINAL_STRATEGY_DF.columns)
)

if missing_final_cols:
    raise RuntimeError(
        "FINAL_STRATEGY_DF is missing required columns: "
        f"{sorted(missing_final_cols)}"
    )


required_utility_cols = {
    "dataset_id",
    "feature",
    "strategy"
}

missing_utility_cols = (
    required_utility_cols
    - set(UTILITY_DF.columns)
)

if missing_utility_cols:
    raise RuntimeError(
        "UTILITY_DF is missing required columns: "
        f"{sorted(missing_utility_cols)}"
    )


# ------------------------------------------------------------
# 2. IDENTIFY FINAL ADAPTIVE WINNERS
# ------------------------------------------------------------

FINAL_WINNER_KEYS = (
    FINAL_STRATEGY_DF[
        FINAL_STRATEGY_DF[
            "decision_status"
        ].astype(str).eq("SELECT")
    ][
        [
            "dataset_id",
            "feature",
            "selected_strategy"
        ]
    ]
    .drop_duplicates()
    .copy()
)


# ------------------------------------------------------------
# 3. IDENTIFY PROVISIONAL WINNERS
# ------------------------------------------------------------

PROVISIONAL_WINNER_KEYS = (
    FINAL_STRATEGY_DF[
        FINAL_STRATEGY_DF[
            "selected_strategy"
        ].notna()
        &
        ~FINAL_STRATEGY_DF[
            "selected_strategy"
        ].astype(str).isin(["", "nan", "None"])
    ][
        [
            "dataset_id",
            "feature",
            "selected_strategy",
            "decision_status"
        ]
    ]
    .drop_duplicates()
    .copy()
)


print("\nSELECTION STATUS")
print("-" * 100)

print(
    f"Final adaptive winners     : "
    f"{len(FINAL_WINNER_KEYS):,}"
)

print(
    f"Provisional winners        : "
    f"{len(PROVISIONAL_WINNER_KEYS):,}"
)


# ------------------------------------------------------------
# 4. UTILITY COMPONENT DEFINITIONS
# ------------------------------------------------------------

COMPONENT_COLUMNS = [

    "dataset_id",
    "feature",
    "strategy",

    "accuracy_score",
    "distribution_score",
    "dependency_score",
    "predictive_score",
    "efficiency_score",

    "weight_accuracy",
    "weight_distribution",
    "weight_dependency",
    "weight_predictive",
    "weight_efficiency",

    "adaptive_utility"
]


AVAILABLE_COMPONENT_COLUMNS = [
    c
    for c in COMPONENT_COLUMNS
    if c in UTILITY_DF.columns
]


# ------------------------------------------------------------
# 5. FINAL WINNER COMPONENT ANALYSIS
# ------------------------------------------------------------

if not FINAL_WINNER_KEYS.empty:

    FINAL_WINNER_COMPONENTS_DF = (
        UTILITY_DF.merge(
            FINAL_WINNER_KEYS,
            left_on=[
                "dataset_id",
                "feature",
                "strategy"
            ],
            right_on=[
                "dataset_id",
                "feature",
                "selected_strategy"
            ],
            how="inner"
        )
        .drop_duplicates()
        .reset_index(drop=True)
    )

else:

    FINAL_WINNER_COMPONENTS_DF = pd.DataFrame(
        columns=AVAILABLE_COMPONENT_COLUMNS
    )


# ------------------------------------------------------------
# 6. PROVISIONAL WINNER COMPONENT ANALYSIS
# ------------------------------------------------------------

if not PROVISIONAL_WINNER_KEYS.empty:

    PROVISIONAL_WINNER_COMPONENTS_DF = (
        UTILITY_DF.merge(
            PROVISIONAL_WINNER_KEYS[
                [
                    "dataset_id",
                    "feature",
                    "selected_strategy",
                    "decision_status"
                ]
            ],
            left_on=[
                "dataset_id",
                "feature",
                "strategy"
            ],
            right_on=[
                "dataset_id",
                "feature",
                "selected_strategy"
            ],
            how="inner"
        )
        .drop_duplicates()
        .reset_index(drop=True)
    )

else:

    PROVISIONAL_WINNER_COMPONENTS_DF = pd.DataFrame()


# ------------------------------------------------------------
# 7. SAVE FINAL WINNER COMPONENTS
# ------------------------------------------------------------

FINAL_COMPONENT_PATH = (
    AUDIT_DIR
    / "final_winner_utility_components.csv"
)

if not FINAL_WINNER_COMPONENTS_DF.empty:

    FINAL_WINNER_COMPONENTS_DF[
        AVAILABLE_COMPONENT_COLUMNS
    ].to_csv(
        FINAL_COMPONENT_PATH,
        index=False
    )

else:

    # Create an empty schema-valid file so downstream
    # notebooks can detect that no final winners exist.
    pd.DataFrame(
        columns=AVAILABLE_COMPONENT_COLUMNS
    ).to_csv(
        FINAL_COMPONENT_PATH,
        index=False
    )


# ------------------------------------------------------------
# 8. SAVE PROVISIONAL WINNER COMPONENTS
# ------------------------------------------------------------

PROVISIONAL_COMPONENT_PATH = (
    AUDIT_DIR
    / "provisional_winner_utility_components.csv"
)

if not PROVISIONAL_WINNER_COMPONENTS_DF.empty:

    provisional_export_columns = [
        c
        for c in (
            AVAILABLE_COMPONENT_COLUMNS
            + ["decision_status"]
        )
        if c in PROVISIONAL_WINNER_COMPONENTS_DF.columns
    ]

    PROVISIONAL_WINNER_COMPONENTS_DF[
        provisional_export_columns
    ].to_csv(
        PROVISIONAL_COMPONENT_PATH,
        index=False
    )

else:

    pd.DataFrame().to_csv(
        PROVISIONAL_COMPONENT_PATH,
        index=False
    )


# ------------------------------------------------------------
# 9. COMPONENT SUMMARY
# ------------------------------------------------------------

print("\nUTILITY COMPONENT SUMMARY")
print("-" * 100)

print(
    f"Final winner component rows      : "
    f"{len(FINAL_WINNER_COMPONENTS_DF):,}"
)

print(
    f"Provisional winner component rows: "
    f"{len(PROVISIONAL_WINNER_COMPONENTS_DF):,}"
)


# ------------------------------------------------------------
# 10. SHOW FINAL COMPONENTS WHEN AVAILABLE
# ------------------------------------------------------------

if not FINAL_WINNER_COMPONENTS_DF.empty:

    print("\nFINAL ADAPTIVE WINNER COMPONENTS")
    print("-" * 100)

    display(
        FINAL_WINNER_COMPONENTS_DF[
            AVAILABLE_COMPONENT_COLUMNS
        ].head(20)
    )

else:

    print("\nFINAL ADAPTIVE WINNER COMPONENTS")
    print("-" * 100)

    print(
        "No final adaptive winners are currently available."
    )

    print(
        "All current feature decisions remain provisional "
        "because comparative candidate evidence is insufficient."
    )


# ------------------------------------------------------------
# 11. SHOW PROVISIONAL COMPONENTS
# ------------------------------------------------------------

if not PROVISIONAL_WINNER_COMPONENTS_DF.empty:

    print("\nPROVISIONAL EMPIRICAL WINNER COMPONENTS")
    print("-" * 100)

    provisional_display_columns = [
        c
        for c in (
            AVAILABLE_COMPONENT_COLUMNS
            + ["decision_status"]
        )
        if c in PROVISIONAL_WINNER_COMPONENTS_DF.columns
    ]

    display(
        PROVISIONAL_WINNER_COMPONENTS_DF[
            provisional_display_columns
        ].head(20)
    )


# ------------------------------------------------------------
# 12. RESEARCH-INTEGRITY INTERPRETATION
# ------------------------------------------------------------

print("\n" + "!" * 100)
print("RESEARCH-INTEGRITY INTERPRETATION")
print("!" * 100)

if not FINAL_WINNER_COMPONENTS_DF.empty:

    print(
        "Utility components are reported for feature conditions "
        "that satisfy the final adaptive-selection criteria."
    )

else:

    print(
        "No final adaptive-selection utility components are "
        "currently available."
    )

    print(
        "The current evaluation provides provisional empirical "
        "winners only."
    )

    print(
        "Because each feature condition currently has fewer than "
        "two evaluated candidate strategies, comparative utility "
        "evidence is insufficient to establish final superiority."
    )

    print(
        "The provisional utility values are retained for diagnostic "
        "analysis but are not interpreted as final adaptive selections."
    )


# ------------------------------------------------------------
# 13. EXPORT SUMMARY
# ------------------------------------------------------------

print("\nUTILITY COMPONENT EXPORT")
print("-" * 100)

print(
    f"Final components saved:\n"
    f"{FINAL_COMPONENT_PATH}"
)

print(
    f"Provisional components saved:\n"
    f"{PROVISIONAL_COMPONENT_PATH}"
)

print(
    "\n" + "=" * 100
)

print(
    "NOTEBOOK 07.23 UTILITY COMPONENT ANALYSIS : COMPLETE"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 07.23
UTILITY COMPONENT ANALYSIS

SELECTION STATUS
----------------------------------------------------------------------------------------------------
Final adaptive winners     : 0
Provisional winners        : 77

UTILITY COMPONENT SUMMARY
----------------------------------------------------------------------------------------------------
Final winner component rows      : 0
Provisional winner component rows: 5,775

FINAL ADAPTIVE WINNER COMPONENTS
----------------------------------------------------------------------------------------------------
No final adaptive winners are currently available.
All current feature decisions remain provisional because comparative candidate evidence is insufficient.

PROVISIONAL EMPIRICAL WINNER COMPONENTS
----------------------------------------------------------------------------------------------------


,dataset_id,feature,strategy,accuracy_score,distribution_score,dependency_score,efficiency_score,weight_distribution,weight_dependency,weight_efficiency,adaptive_utility,decision_status
0,adult_income,age,mean,NaN,NaN,NaN,0.821487,0.189445,0.147499,0.142083,0.681993,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
1,adult_income,age,mean,NaN,NaN,NaN,0.000000,0.189445,0.147499,0.142083,0.505961,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
2,adult_income,age,mean,NaN,NaN,NaN,0.925594,0.189445,0.147499,0.142083,0.704302,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
3,adult_income,age,mean,NaN,NaN,NaN,0.968097,0.189445,0.147499,0.142083,0.713410,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
4,adult_income,age,mean,NaN,NaN,NaN,0.818830,0.189445,0.147499,0.142083,0.681424,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
5,adult_income,age,mean,NaN,NaN,NaN,0.000000,0.189445,0.147499,0.142083,0.574587,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
6,adult_income,age,mean,NaN,NaN,NaN,0.000000,0.189445,0.147499,0.142083,0.574587,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
7,adult_income,age,mean,NaN,NaN,NaN,0.901180,0.189445,0.147499,0.142083,0.767697,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
8,adult_income,age,mean,NaN,NaN,NaN,0.861334,0.189445,0.147499,0.142083,0.759158,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE
9,adult_income,age,mean,NaN,NaN,NaN,0.056810,0.189445,0.147499,0.142083,0.586760,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE



!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
RESEARCH-INTEGRITY INTERPRETATION
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
No final adaptive-selection utility components are currently available.
The current evaluation provides provisional empirical winners only.
Because each feature condition currently has fewer than two evaluated candidate strategies, comparative utility evidence is insufficient to establish final superiority.
The provisional utility values are retained for diagnostic analysis but are not interpreted as final adaptive selections.

UTILITY COMPONENT EXPORT
----------------------------------------------------------------------------------------------------
Final components saved:
/content/drive/MyDrive/AIR_LLM_Research/data/notebook_07/audit/final_winner_utility_components.csv
Provisional components saved:
/content/drive/MyDrive/AIR_LLM_Research/data/noteboo

In [60]:
# ============================================================
# NOTEBOOK 07.24 — STRATEGY SELECTION SUMMARY
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.24")
print("STRATEGY SELECTION SUMMARY")
print("=" * 100)


# ------------------------------------------------------------
# 1. VALIDATE INPUT
# ------------------------------------------------------------

REQUIRED_COLUMNS = {
    "dataset_id",
    "feature",
    "selected_strategy",
    "selected_utility",
    "empirical_confidence",
    "selection_margin",
    "stability",
    "decision_status"
}

MISSING_COLUMNS = (
    REQUIRED_COLUMNS
    - set(FINAL_STRATEGY_DF.columns)
)

if MISSING_COLUMNS:
    raise RuntimeError(
        "FINAL_STRATEGY_DF is missing required columns: "
        f"{sorted(MISSING_COLUMNS)}"
    )


# ------------------------------------------------------------
# 2. CLASSIFY FINAL VS PROVISIONAL
# ------------------------------------------------------------

SUMMARY_SOURCE_DF = (
    FINAL_STRATEGY_DF.copy()
)


SUMMARY_SOURCE_DF[
    "selection_class"
] = np.where(
    SUMMARY_SOURCE_DF[
        "decision_status"
    ].astype(str).eq("SELECT"),
    "FINAL_ADAPTIVE_SELECTION",
    np.where(
        SUMMARY_SOURCE_DF[
            "selected_strategy"
        ].notna(),
        "PROVISIONAL_EMPIRICAL_WINNER",
        "NO_SELECTION"
    )
)


# ------------------------------------------------------------
# 3. STRATEGY SUMMARY
# ------------------------------------------------------------

STRATEGY_SUMMARY_DF = (
    SUMMARY_SOURCE_DF
    .groupby(
        [
            "selected_strategy",
            "selection_class"
        ],
        dropna=False
    )
    .agg(
        feature_count=(
            "feature",
            "count"
        ),

        mean_utility=(
            "selected_utility",
            "mean"
        ),

        mean_confidence=(
            "empirical_confidence",
            "mean"
        ),

        mean_margin=(
            "selection_margin",
            "mean"
        ),

        mean_stability=(
            "stability",
            "mean"
        )
    )
    .reset_index()
    .sort_values(
        [
            "selection_class",
            "feature_count"
        ],
        ascending=[
            True,
            False
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 4. ROUND NUMERIC VALUES
# ------------------------------------------------------------

NUMERIC_COLUMNS = [
    "mean_utility",
    "mean_confidence",
    "mean_margin",
    "mean_stability"
]

for col in NUMERIC_COLUMNS:

    if col in STRATEGY_SUMMARY_DF.columns:

        STRATEGY_SUMMARY_DF[col] = (
            pd.to_numeric(
                STRATEGY_SUMMARY_DF[col],
                errors="coerce"
            )
            .round(6)
        )


# ------------------------------------------------------------
# 5. SAVE SUMMARY
# ------------------------------------------------------------

STRATEGY_SUMMARY_PATH = (
    DIAGNOSTIC_DIR
    / "strategy_selection_summary.csv"
)

STRATEGY_SUMMARY_DF.to_csv(
    STRATEGY_SUMMARY_PATH,
    index=False
)


# ------------------------------------------------------------
# 6. SUMMARY COUNTS
# ------------------------------------------------------------

TOTAL_FEATURE_CONDITIONS = (
    len(
        SUMMARY_SOURCE_DF
    )
)

FINAL_SELECTION_COUNT = (
    SUMMARY_SOURCE_DF[
        SUMMARY_SOURCE_DF[
            "selection_class"
        ]
        ==
        "FINAL_ADAPTIVE_SELECTION"
    ]
    .shape[0]
)

PROVISIONAL_COUNT = (
    SUMMARY_SOURCE_DF[
        SUMMARY_SOURCE_DF[
            "selection_class"
        ]
        ==
        "PROVISIONAL_EMPIRICAL_WINNER"
    ]
    .shape[0]
)

NO_SELECTION_COUNT = (
    SUMMARY_SOURCE_DF[
        SUMMARY_SOURCE_DF[
            "selection_class"
        ]
        ==
        "NO_SELECTION"
    ]
    .shape[0]
)


# ------------------------------------------------------------
# 7. PRINT RESEARCH SUMMARY
# ------------------------------------------------------------

print("\nSELECTION SUMMARY")
print("-" * 100)

print(
    f"Total feature conditions       : "
    f"{TOTAL_FEATURE_CONDITIONS:,}"
)

print(
    f"Final adaptive selections      : "
    f"{FINAL_SELECTION_COUNT:,}"
)

print(
    f"Provisional empirical winners  : "
    f"{PROVISIONAL_COUNT:,}"
)

print(
    f"No selection                   : "
    f"{NO_SELECTION_COUNT:,}"
)


# ------------------------------------------------------------
# 8. STRATEGY DISTRIBUTION
# ------------------------------------------------------------

print("\nSTRATEGY DISTRIBUTION")
print("-" * 100)

if not STRATEGY_SUMMARY_DF.empty:

    for _, row in STRATEGY_SUMMARY_DF.iterrows():

        strategy = row[
            "selected_strategy"
        ]

        strategy_label = (
            "NA"
            if pd.isna(strategy)
            else str(strategy)
        )

        print(
            f"{strategy_label:<25} : "
            f"{int(row['feature_count']):>3} "
            f"features | "
            f"{row['selection_class']}"
        )

else:

    print(
        "No strategy-selection records available."
    )


# ------------------------------------------------------------
# 9. DISPLAY SUMMARY TABLE
# ------------------------------------------------------------

print("\nSTRATEGY SELECTION SUMMARY TABLE")
print("-" * 100)

display(
    STRATEGY_SUMMARY_DF
)


# ------------------------------------------------------------
# 10. RESEARCH-INTEGRITY INTERPRETATION
# ------------------------------------------------------------

print("\n" + "!" * 100)
print("RESEARCH-INTEGRITY INTERPRETATION")
print("!" * 100)

if FINAL_SELECTION_COUNT > 0:

    print(
        "The table contains final adaptive selections "
        "supported by the current comparative evaluation."
    )

    if PROVISIONAL_COUNT > 0:

        print(
            f"{PROVISIONAL_COUNT:,} feature conditions remain "
            "provisional because they do not satisfy the "
            "requirements for final adaptive selection."
        )

else:

    print(
        "No feature condition currently has a final adaptive "
        "selection."
    )

    print(
        f"All {PROVISIONAL_COUNT:,} available strategy records "
        "represent provisional empirical winners."
    )

    print(
        "The strategy counts therefore describe the distribution "
        "of provisional winners, not the distribution of final "
        "adaptive selections."
    )

    print(
        "Mean and mode are retained as provisional empirical "
        "winners because they are the currently evaluated "
        "strategies for their respective feature conditions."
    )

    print(
        "The absence of a comparative selection margin prevents "
        "these provisional winners from being interpreted as "
        "demonstrably superior to alternative strategies."
    )


# ------------------------------------------------------------
# 11. MARGIN DIAGNOSTIC
# ------------------------------------------------------------

VALID_MARGIN_COUNT = (
    SUMMARY_SOURCE_DF[
        pd.to_numeric(
            SUMMARY_SOURCE_DF[
                "selection_margin"
            ],
            errors="coerce"
        ).notna()
    ]
    .shape[0]
)

VALID_STABILITY_COUNT = (
    SUMMARY_SOURCE_DF[
        pd.to_numeric(
            SUMMARY_SOURCE_DF[
                "stability"
            ],
            errors="coerce"
        ).notna()
    ]
    .shape[0]
)

print("\nCOMPARATIVE EVIDENCE DIAGNOSTIC")
print("-" * 100)

print(
    f"Conditions with valid selection margin : "
    f"{VALID_MARGIN_COUNT:,}"
)

print(
    f"Conditions with valid stability         : "
    f"{VALID_STABILITY_COUNT:,}"
)

print(
    "\nA valid selection margin requires at least "
    "two empirically evaluated candidate strategies "
    "for the same dataset-feature condition."
)


# ------------------------------------------------------------
# 12. EXPORT CONFIRMATION
# ------------------------------------------------------------

print("\nEXPORT")
print("-" * 100)

print(
    f"Strategy summary saved:\n"
    f"{STRATEGY_SUMMARY_PATH}"
)

print("\n" + "=" * 100)
print(
    "NOTEBOOK 07.24 STRATEGY SELECTION SUMMARY : COMPLETE"
)
print("=" * 100)

AIR-LLM — NOTEBOOK 07.24
STRATEGY SELECTION SUMMARY

SELECTION SUMMARY
----------------------------------------------------------------------------------------------------
Total feature conditions       : 77
Final adaptive selections      : 0
Provisional empirical winners  : 77
No selection                   : 0

STRATEGY DISTRIBUTION
----------------------------------------------------------------------------------------------------
mode                      :  53 features | PROVISIONAL_EMPIRICAL_WINNER
mean                      :  24 features | PROVISIONAL_EMPIRICAL_WINNER

STRATEGY SELECTION SUMMARY TABLE
----------------------------------------------------------------------------------------------------


,selected_strategy,selection_class,feature_count,mean_utility,mean_confidence,mean_margin,mean_stability
0,mode,PROVISIONAL_EMPIRICAL_WINNER,53,0.509274,0.798874,NaN,0.712677
1,mean,PROVISIONAL_EMPIRICAL_WINNER,24,0.603430,0.836854,NaN,0.766934



!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
RESEARCH-INTEGRITY INTERPRETATION
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
No feature condition currently has a final adaptive selection.
All 77 available strategy records represent provisional empirical winners.
The strategy counts therefore describe the distribution of provisional winners, not the distribution of final adaptive selections.
Mean and mode are retained as provisional empirical winners because they are the currently evaluated strategies for their respective feature conditions.
The absence of a comparative selection margin prevents these provisional winners from being interpreted as demonstrably superior to alternative strategies.

COMPARATIVE EVIDENCE DIAGNOSTIC
----------------------------------------------------------------------------------------------------
Conditions with valid selection margin : 0
Conditio

In [62]:
# ============================================================
# NOTEBOOK 07.25 — DATASET SELECTION SUMMARY
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.25")
print("DATASET-LEVEL SELECTION SUMMARY")
print("=" * 100)


# ------------------------------------------------------------
# 1. VALIDATE INPUT
# ------------------------------------------------------------

REQUIRED_COLUMNS = {
    "dataset_id",
    "feature",
    "selected_strategy",
    "selected_utility",
    "empirical_confidence",
    "stability",
    "decision_status"
}

MISSING_COLUMNS = (
    REQUIRED_COLUMNS
    - set(FINAL_STRATEGY_DF.columns)
)

if MISSING_COLUMNS:
    raise RuntimeError(
        "FINAL_STRATEGY_DF is missing required columns: "
        f"{sorted(MISSING_COLUMNS)}"
    )


# ------------------------------------------------------------
# 2. CREATE ROBUST DECISION CLASS
# ------------------------------------------------------------

DATASET_SOURCE_DF = (
    FINAL_STRATEGY_DF.copy()
)


STATUS = (
    DATASET_SOURCE_DF[
        "decision_status"
    ]
    .astype(str)
    .str.upper()
    .str.strip()
)


DATASET_SOURCE_DF[
    "final_selected_flag"
] = (
    STATUS == "SELECT"
)


DATASET_SOURCE_DF[
    "abstained_flag"
] = (
    STATUS.str.startswith(
        "ABSTAIN"
    )
)


DATASET_SOURCE_DF[
    "provisional_winner_flag"
] = (
    DATASET_SOURCE_DF[
        "selected_strategy"
    ].notna()
    &
    ~DATASET_SOURCE_DF[
        "final_selected_flag"
    ]
)


# ------------------------------------------------------------
# 3. DATASET-LEVEL AGGREGATION
# ------------------------------------------------------------

DATASET_SELECTION_SUMMARY_DF = (
    DATASET_SOURCE_DF
    .groupby(
        "dataset_id",
        dropna=False
    )
    .agg(
        features=(
            "feature",
            "nunique"
        ),

        evaluated_conditions=(
            "selected_strategy",
            lambda s:
                int(
                    s.notna().sum()
                )
        ),

        final_selected=(
            "final_selected_flag",
            "sum"
        ),

        abstained=(
            "abstained_flag",
            "sum"
        ),

        provisional_winners=(
            "provisional_winner_flag",
            "sum"
        ),

        mean_utility=(
            "selected_utility",
            "mean"
        ),

        mean_confidence=(
            "empirical_confidence",
            "mean"
        ),

        mean_stability=(
            "stability",
            "mean"
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# 4. CONVERT COUNTS TO INTEGER
# ------------------------------------------------------------

COUNT_COLUMNS = [
    "features",
    "evaluated_conditions",
    "final_selected",
    "abstained",
    "provisional_winners"
]

for col in COUNT_COLUMNS:

    DATASET_SELECTION_SUMMARY_DF[
        col
    ] = (
        pd.to_numeric(
            DATASET_SELECTION_SUMMARY_DF[col],
            errors="coerce"
        )
        .fillna(0)
        .astype(int)
    )


# ------------------------------------------------------------
# 5. CALCULATE FINAL SELECTION RATE
# ------------------------------------------------------------

DATASET_SELECTION_SUMMARY_DF[
    "final_selection_rate"
] = np.where(
    DATASET_SELECTION_SUMMARY_DF[
        "features"
    ] > 0,

    DATASET_SELECTION_SUMMARY_DF[
        "final_selected"
    ]
    /
    DATASET_SELECTION_SUMMARY_DF[
        "features"
    ],

    np.nan
)


# ------------------------------------------------------------
# 6. CALCULATE ABSTENTION RATE
# ------------------------------------------------------------

DATASET_SELECTION_SUMMARY_DF[
    "abstention_rate"
] = np.where(
    DATASET_SELECTION_SUMMARY_DF[
        "features"
    ] > 0,

    DATASET_SELECTION_SUMMARY_DF[
        "abstained"
    ]
    /
    DATASET_SELECTION_SUMMARY_DF[
        "features"
    ],

    np.nan
)


# ------------------------------------------------------------
# 7. ROUND NUMERIC VALUES
# ------------------------------------------------------------

RATE_COLUMNS = [
    "final_selection_rate",
    "abstention_rate",
    "mean_utility",
    "mean_confidence",
    "mean_stability"
]

for col in RATE_COLUMNS:

    DATASET_SELECTION_SUMMARY_DF[
        col
    ] = (
        pd.to_numeric(
            DATASET_SELECTION_SUMMARY_DF[col],
            errors="coerce"
        )
        .round(6)
    )


# ------------------------------------------------------------
# 8. SORT
# ------------------------------------------------------------

DATASET_SELECTION_SUMMARY_DF = (
    DATASET_SELECTION_SUMMARY_DF
    .sort_values(
        "dataset_id"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 9. SAVE
# ------------------------------------------------------------

DATASET_SUMMARY_PATH = (
    DIAGNOSTIC_DIR
    / "dataset_selection_summary.csv"
)


DATASET_SELECTION_SUMMARY_DF.to_csv(
    DATASET_SUMMARY_PATH,
    index=False
)


# ------------------------------------------------------------
# 10. PRINT SUMMARY
# ------------------------------------------------------------

print("\nDATASET-LEVEL SUMMARY")
print("-" * 100)

for _, row in DATASET_SELECTION_SUMMARY_DF.iterrows():

    print(
        f"{row['dataset_id']:<20} | "
        f"features={row['features']:>3} | "
        f"final={row['final_selected']:>3} | "
        f"abstained={row['abstained']:>3} | "
        f"provisional={row['provisional_winners']:>3}"
    )


# ------------------------------------------------------------
# 11. DISPLAY
# ------------------------------------------------------------

print("\nDATASET SELECTION SUMMARY TABLE")
print("-" * 100)

display(
    DATASET_SELECTION_SUMMARY_DF
)


# ------------------------------------------------------------
# 12. GLOBAL CONSISTENCY CHECK
# ------------------------------------------------------------

GLOBAL_FEATURES = (
    int(
        DATASET_SELECTION_SUMMARY_DF[
            "features"
        ].sum()
    )
)

GLOBAL_FINAL = (
    int(
        DATASET_SELECTION_SUMMARY_DF[
            "final_selected"
        ].sum()
    )
)

GLOBAL_ABSTAINED = (
    int(
        DATASET_SELECTION_SUMMARY_DF[
            "abstained"
        ].sum()
    )
)

GLOBAL_PROVISIONAL = (
    int(
        DATASET_SELECTION_SUMMARY_DF[
            "provisional_winners"
        ].sum()
    )
)


print("\nGLOBAL CONSISTENCY CHECK")
print("-" * 100)

print(
    f"Total feature conditions : "
    f"{GLOBAL_FEATURES:,}"
)

print(
    f"Final SELECT decisions   : "
    f"{GLOBAL_FINAL:,}"
)

print(
    f"Abstained decisions      : "
    f"{GLOBAL_ABSTAINED:,}"
)

print(
    f"Provisional winners      : "
    f"{GLOBAL_PROVISIONAL:,}"
)


# ------------------------------------------------------------
# 13. VALIDATE AGAINST FINAL STRATEGY TABLE
# ------------------------------------------------------------

EXPECTED_TOTAL = len(
    FINAL_STRATEGY_DF
)

if GLOBAL_FEATURES != EXPECTED_TOTAL:

    raise RuntimeError(
        "Dataset summary consistency failure: "
        f"dataset-level feature total={GLOBAL_FEATURES}, "
        f"FINAL_STRATEGY_DF rows={EXPECTED_TOTAL}"
    )


if GLOBAL_FINAL + GLOBAL_ABSTAINED > GLOBAL_FEATURES:

    raise RuntimeError(
        "Invalid decision counts detected."
    )


# ------------------------------------------------------------
# 14. RESEARCH-INTEGRITY INTERPRETATION
# ------------------------------------------------------------

print("\n" + "!" * 100)
print("RESEARCH-INTEGRITY INTERPRETATION")
print("!" * 100)

if GLOBAL_FINAL > 0:

    print(
        f"{GLOBAL_FINAL:,} feature conditions have reached "
        "final adaptive selection."
    )

else:

    print(
        "No feature condition currently reaches final adaptive "
        "selection."
    )

    print(
        f"All {GLOBAL_ABSTAINED:,} applicable feature conditions "
        "remain abstained because comparative candidate evidence "
        "is insufficient."
    )

    print(
        f"{GLOBAL_PROVISIONAL:,} provisional empirical winners "
        "are retained for diagnostic and downstream evaluation "
        "purposes."
    )

    print(
        "Dataset-level strategy frequencies must therefore be "
        "interpreted as provisional empirical winner frequencies, "
        "not final adaptive-selection frequencies."
    )


# ------------------------------------------------------------
# 15. EXPORT CONFIRMATION
# ------------------------------------------------------------

print("\nEXPORT")
print("-" * 100)

print(
    f"Dataset selection summary saved:\n"
    f"{DATASET_SUMMARY_PATH}"
)

print("\n" + "=" * 100)
print(
    "NOTEBOOK 07.25 DATASET-LEVEL SELECTION SUMMARY : COMPLETE"
)
print("=" * 100)

AIR-LLM — NOTEBOOK 07.25
DATASET-LEVEL SELECTION SUMMARY

DATASET-LEVEL SUMMARY
----------------------------------------------------------------------------------------------------
adult_income         | features= 14 | final=  0 | abstained= 14 | provisional= 14
bank_marketing       | features= 16 | final=  0 | abstained= 16 | provisional= 16
diabetes_130us       | features= 47 | final=  0 | abstained= 47 | provisional= 47

DATASET SELECTION SUMMARY TABLE
----------------------------------------------------------------------------------------------------


,dataset_id,features,evaluated_conditions,final_selected,abstained,provisional_winners,mean_utility,mean_confidence,mean_stability,final_selection_rate,abstention_rate
0,adult_income,14,14,0,14,14,0.529766,0.803904,0.719863,0.0,1.0
1,bank_marketing,16,16,0,16,16,0.509594,0.795086,0.707265,0.0,1.0
2,diabetes_130us,47,47,0,47,47,0.551141,0.818059,0.740084,0.0,1.0



GLOBAL CONSISTENCY CHECK
----------------------------------------------------------------------------------------------------
Total feature conditions : 77
Final SELECT decisions   : 0
Abstained decisions      : 77
Provisional winners      : 77

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
RESEARCH-INTEGRITY INTERPRETATION
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
No feature condition currently reaches final adaptive selection.
All 77 applicable feature conditions remain abstained because comparative candidate evidence is insufficient.
77 provisional empirical winners are retained for diagnostic and downstream evaluation purposes.
Dataset-level strategy frequencies must therefore be interpreted as provisional empirical winner frequencies, not final adaptive-selection frequencies.

EXPORT
-------------------------------------------------------------------------------------

In [63]:
# ============================================================
# NOTEBOOK 07.26 — FINAL VALIDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.26")
print("FINAL SELECTION VALIDATION")
print("=" * 100)


VALIDATION_ROWS = []


# ------------------------------------------------------------
# Dataset coverage
# ------------------------------------------------------------

for dataset_id in DATASETS:

    count = int(
        (
            FINAL_STRATEGY_DF[
                "dataset_id"
            ]
            == dataset_id
        ).sum()
    )

    VALIDATION_ROWS.append({

        "check":
            f"Dataset coverage: {dataset_id}",

        "passed":
            count > 0,

        "value":
            count
    })


# ------------------------------------------------------------
# Utility bounds
# ------------------------------------------------------------

utility_valid = (
    FINAL_STRATEGY_DF[
        "selected_utility"
    ]
    .between(0, 1)
    .all()
)

VALIDATION_ROWS.append({

    "check":
        "Utility values in [0,1]",

    "passed":
        bool(utility_valid),

    "value":
        float(
            FINAL_STRATEGY_DF[
                "selected_utility"
            ].min()
        )
})


# ------------------------------------------------------------
# Confidence bounds
# ------------------------------------------------------------

confidence_valid = (
    FINAL_STRATEGY_DF[
        "empirical_confidence"
    ]
    .between(0, 1)
    .all()
)

VALIDATION_ROWS.append({

    "check":
        "Confidence values in [0,1]",

    "passed":
        bool(confidence_valid),

    "value":
        float(
            FINAL_STRATEGY_DF[
                "empirical_confidence"
            ].min()
        )
})


# ------------------------------------------------------------
# Stability bounds
# ------------------------------------------------------------

stability_valid = (
    FINAL_STRATEGY_DF[
        "stability"
    ]
    .between(0, 1)
    .all()
)

VALIDATION_ROWS.append({

    "check":
        "Stability values in [0,1]",

    "passed":
        bool(stability_valid),

    "value":
        float(
            FINAL_STRATEGY_DF[
                "stability"
            ].min()
        )
})


# ------------------------------------------------------------
# Strategy non-null
# ------------------------------------------------------------

strategy_valid = (
    FINAL_STRATEGY_DF[
        "selected_strategy"
    ]
    .notna()
    .all()
)

VALIDATION_ROWS.append({

    "check":
        "Selected strategy available",

    "passed":
        bool(strategy_valid),

    "value":
        int(
            FINAL_STRATEGY_DF[
                "selected_strategy"
            ].notna().sum()
        )
})


# ------------------------------------------------------------
# No target selection
# ------------------------------------------------------------

target_failures = []

for _, row in FINAL_STRATEGY_DF.iterrows():

    target = TARGET_REGISTRY[
        row["dataset_id"]
    ]

    if str(
        row["feature"]
    ) == str(target):

        target_failures.append(
            row["dataset_id"]
        )


VALIDATION_ROWS.append({

    "check":
        "Target variables excluded",

    "passed":
        len(target_failures) == 0,

    "value":
        len(target_failures)
})


VALIDATION_DF = pd.DataFrame(
    VALIDATION_ROWS
)


VALIDATION_PATH = (
    DIAGNOSTIC_DIR
    / "notebook_07_validation.csv"
)


VALIDATION_DF.to_csv(
    VALIDATION_PATH,
    index=False
)


display(
    VALIDATION_DF
)


if not VALIDATION_DF[
    "passed"
].all():

    raise RuntimeError(
        "Notebook 07 validation failed."
    )


print(
    "\nALL NOTEBOOK 07 SELECTION VALIDATIONS PASSED."
)

AIR-LLM — NOTEBOOK 07.26
FINAL SELECTION VALIDATION


,check,passed,value
0,Dataset coverage: adult_income,True,14.000000
1,Dataset coverage: bank_marketing,True,16.000000
2,Dataset coverage: diabetes_130us,True,47.000000
3,"Utility values in [0,1]",True,0.272874
4,"Confidence values in [0,1]",True,0.707704
5,"Stability values in [0,1]",True,0.582434
6,Selected strategy available,True,77.000000
7,Target variables excluded,True,0.000000



ALL NOTEBOOK 07 SELECTION VALIDATIONS PASSED.


In [64]:
# ============================================================
# NOTEBOOK 07.27 — DRIVE PERSISTENCE VALIDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.27")
print("DRIVE PERSISTENCE VALIDATION")
print("=" * 100)


REQUIRED_OUTPUTS = [

    ADAPTIVE_SELECTION_PATH,

    FINAL_STRATEGY_PATH,

    AUDIT_PATH,

    DECISION_MAP_PATH,

    STRATEGY_SUMMARY_PATH,

    DATASET_SUMMARY_PATH,

    VALIDATION_PATH
]


PERSISTENCE_ROWS = []


for path in REQUIRED_OUTPUTS:

    exists = (
        path.exists()
        and path.is_file()
        and path.stat().st_size > 0
    )

    PERSISTENCE_ROWS.append({

        "file":
            str(path),

        "exists":
            exists,

        "size_bytes":
            path.stat().st_size
            if path.exists()
            else 0
    })


PERSISTENCE_DF = pd.DataFrame(
    PERSISTENCE_ROWS
)


display(
    PERSISTENCE_DF
)


if not PERSISTENCE_DF[
    "exists"
].all():

    raise RuntimeError(
        "One or more Notebook 07 output files "
        "failed persistence validation."
    )


print(
    "\nDRIVE PERSISTENCE VALIDATION PASSED."
)

AIR-LLM — NOTEBOOK 07.27
DRIVE PERSISTENCE VALIDATION


,file,exists,size_bytes
0,/content/drive/MyDrive/AIR_LLM_Research/data/n...,True,17465
1,/content/drive/MyDrive/AIR_LLM_Research/data/n...,True,15905
2,/content/drive/MyDrive/AIR_LLM_Research/data/n...,True,12540
3,/content/drive/MyDrive/AIR_LLM_Research/data/n...,True,31753
4,/content/drive/MyDrive/AIR_LLM_Research/data/n...,True,233
5,/content/drive/MyDrive/AIR_LLM_Research/data/n...,True,357
6,/content/drive/MyDrive/AIR_LLM_Research/data/n...,True,375



DRIVE PERSISTENCE VALIDATION PASSED.


In [66]:
# ============================================================
# NOTEBOOK 07.28 — DECISION EXPLANATIONS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.28")
print("AUDITABLE DECISION EXPLANATIONS")
print("=" * 100)


# ------------------------------------------------------------
# 1. SOURCE DATA
# ------------------------------------------------------------

EXPLANATION_SOURCE_DF = FINAL_STRATEGY_DF.copy()


# ------------------------------------------------------------
# 2. OPTIONAL LLM INFORMATION
# ------------------------------------------------------------
# FINAL_STRATEGY_DF does not currently contain these columns.
# Recover them from LLM_ALIGNMENT_DF when available.

LLM_INFO_COLUMNS = [
    "dataset_id",
    "feature",
    "llm_recommended_strategy",
    "llm_alignment"
]


if (
    "llm_recommended_strategy" in EXPLANATION_SOURCE_DF.columns
    and
    "llm_alignment" in EXPLANATION_SOURCE_DF.columns
):

    # LLM information already exists
    pass

elif all(
    c in LLM_ALIGNMENT_DF.columns
    for c in LLM_INFO_COLUMNS
):

    LLM_INFO_DF = (
        LLM_ALIGNMENT_DF[
            LLM_INFO_COLUMNS
        ]
        .drop_duplicates(
            subset=[
                "dataset_id",
                "feature"
            ]
        )
    )

    EXPLANATION_SOURCE_DF = (
        EXPLANATION_SOURCE_DF
        .drop(
            columns=[
                c
                for c in [
                    "llm_recommended_strategy",
                    "llm_alignment"
                ]
                if c in EXPLANATION_SOURCE_DF.columns
            ],
            errors="ignore"
        )
        .merge(
            LLM_INFO_DF,
            on=[
                "dataset_id",
                "feature"
            ],
            how="left"
        )
    )

else:

    EXPLANATION_SOURCE_DF[
        "llm_recommended_strategy"
    ] = pd.NA

    EXPLANATION_SOURCE_DF[
        "llm_alignment"
    ] = np.nan


# ------------------------------------------------------------
# 3. SAFE COLUMN CREATION
# ------------------------------------------------------------

for column in [
    "llm_recommended_strategy",
    "llm_alignment"
]:

    if column not in EXPLANATION_SOURCE_DF.columns:

        EXPLANATION_SOURCE_DF[
            column
        ] = pd.NA


# ------------------------------------------------------------
# 4. BUILD EXPLANATIONS
# ------------------------------------------------------------

EXPLANATION_ROWS = []


for _, row in EXPLANATION_SOURCE_DF.iterrows():

    dataset_id = row["dataset_id"]

    feature = row["feature"]

    strategy = row["selected_strategy"]

    status = row["decision_status"]


    # --------------------------------------------------------
    # Context / Missingness
    # --------------------------------------------------------

    context = CONTEXT_DF[
        (
            CONTEXT_DF["dataset_id"]
            .astype(str)
            ==
            str(dataset_id)
        )
        &
        (
            CONTEXT_DF["feature"]
            .astype(str)
            ==
            str(feature)
        )
    ]


    if (
        not context.empty
        and
        "natural_missing_rate"
        in context.columns
        and
        pd.notna(
            context.iloc[0]["natural_missing_rate"]
        )
    ):

        missing_rate = float(
            context.iloc[0][
                "natural_missing_rate"
            ]
        )

    else:

        missing_rate = np.nan


    # --------------------------------------------------------
    # Numeric values — safely handle NaN
    # --------------------------------------------------------

    utility = row.get(
        "selected_utility",
        np.nan
    )

    confidence = row.get(
        "empirical_confidence",
        np.nan
    )

    margin = row.get(
        "selection_margin",
        np.nan
    )

    stability = row.get(
        "stability",
        np.nan
    )

    llm_strategy = row.get(
        "llm_recommended_strategy",
        pd.NA
    )

    llm_alignment = row.get(
        "llm_alignment",
        np.nan
    )


    # --------------------------------------------------------
    # Candidate count
    # --------------------------------------------------------

    candidate_count = np.nan

    if "candidate_count" in row.index:

        candidate_count = row[
            "candidate_count"
        ]

    elif (
        "dataset_id" in AGGREGATION_DF.columns
        and
        "feature" in AGGREGATION_DF.columns
    ):

        candidate_count = int(
            AGGREGATION_DF[
                (
                    AGGREGATION_DF[
                        "dataset_id"
                    ]
                    == dataset_id
                )
                &
                (
                    AGGREGATION_DF[
                        "feature"
                    ]
                    == feature
                )
            ]["strategy"]
            .nunique()
        )


    if pd.isna(candidate_count):

        candidate_count = 0

    candidate_count = int(
        candidate_count
    )


    # --------------------------------------------------------
    # Decision explanation
    # --------------------------------------------------------

    if status == "SELECT":

        if (
            pd.notna(utility)
            and
            pd.notna(confidence)
            and
            pd.notna(stability)
        ):

            decision_text = (
                f"{strategy} was selected as the final adaptive "
                f"strategy because it achieved the highest "
                f"empirical utility ({float(utility):.4f}) "
                f"with empirical confidence "
                f"({float(confidence):.4f}) and stability "
                f"({float(stability):.4f})."
            )

        else:

            decision_text = (
                f"{strategy} was selected as the final adaptive "
                f"strategy based on the available empirical "
                f"evaluation evidence."
            )


    elif (
        status
        ==
        "ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE"
    ):

        decision_text = (
            f"{strategy} is retained only as a provisional "
            f"empirical winner for {feature}. "
            f"Only {candidate_count} empirically evaluated "
            f"candidate strategy was available for this feature, "
            f"so comparative superiority could not be established. "
            f"The framework therefore abstains from declaring "
            f"a final adaptive selection."
        )


    elif (
        status
        ==
        "ABSTAIN_ADDITIONAL_EVALUATION"
    ):

        decision_text = (
            f"{strategy} remains provisional because additional "
            f"empirical evaluation is required before final "
            f"acceptance."
        )


    else:

        decision_text = (
            f"{strategy} is currently treated as a provisional "
            f"empirical winner. The decision status is "
            f"{status}, and the framework does not interpret "
            f"this record as a final adaptive selection."
        )


    # --------------------------------------------------------
    # 5. STORE AUDIT RECORD
    # --------------------------------------------------------

    EXPLANATION_ROWS.append({

        "dataset_id":
            dataset_id,

        "feature":
            feature,

        "missing_rate":
            missing_rate,

        "candidate_count":
            candidate_count,

        "selected_strategy":
            strategy,

        "empirical_utility":
            utility,

        "empirical_confidence":
            confidence,

        "selection_margin":
            margin,

        "stability":
            stability,

        "llm_recommended_strategy":
            llm_strategy,

        "llm_alignment":
            llm_alignment,

        "decision_status":
            status,

        "decision_explanation":
            decision_text
    })


# ------------------------------------------------------------
# 6. FINAL EXPLANATION DATAFRAME
# ------------------------------------------------------------

DECISION_EXPLANATION_DF = pd.DataFrame(
    EXPLANATION_ROWS
)


# ------------------------------------------------------------
# 7. SAVE
# ------------------------------------------------------------

EXPLANATION_PATH = (
    AUDIT_DIR
    / "decision_explanations.csv"
)


DECISION_EXPLANATION_DF.to_csv(
    EXPLANATION_PATH,
    index=False
)


# ------------------------------------------------------------
# 8. SUMMARY
# ------------------------------------------------------------

print("\nDECISION EXPLANATION SUMMARY")
print("-" * 100)

print(
    f"Explanation records : "
    f"{len(DECISION_EXPLANATION_DF):,}"
)

print(
    f"SELECT decisions    : "
    f"{(
        DECISION_EXPLANATION_DF[
            'decision_status'
        ] == 'SELECT'
    ).sum():,}"
)

print(
    f"Abstentions         : "
    f"{(
        DECISION_EXPLANATION_DF[
            'decision_status'
        ]
        .astype(str)
        .str.startswith("ABSTAIN")
    ).sum():,}"
)

print(
    f"Saved to:\n"
    f"{EXPLANATION_PATH}"
)


# ------------------------------------------------------------
# 9. DISPLAY
# ------------------------------------------------------------

display(
    DECISION_EXPLANATION_DF.head(20)
)


print(
    "\nNOTEBOOK 07.28 DECISION EXPLANATIONS : COMPLETE"
)

AIR-LLM — NOTEBOOK 07.28
AUDITABLE DECISION EXPLANATIONS

DECISION EXPLANATION SUMMARY
----------------------------------------------------------------------------------------------------
Explanation records : 77
SELECT decisions    : 0
Abstentions         : 77
Saved to:
/content/drive/MyDrive/AIR_LLM_Research/data/notebook_07/audit/decision_explanations.csv


,dataset_id,feature,missing_rate,candidate_count,selected_strategy,empirical_utility,empirical_confidence,selection_margin,stability,llm_recommended_strategy,llm_alignment,decision_status,decision_explanation
0,adult_income,age,0.000000,1,mean,0.598182,0.829193,NaN,0.755990,NaN,NaN,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE,mean is retained only as a provisional empiric...
1,adult_income,capital_gain,0.000000,1,mean,0.607990,0.846080,NaN,0.780114,NaN,NaN,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE,mean is retained only as a provisional empiric...
2,adult_income,capital_loss,0.000000,1,mean,0.627236,0.839302,NaN,0.770431,NaN,NaN,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE,mean is retained only as a provisional empiric...
3,adult_income,education,0.000000,1,mode,0.544829,0.812178,NaN,0.731684,NaN,NaN,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE,mode is retained only as a provisional empiric...
4,adult_income,education_num,0.000000,1,mean,0.550408,0.804685,NaN,0.720978,NaN,NaN,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE,mean is retained only as a provisional empiric...
5,adult_income,fnlwgt,0.000000,1,mean,0.564509,0.823262,NaN,0.747517,NaN,NaN,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE,mean is retained only as a provisional empiric...
6,adult_income,hours_per_week,0.000000,1,mean,0.587983,0.825379,NaN,0.750542,NaN,NaN,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE,mean is retained only as a provisional empiric...
7,adult_income,marital_status,0.000000,1,mode,0.488762,0.784588,NaN,0.692268,NaN,NaN,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE,mode is retained only as a provisional empiric...
8,adult_income,native_country,0.017857,1,mode,0.531706,0.810989,NaN,0.729984,gradient_boosting,0.0,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE,mode is retained only as a provisional empiric...
9,adult_income,occupation,0.056204,1,mode,0.339888,0.750031,NaN,0.642902,gradient_boosting,0.0,ABSTAIN_INSUFFICIENT_COMPARATIVE_EVIDENCE,mode is retained only as a provisional empiric...



NOTEBOOK 07.28 DECISION EXPLANATIONS : COMPLETE


In [67]:
# ============================================================
# NOTEBOOK 07.29 — MANIFEST
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.29")
print("NOTEBOOK MANIFEST")
print("=" * 100)


def file_hash(path):

    if not path.exists():
        return None

    sha256 = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda:
                f.read(1024 * 1024),
            b""
        ):

            sha256.update(chunk)

    return sha256.hexdigest()


OUTPUT_MANIFEST = {}


for path in REQUIRED_OUTPUTS + [
    EXPLANATION_PATH
]:

    if path.exists():

        OUTPUT_MANIFEST[
            str(path.relative_to(PROJECT_ROOT))
        ] = {

            "size_bytes":
                path.stat().st_size,

            "sha256":
                file_hash(path)
        }


MANIFEST = {

    "notebook":
        "07_Adaptive_Utility_Based_Selection",

    "framework":
        "AIR-LLM",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "master_seed":
        MASTER_SEED,

    "datasets":
        DATASETS,

    "target_registry":
        TARGET_REGISTRY,

    "utility_criteria":
        CRITERIA,

    "base_weights":
        BASE_WEIGHTS,

    "confidence_threshold":
        CONFIDENCE_THRESHOLD,

    "margin_threshold":
        MARGIN_THRESHOLD,

    "stability_threshold":
        STABILITY_THRESHOLD,

    "input_artifacts": {

        "feature_profile":
            str(FEATURE_PROFILE_PATH),

        "missingness_profile":
            str(MISSINGNESS_PROFILE_PATH),

        "candidate_evaluation":
            str(EVALUATION_PATH),

        "llm_recommendations":
            (
                str(LLM_RECOMMENDATION_PATH)
                if LLM_RECOMMENDATION_PATH
                else None
            )
    },

    "results": {

        "candidate_combinations":
            len(AGGREGATION_DF),

        "feature_decisions":
            len(FINAL_STRATEGY_DF),

        "automatic_selections":
            int(
                (
                    FINAL_STRATEGY_DF[
                        "decision_status"
                    ]
                    == "SELECT"
                ).sum()
            ),

        "abstentions":
            int(
                (
                    FINAL_STRATEGY_DF[
                        "decision_status"
                    ]
                    == "ABSTAIN_ADDITIONAL_EVALUATION"
                ).sum()
            )
    },

    "outputs":
        OUTPUT_MANIFEST
}


MANIFEST_PATH = (
    METADATA_DIR
    / "notebook_07_manifest.json"
)


with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        MANIFEST,
        f,
        indent=2,
        allow_nan=False
    )


print(
    f"Manifest saved:\n{MANIFEST_PATH}"
)

AIR-LLM — NOTEBOOK 07.29
NOTEBOOK MANIFEST
Manifest saved:
/content/drive/MyDrive/AIR_LLM_Research/data/notebook_07/metadata/notebook_07_manifest.json


In [71]:
# ============================================================
# NOTEBOOK 07.30 — FINAL NOTEBOOK STATUS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07 COMPLETE")
print("=" * 100)


TOTAL_FEATURES = len(
    FINAL_STRATEGY_DF
)

TOTAL_SELECTED = int(
    (
        FINAL_STRATEGY_DF[
            "decision_status"
        ]
        == "SELECT"
    ).sum()
)

TOTAL_ABSTAINED = int(
    (
        FINAL_STRATEGY_DF[
            "decision_status"
        ]
        == "ABSTAIN_ADDITIONAL_EVALUATION"
    ).sum()
)


print("\nEXPERIMENTAL SELECTION")
print("-" * 100)

print(
    f"Datasets              : {len(DATASETS)}"
)

print(
    f"Feature decisions     : {TOTAL_FEATURES}"
)

print(
    f"Automatically selected: {TOTAL_SELECTED}"
)

print(
    f"Abstained             : {TOTAL_ABSTAINED}"
)


print("\nSELECTION QUALITY")
print("-" * 100)

print(
    "Mean empirical utility : "
    f"{FINAL_STRATEGY_DF['selected_utility'].mean():.4f}"
)

print(
    "Mean empirical confidence: "
    f"{FINAL_STRATEGY_DF['empirical_confidence'].mean():.4f}"
)

print(
    "Mean stability         : "
    f"{FINAL_STRATEGY_DF['stability'].mean():.4f}"
)

print(
    "Mean selection margin  : "
    f"{FINAL_STRATEGY_DF['selection_margin'].mean():.4f}"
)


print("\nVALIDATION")
print("-" * 100)

print(
    "Utility validation     : PASSED"
)

print(
    "Confidence validation  : PASSED"
)

print(
    "Stability validation   : PASSED"
)

print(
    "Target exclusion       : PASSED"
)

print(
    "Drive persistence      : PASSED"
)


print("\nOUTPUTS")
print("-" * 100)

print(
    f"Adaptive selection:\n"
    f"{ADAPTIVE_SELECTION_PATH}"
)

print(
    f"Final strategy:\n"
    f"{FINAL_STRATEGY_PATH}"
)

print(
    f"Selection audit:\n"
    f"{AUDIT_PATH}"
)

print(
    f"Decision map:\n"
    f"{DECISION_MAP_PATH}"
)

print(
    f"Decision explanations:\n"
    f"{EXPLANATION_PATH}"
)

print(
    f"Manifest:\n"
    f"{MANIFEST_PATH}"
)


print("\n" + "=" * 100)

if TOTAL_ABSTAINED > 0:

    print(
        "NOTEBOOK 07 COMPLETED WITH CONTROLLED ABSTENTIONS"
    )

    print(
        "Abstained features require additional candidate "
        "evaluation before final acceptance."
    )

else:

    print(
        "ALL FEATURES RECEIVED EMPIRICALLY SUPPORTED "
        "STRATEGY SELECTIONS."
    )

print(
    "NOTEBOOK 07 — ADAPTIVE UTILITY-BASED SELECTION COMPLETE"
)

print(
    "NEXT: NOTEBOOK 08 — FINAL IMPUTATION AND END-TO-END EVALUATION"
)

print("=" * 100)


gc.collect()

AIR-LLM — NOTEBOOK 07 COMPLETE

EXPERIMENTAL SELECTION
----------------------------------------------------------------------------------------------------
Datasets              : 3
Feature decisions     : 77
Automatically selected: 0
Abstained             : 0

SELECTION QUALITY
----------------------------------------------------------------------------------------------------
Mean empirical utility : 0.5386
Mean empirical confidence: 0.8107
Mean stability         : 0.7296
Mean selection margin  : nan

VALIDATION
----------------------------------------------------------------------------------------------------
Utility validation     : PASSED
Confidence validation  : PASSED
Stability validation   : PASSED
Target exclusion       : PASSED
Drive persistence      : PASSED

OUTPUTS
----------------------------------------------------------------------------------------------------
Adaptive selection:
/content/drive/MyDrive/AIR_LLM_Research/data/notebook_07/selection/adaptive_selection.csv

644